<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.it/cap05/cap05_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 5 Trasformate e Compressione

Nei capitoli precedenti, tutte le operazioni sono state eseguite **nel dominio spaziale**, in cui gli algoritmi agiscono direttamente sui valori di intensità dei pixel.

In questo capitolo verrà presentato un approccio complementare: il **dominio della frequenza**, nel quale l'immagine è rappresentata dalle variazioni spaziali di intensità, e non solo dai valori individuali dei pixel.

Il concetto di **frequenza spaziale** descrive la rapidità con cui l'intensità varia lungo l'immagine. Le variazioni lente corrispondono a **basse frequenze**, mentre bordi, dettagli fini e rumori corrispondono ad **alte frequenze**.

Questa rappresentazione si basa sul fatto che qualsiasi immagine digitale discreta può essere decomposta in una combinazione di **funzioni ortogonali**. La **Trasformata di Fourier** utilizza una **base di esponenziali complesse bidimensionali** (equivalenti a sinusoidi con orientamento e frequenza specifici). Altre trasformate, come la **Trasformata del Coseno (DCT)** e la **Trasformata *Wavelet* (DWT)**, utilizzano diverse famiglie di funzioni di base — coseni bidimensionali nel caso della DCT, e funzioni con supporto compatto nel caso delle *wavelet*.

Tra le principali applicazioni di questa rappresentazione si evidenziano:

1. **Filtraggio nel dominio della frequenza**, per attenuare o enfatizzare determinate bande di frequenza;
2. **Analisi multirisoluzione tramite trasformate *wavelet***, che rappresenta strutture a diverse scale;
3. **Compressione delle immagini**, mediante la riduzione del numero di coefficienti necessari per rappresentare l'immagine.

## 5.1 Obiettivi

Al termine di questo capitolo, sarai in grado di:

- **Interpretare lo spettro di Fourier** di un'immagine, distinguendo magnitudine, fase e componenti di frequenza;
- **Applicare il Teorema della Convoluzione** per effettuare filtraggi nel dominio della frequenza utilizzando la Trasformata Veloce di Fourier (FFT);
- **Progettare e analizzare filtri nel dominio della frequenza**, comprendendo il funzionamento di filtri passa-basso, passa-alto e *notch*;
- **Comprendere l'analisi multirisoluzione tramite trasformate *wavelet*** e la loro applicazione nella rappresentazione gerarchica delle immagini;
- **Descrivere il processo di compressione delle immagini**, inclusa la Trasformata Discreta del Coseno (DCT) e la quantizzazione dei coefficienti;
- **Selezionare formati di archiviazione delle immagini**, come JPEG, PNG e WebP, in base ai requisiti dell'applicazione.

## 5.2 Configurazione dell'Ambiente

In [1]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatti di build della traccia C++

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# Kernel Python anche nella traccia C++. cpp=True scarica morph.hpp + stb; le
# celle %%writefile *.cpp di questo capitolo compilano CON OpenCV
# (-DMM_USE_OPENCV + pkg-config opencv4).
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

✅ Ambiente pronto. Morph: 1.1.9 | OpenCV: 5.0.0


## 5.3 Trasformata di Fourier Discreta 2D

L'analisi di Fourier si basa sul principio secondo cui qualsiasi segnale periodico può essere rappresentato come una somma di funzioni sinusoidali con differenti frequenze, ampiezze e fasi. Questo concetto si applica anche alle immagini digitali, consentendo di rappresentarle nel **dominio della frequenza** invece che nel dominio spaziale.

La [Figura 5.1](#fig-decomposicao-1d) illustra questa decomposizione per un segnale unidimensionale. Nel caso di un'immagine, la Trasformata Discreta di Fourier (DFT) converte la matrice delle intensità $f(x,y)$ in un insieme di coefficienti che descrive il contributo delle diverse frequenze spaziali presenti nell'immagine.

In [2]:
%%writefile tmp/fig_decomposicao_1d.cpp
#define MM_OUT "tmp/fig_decomposicao_1d.png"
//| label: fig-decomposicao-1d
//| fig-cap: "Decomposição de Fourier 1D: uma onda quadrada (linha tracejada) é aproximada pela soma das primeiras senoides (linhas coloridas). Quanto mais termos, melhor a aproximação."
//| echo: false
//| output: true

#include <vector>
#include <string>
#include <cmath>
#include <iostream>
#include "morph.hpp"
#include <filesystem>

int main() {
    // Cria um vetor com 400 pontos entre 0 e 2*pi
    std::vector<double> x(400);
    for (int i = 0; i < 400; i++) {
        x[i] = i * 2.0 * M_PI / 399.0;  // equivalente a np.linspace(0, 2*pi, 400)
    }

    // Onda quadrada: 1 para x < pi, -1 caso contrário
    std::vector<double> square(400);
    for (int i = 0; i < 400; i++) {
        square[i] = (x[i] < M_PI) ? 1.0 : -1.0;
    }

    // Soma das harmônicas
    std::vector<double> soma(400, 0.0);
    std::vector<std::vector<double>> ys;
    std::vector<double> h1(400), h2(400), h3(400);

    for (int n = 0; n < 3; n++) {
        std::vector<double> h(400);
        for (int i = 0; i < 400; i++) {
            h[i] = (4.0 / M_PI) * (1.0 / (2 * n + 1)) * std::sin((2 * n + 1) * x[i]);
            soma[i] += h[i];
        }
        if (n == 0) h1 = h;
        if (n == 1) h2 = h;
        if (n == 2) h3 = h;
    }

    ys.push_back(square);
    ys.push_back(h1);
    ys.push_back(h2);
    ys.push_back(h3);
    ys.push_back(soma);

    std::vector<std::string> labels = {
        "Onda quadrada ideal", "1a harmonica", "3a harmonica", "5a harmonica", "Soma (3 primeiras)"
    };
    std::vector<cv::Scalar> colors = {
        cv::Scalar(40, 40, 40), cv::Scalar(60, 160, 80), cv::Scalar(180, 120, 60),
        cv::Scalar(150, 80, 160), cv::Scalar(60, 60, 220)
    };

    mm::Image chart = mm::lineChart(
        x, ys, labels, colors,
        "Sintese de Fourier: de senos a uma onda quadrada",
        "Posicao", "Intensidade"
    );

    std::vector<mm::Image> charts = {chart};
    mm::show(charts, MM_OUT, {"Decomposicao de Fourier 1D"}, 1);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(chart, "tmp/fig_decomposicao_1d_0.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_decomposicao_1d.cpp


In [3]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_decomposicao_1d.cpp -o tmp/fig_decomposicao_1d -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_decomposicao_1d \
  && test -f "tmp/fig_decomposicao_1d.png" \
  || echo "⚠ mm::show não gravou tmp/fig_decomposicao_1d.png"

[1] Decomposicao de Fourier 1D


In [4]:
try:
    mm.show(
        [
            mm.read("tmp/fig_decomposicao_1d_0.png"),
        ],
        titles=[
            'Decomposicao de Fourier 1D',
        ],
        cols=1,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_decomposicao_1d_0.png (ver a versao Python)")

<Figure size 750x750 with 1 Axes>

**Figura 5.1:** Decomposição de Fourier 1D: uma onda quadrada (linha tracejada) é aproximada pela soma das primeiras senoides (linhas coloridas). Quanto mais termos, melhor a aproximação.


### 5.3.1 Simulatore: Ricostruire Segnali con Sinusoidi

Prima di studiare le immagini bidimensionali, il simulatore della [Figura 5.2](#fig-05-sim-05-freq) illustra il principio dell'analisi di Fourier per segnali unidimensionali: **una forma d'onda può essere approssimata dalla somma di sinusoidi con diverse frequenze e ampiezze**.

Man mano che si aggiungono nuovi termini, la somma delle sinusoidi (curva nera) si avvicina alla forma d'onda di riferimento (tratteggiata). Il grafico inferiore mostra lo spettro delle ampiezze, indicando il contributo di ciascuna frequenza alla ricostruzione del segnale.

> ### 💡 Attività
>
> Esplora il simulatore e rispondi:
>
> 1. Quanti termini sono necessari per ottenere una buona approssimazione dell'onda quadra?
> 2. Quale delle tre forme d'onda converge più rapidamente? Giustifica la tua risposta.
> 3. Come cambia lo spettro delle ampiezze passando dall'onda quadra a quella triangolare?

> ### 📝 Risposte
>
> **1. Quanti termini sono necessari per una buona approssimazione dell'onda quadra?**
>
> Con circa 15-20 termini, la forma dell'onda si avvicina già bene al riferimento. Tuttavia, in prossimità delle discontinuità permane una piccola oscillazione, nota come **fenomeno di Gibbs**, che non scompare nemmeno aggiungendo più termini.
>
> **2. Quale forma converge più rapidamente? Perché?**
>
> L'**onda triangolare** converge più rapidamente, poiché le ampiezze delle sue armoniche decadono più velocemente di quelle dell'onda quadra e dell'onda a dente di sega. Di conseguenza, pochi termini già producono una buona approssimazione.
>
> **3. Come cambia lo spettro tra l'onda quadra e quella triangolare?**
>
> Entrambe presentano solo **armoniche dispari**, ma nell'onda triangolare le ampiezze diminuiscono molto più rapidamente. Pertanto, poche armoniche sono sufficienti per ricostruire il segnale con buona precisione.

In [5]:
# @title { display-mode: "form" }
from IPython.display import HTML

HTML("""
<div id="sim-05-freq" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-freq * { box-sizing: border-box; }
  #sim-05-freq canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-freq button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; }
  #sim-05-freq button:hover { background: #e8dfcf; }
  #sim-05-freq button.sim05fft_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim05fft_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05fft_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim05fft_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 90px; }
  .sim05fft_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim05fft_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">∿ Simulatore: Scomposizione di Fourier 1D</span>
  <span class="sim05fft_pill">somma di sinusoidi</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Termini</div><div id="sim05fft_nTerms" class="sim05fft_stat_value" style="color:#2980b9;">1</div></div>
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Errore RMS</div><div id="sim05fft_rms" class="sim05fft_stat_value" style="color:#c0392b;">–</div></div>
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Forma Target</div><div id="sim05fft_target" class="sim05fft_stat_value" style="color:#27ae60; font-size:13px;">quadra</div></div>
  </div>

  <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:12px;text-align:center;">
    <canvas id="sim05fft_Canvas" width="660" height="200" style="margin:0 auto;"></canvas>
    <canvas id="sim05fft_SpecCanvas" width="660" height="80" style="margin:8px auto 0 auto;"></canvas>
  </div>

  <div style="display:flex; gap:12px; margin-top:12px; flex-wrap:wrap;">
    <div class="sim05fft_panel" style="flex:1; min-width:200px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Forma Target</div>
      <div style="display:flex; gap:6px; flex-wrap:wrap;">
        <button id="sim05fft_sq" class="sim05fft_active" onclick="sim05fft_setTarget('square')" style="flex:1; justify-content:center;">Quadra</button>
        <button id="sim05fft_tr" onclick="sim05fft_setTarget('triangle')" style="flex:1; justify-content:center;">Triangolare</button>
        <button id="sim05fft_sw" onclick="sim05fft_setTarget('sawtooth')" style="flex:1; justify-content:center;">A dente di sega</button>
      </div>
    </div>
    
    <div class="sim05fft_panel" style="flex:1; min-width:180px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Numero di Termini</div>
      <div style="display:flex; align-items:center; gap:8px;">
        <input type="range" id="sim05fft_slider" min="1" max="25" value="1" style="flex:1; cursor:pointer; height:4px;">
        <span id="sim05fft_slVal" style="font-size:12px; font-family:monospace; font-weight:700; min-width:22px; color:#26241d;">1</span>
      </div>
    </div>

    <div class="sim05fft_panel" style="flex:1; min-width:140px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Visualizzazione</div>
      <div style="display:flex; gap:6px;">
        <button id="sim05fft_chk_comp" class="sim05fft_active" onclick="sim05fft_toggleComp()" style="flex:1; justify-content:center;">Componenti</button>
        <button id="sim05fft_chk_sum" class="sim05fft_active" onclick="sim05fft_toggleSum()" style="flex:1; justify-content:center;">Somma</button>
      </div>
    </div>
  </div>

  <div id="sim05fft_termList" style="margin-top:12px; display:flex; gap:6px; flex-wrap:wrap; justify-content:center;"></div>

</div>
</div>

<script>
(function(){
  function initSim05FFT(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const C = root.querySelector('#sim05fft_Canvas');
    const S = root.querySelector('#sim05fft_SpecCanvas');
    const ctx = C.getContext('2d');
    const sctx = S.getContext('2d');
    const sim05fft_N = 512;
    let sim05fft_nTerms = 1, sim05fft_showComp = true, sim05fft_showSum = true, sim05fft_targetType = 'square';

    const sim05fft_PALETTE = ['#2980b9','#27ae60','#b9770e','#c0392b','#8e44ad','#16a085','#d35400','#2c3e50'];

    function sim05fft_getTerms(type, n) {
      const terms = [];
      for (let k = 1; k <= n; k++) {
        let freq, amp, phase = 0;
        if (type === 'square') {
          const m = 2*k - 1;
          freq = m; amp = (4/Math.PI) * (1/m);
        } else if (type === 'triangle') {
          const m = 2*k - 1;
          freq = m; amp = (8/Math.PI**2) * (1/m**2);
          phase = -Math.PI/2;
        } else {
          freq = k; amp = (2/Math.PI) * (1/k);
          phase = Math.PI;
        }
        terms.push({freq, amp, phase});
      }
      return terms;
    }

    function sim05fft_getTarget(type) {
      const t = new Float32Array(sim05fft_N);
      for (let i = 0; i < sim05fft_N; i++) {
        const x = i / sim05fft_N;
        if (type === 'square') t[i] = x < 0.5 ? 1 : -1;
        else if (type === 'triangle') t[i] = x < 0.5 ? (4*x - 1) : (3 - 4*x);
        else t[i] = 2*x - 1;
      }
      return t;
    }

    function sim05fft_evalTerms(terms) {
      const sig = new Float32Array(sim05fft_N);
      for (const {freq, amp, phase} of terms) {
        for (let i = 0; i < sim05fft_N; i++) {
          sig[i] += amp * Math.sin(2*Math.PI*freq*i/sim05fft_N + phase);
        }
      }
      return sig;
    }

    function sim05fft_draw() {
      const W = C.width, H = C.height;
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#ffffff'; ctx.fillRect(0, 0, W, H);

      const terms = sim05fft_getTerms(sim05fft_targetType, sim05fft_nTerms);
      const sum = sim05fft_evalTerms(terms);
      const target = sim05fft_getTarget(sim05fft_targetType);
      let rms = 0;
      for (let i = 0; i < sim05fft_N; i++) rms += (sum[i]-target[i])**2;
      rms = Math.sqrt(rms/sim05fft_N);

      // Grid
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 0.5; ctx.setLineDash([3,3]);
      ctx.beginPath(); ctx.moveTo(0,H/2); ctx.lineTo(W,H/2); ctx.stroke();
      ctx.setLineDash([]);

      // Componentes individuais
      if (sim05fft_showComp) {
        for (let k = 0; k < terms.length; k++) {
          const {freq, amp, phase} = terms[k];
          ctx.strokeStyle = sim05fft_PALETTE[k % sim05fft_PALETTE.length] + '55';
          ctx.lineWidth = 1;
          ctx.beginPath();
          for (let i = 0; i < sim05fft_N; i++) {
            const x = i * W / sim05fft_N;
            const y = H/2 - amp * Math.sin(2*Math.PI*freq*i/sim05fft_N + phase) * H/3;
            i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
          }
          ctx.stroke();
        }
      }

      // Sinal alvo
      ctx.strokeStyle = '#8a8371'; ctx.lineWidth = 1.5; ctx.setLineDash([4,4]);
      ctx.beginPath();
      for (let i = 0; i < sim05fft_N; i++) {
        const x = i * W / sim05fft_N;
        const y = H/2 - target[i] * H/3;
        i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
      }
      ctx.stroke(); ctx.setLineDash([]);

      // Soma
      if (sim05fft_showSum) {
        ctx.strokeStyle = '#26241d'; ctx.lineWidth = 2.5;
        ctx.beginPath();
        for (let i = 0; i < sim05fft_N; i++) {
          const x = i * W / sim05fft_N;
          const y = H/2 - sum[i] * H/3;
          i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
        }
        ctx.stroke();
      }

      // Espectro
      const SW = S.width, SH = S.height;
      sctx.clearRect(0, 0, SW, SH);
      sctx.fillStyle = '#fafaf7'; sctx.fillRect(0,0,SW,SH);
      const maxFreq = sim05fft_getTerms(sim05fft_targetType, 25)[24].freq;
      for (let k = 0; k < terms.length; k++) {
        const {freq, amp} = terms[k];
        const x = freq/maxFreq * SW;
        const h2 = amp / 2 * SH * 0.8;
        sctx.fillStyle = sim05fft_PALETTE[k % sim05fft_PALETTE.length];
        sctx.fillRect(x-2, SH-h2, 4, h2);
      }
      sctx.strokeStyle = '#e4dcc8'; sctx.lineWidth = 0.5;
      sctx.strokeRect(0,0,SW,SH);
      sctx.fillStyle = '#8a8371'; sctx.font = '9.5px monospace'; sctx.textAlign='left';
      sctx.fillText('Espectro de Amplitude (frequência →)', 6, 14);

      // Stats
      root.querySelector('#sim05fft_nTerms').textContent = sim05fft_nTerms;
      root.querySelector('#sim05fft_rms').textContent = rms.toFixed(3);

      // Term list
      const tl = root.querySelector('#sim05fft_termList');
      tl.innerHTML = '';
      for (let k = 0; k < Math.min(terms.length, 8); k++) {
        const {freq, amp} = terms[k];
        const d = document.createElement('span');
        d.style.cssText = 'font-size:10px; display:flex; align-items:center; gap:4px; background:#fafaf7; border:1px solid #e9e3d3; padding:4px 8px; border-radius:6px;';
        d.innerHTML = '<span style="width:10px;height:10px;border-radius:3px;background:' + sim05fft_PALETTE[k%sim05fft_PALETTE.length] + ';display:inline-block;"></span><span style="font-weight:700; color:#5e5a4a;">k=' + freq + ' A=' + amp.toFixed(2) + '</span>';
        tl.appendChild(d);
      }
    }

    window.sim05fft_setTarget = function(t) {
      sim05fft_targetType = t;
      ['sq','tr','sw'].forEach(id => {
        const el = root.querySelector('#sim05fft_' + id);
        if (el) el.classList.remove('sim05fft_active');
      });
      const map = {square:'sq', triangle:'tr', sawtooth:'sw'};
      root.querySelector('#sim05fft_' + map[t]).classList.add('sim05fft_active');
      root.querySelector('#sim05fft_target').textContent = {square:'quadrada',triangle:'triangular',sawtooth:'dente-serra'}[t];
      sim05fft_draw();
    };

    window.sim05fft_toggleComp = function() {
      sim05fft_showComp = !sim05fft_showComp;
      root.querySelector('#sim05fft_chk_comp').classList.toggle('sim05fft_active', sim05fft_showComp);
      sim05fft_draw();
    };
    
    window.sim05fft_toggleSum = function() {
      sim05fft_showSum = !sim05fft_showSum;
      root.querySelector('#sim05fft_chk_sum').classList.toggle('sim05fft_active', sim05fft_showSum);
      sim05fft_draw();
    };

    root.querySelector('#sim05fft_slider').addEventListener('input', function(){
      sim05fft_nTerms = +this.value;
      root.querySelector('#sim05fft_slVal').textContent = sim05fft_nTerms;
      sim05fft_draw();
    });

    sim05fft_draw();
  }

  function tryInitSim05FFT(){
    var root = document.getElementById('sim-05-freq');
    if (root) initSim05FFT(root); else setTimeout(tryInitSim05FFT, 200);
  }
  tryInitSim05FFT();
})();
</script>
""")

**Figura 5.2:** Simulatore interattivo della decomposizione di Fourier 1D: visualizzazione della somma di sinusoidi con diverse frequenze, ampiezze e fasi. Aggiungi termini e osserva la convergenza verso forme d


<figure id="fig-05-sim-05-freq">
  <img src="imagens/fig-05-sim-05-freq.png" alt=" Simulatore interattivo della decomposizione di Fourier 1D: visualizzazione della somma di sinusoidi con diverse frequenze, ampiezze e fasi. Aggiungi termini e osserva la convergenza verso forme d'onda arbitrarie. " style="max-width:80%" />
  <figcaption><strong>Figura 5.2:</strong>  Simulatore interattivo della decomposizione di Fourier 1D: visualizzazione della somma di sinusoidi con diverse frequenze, ampiezze e fasi. Aggiungi termini e osserva la convergenza verso forme d'onda arbitrarie. </figcaption>
</figure>

### 5.3.2 Interpretazione dello spettro di frequenza

Applicando la Trasformata Discreta di Fourier (DFT) a un'immagine e visualizzando il modulo dei suoi coefficienti (vedi [Figura 5.5](#fig-05-espectro-conceitual)), si ottiene lo **spettro di ampiezza**, che mostra la distribuzione delle frequenze spaziali presenti nell'immagine.

Il coefficiente situato all'origine della DFT, denominato **componente DC** (*Direct Current*), corrisponde alla frequenza nulla e rappresenta l'intensità media dell'immagine. Per convenzione, tale coefficiente è memorizzato nell'angolo superiore sinistro dello spettro. Per facilitarne l'interpretazione, si applica l'operazione **FFT Shift**, che sposta la componente DC al centro dell'immagine. Dopo questo spostamento, le basse frequenze si concentrano nella regione centrale, mentre le alte frequenze si trovano in prossimità dei bordi, come riassunto nella [Tabela 5.1](#tbl-05-espectro-regioes).

<a id="tbl-05-espectro-regioes"></a>

**Tabela 5.1:** Corrispondenza tra le regioni dello spettro di ampiezza dopo l'applicazione dell'FFT Shift.

| Regione dello spettro | Componenti predominanti | Esempi nell'immagine |
|:---|:---|:---|
| **Centro** (basse frequenze) | Variazioni spaziali lente | Illuminazione, regioni omogenee e forme globali |
| **Regione intermedia** (medie frequenze) | Variazioni di scala intermedia | Trame e pattern ripetitivi |
| **Bordi** (alte frequenze) | Variazioni spaziali rapide | Contorni, dettagli fini e rumore |


Questa organizzazione facilita l'interpretazione dello spettro e la progettazione di filtri. L'attenuazione delle basse frequenze riduce le variazioni globali di intensità, mentre l'attenuazione delle alte frequenze smussa l'immagine riducendo i dettagli fini e parte del rumore.

### 5.3.3 L'Esperimento della Griglia: Costruire un'Immagine da un Singolo Coefficiente

Prima di presentare la formulazione matematica della Trasformata Discreta di Fourier (DFT), è utile analizzare la sua inversa, denominata Trasformata Discreta Inversa di Fourier (IDFT). Si consideri uno spettro in cui tutti i coefficienti siano nulli, eccetto uno. Un esempio di questa costruzione è presentato nel codice della [Figura 5.3](#fig-05-grade-2d) e può essere esplorato interattivamente nel simulatore della [Figura 5.4](#fig-05-sim-05-grade-2d)..

L'immagine ricostruita è una sinusoide bidimensionale. La posizione del coefficiente nello spettro determina la sua **orientazione** e la sua **frequenza spaziale**, mentre la sua magnitudine e la sua fase definiscono, rispettivamente, la sua ampiezza e il suo spostamento spaziale. Pertanto, ciascun coefficiente della DFT rappresenta una componente sinusoidale, e l'immagine originale può essere ricostruita mediante la somma di tutte queste componenti.

In [6]:
%%writefile tmp/fig_05_grade_2d.cpp
#define MM_OUT "tmp/fig_05_grade_2d.png"
//| label: fig-05-grade-2d
//| fig-cap: "Toda frequência no espectro (ponto isolado) corresponde a uma onda senoidal 2D rotacionada no domínio espacial."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <complex>
#include "morph.hpp"
#include <filesystem>

int main() {
    int N_grid = 100;
    cv::Mat espectro_vazio(N_grid, N_grid, CV_64FC2, cv::Scalar(0, 0));

    // Acendendo um único ponto (frequência) fora do centro
    int u0 = 10, v0 = 5;
    espectro_vazio.at<cv::Vec2d>(N_grid/2 - v0, N_grid/2 - u0) = cv::Vec2d(1000, 0);

    // Retornando para o domínio espacial (IDFT)
    cv::Mat espectro_shifted;
    cv::Mat planes_shifted[2];
    cv::split(espectro_vazio, planes_shifted);

    // Manual ifftshift (swap quadrants)
    cv::Mat re_shifted = planes_shifted[0].clone();
    cv::Mat im_shifted = planes_shifted[1].clone();
    int cx = N_grid / 2;
    int cy = N_grid / 2;
    cv::Mat q0_re(re_shifted, cv::Rect(0, 0, cx, cy));
    cv::Mat q1_re(re_shifted, cv::Rect(cx, 0, cx, cy));
    cv::Mat q2_re(re_shifted, cv::Rect(0, cy, cx, cy));
    cv::Mat q3_re(re_shifted, cv::Rect(cx, cy, cx, cy));
    cv::Mat tmp_re;
    q0_re.copyTo(tmp_re);
    q3_re.copyTo(q0_re);
    tmp_re.copyTo(q3_re);
    q1_re.copyTo(tmp_re);
    q2_re.copyTo(q1_re);
    tmp_re.copyTo(q2_re);

    cv::Mat q0_im(im_shifted, cv::Rect(0, 0, cx, cy));
    cv::Mat q1_im(im_shifted, cv::Rect(cx, 0, cx, cy));
    cv::Mat q2_im(im_shifted, cv::Rect(0, cy, cx, cy));
    cv::Mat q3_im(im_shifted, cv::Rect(cx, cy, cx, cy));
    cv::Mat tmp_im;
    q0_im.copyTo(tmp_im);
    q3_im.copyTo(q0_im);
    tmp_im.copyTo(q3_im);
    q1_im.copyTo(tmp_im);
    q2_im.copyTo(q1_im);
    tmp_im.copyTo(q2_im);

    std::vector<cv::Mat> planes_shifted_vec = {re_shifted, im_shifted};
    cv::Mat espectro_shifted_complex;
    cv::merge(planes_shifted_vec, espectro_shifted_complex);

    cv::Mat onda_2d_complex;
    cv::idft(espectro_shifted_complex, onda_2d_complex, cv::DFT_SCALE | cv::DFT_COMPLEX_OUTPUT);

    std::vector<cv::Mat> onda_planes;
    cv::split(onda_2d_complex, onda_planes);
    cv::Mat onda_2d = onda_planes[0];  // parte real

    cv::Mat onda_vis;
    cv::normalize(onda_2d, onda_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    cv::Mat espectro_mag;
    cv::magnitude(planes_shifted[0], planes_shifted[1], espectro_mag);
    cv::Mat espectro_vis;
    cv::normalize(espectro_mag, espectro_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Destaque visual do ponto
    cv::Mat espectro_color;
    cv::cvtColor(espectro_vis, espectro_color, cv::COLOR_GRAY2BGR);
    cv::circle(espectro_color, cv::Point(N_grid/2 - u0, N_grid/2 - v0), 2, cv::Scalar(0, 0, 255), -1);

    mm::Image img1(espectro_color);
    mm::Image img2(onda_vis);
    std::vector<mm::Image> imgs = {img1, img2};
    std::vector<std::string> titles = {"Espectro (1 ponto ativo)", "Onda 2D Resultante (IDFT)"};
    mm::show(imgs, MM_OUT, titles, 2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(espectro_color, "tmp/fig_05_grade_2d_0.png");
mm::write(onda_vis, "tmp/fig_05_grade_2d_1.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_grade_2d.cpp


In [7]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_grade_2d.cpp -o tmp/fig_05_grade_2d -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_grade_2d \
  && test -f "tmp/fig_05_grade_2d.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_grade_2d.png"

[1] Espectro (1 ponto ativo)
[2] Onda 2D Resultante (IDFT)


In [8]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_grade_2d_0.png"),
            mm.read("tmp/fig_05_grade_2d_1.png"),
        ],
        titles=[
            'Espectro (1 ponto ativo)',
            'Onda 2D Resultante (IDFT)',
        ],
        cols=2,
        figsize=(10, 4),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_grade_2d_0.png (ver a versao Python)")

<Figure size 1500x600 with 2 Axes>

**Figura 5.3:** Toda frequência no espectro (ponto isolado) corresponde a uma onda senoidal 2D rotacionada no domínio espacial.


In [9]:
# @title { display-mode: "form" }
from IPython.display import HTML

HTML("""
<div id="sim-05-grade-2d" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-grade-2d * { box-sizing: border-box; }
  #sim-05-grade-2d canvas { display: block; background: #ffffff; border: 1px solid #e4dcc8; border-radius: 8px; }
  #sim-05-grade-2d button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-grade-2d button:hover { background: #e8dfcf; }
  .sim-05-grade-2d_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-05-grade-2d_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-05-grade-2d_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 80px; }
  .sim-05-grade-2d_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-05-grade-2d_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-05-grade-2d_slider_container { display: flex; align-items: center; gap: 8px; margin-bottom: 8px; }
  .sim-05-grade-2d_slider_container label { font-size: 11px; font-weight: 700; min-width: 120px; display: inline-block; color: #5e5a4a; }
  .sim-05-grade-2d_slider_container input[type=range] { flex: 1; cursor: pointer; height: 4px; }
  .sim-05-grade-2d_slider_val { font-size: 12px; font-family: monospace; font-weight: 700; min-width: 25px; text-align: right; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">∿ Simulatore: Sintesi di Frequenza 2D (IDFT)</span>
  <span class="sim-05-grade-2d_pill">Spazio di Fourier</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Frequenza u</div><div id="sim-05-grade-2d_valU" class="sim-05-grade-2d_stat_value" style="color:#2980b9;">10</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Frequenza v</div><div id="sim-05-grade-2d_valV" class="sim-05-grade-2d_stat_value" style="color:#27ae60;">5</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Distanza R</div><div id="sim-05-grade-2d_valR" class="sim-05-grade-2d_stat_value" style="color:#b9770e;">11.18</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Angolo θ</div><div id="sim-05-grade-2d_valAng" class="sim-05-grade-2d_stat_value" style="color:#c0392b;">26.6°</div></div>
  </div>

  <!-- Exibição Central (Espectro e Espaço) -->
  <div style="display: flex; gap: 16px; justify-content: center; align-items: center; margin-bottom: 14px; flex-wrap: wrap;">
    <div style="text-align: center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
      <div style="font-size: 10.5px; font-weight: 700; color: #5e5a4a; margin-bottom: 8px;">Spettro (Clicca per spostare il punto)</div>
      <canvas id="sim-05-grade-2d_CanvasSpec" width="220" height="220" style="margin:0 auto;"></canvas>
    </div>
    <div style="font-size: 20px; color: #8a8371; font-weight: bold;">➔</div>
    <div style="text-align: center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
      <div style="font-size: 10.5px; font-weight: 700; color: #5e5a4a; margin-bottom: 8px;">Onda 2D risultante (Dominio Spaziale)</div>
      <canvas id="sim-05-grade-2d_CanvasSpace" width="220" height="220" style="margin:0 auto;"></canvas>
    </div>
  </div>

  <!-- Controles Deslizantes -->
  <div class="sim-05-grade-2d_panel">
    <div class="sim-05-grade-2d_slider_container">
      <label style="color:#2980b9;">Spostamento u (X):</label>
      <input type="range" id="sim-05-grade-2d_sliderU" min="-30" max="30" value="10">
      <span id="sim-05-grade-2d_slValU" class="sim-05-grade-2d_slider_val">10</span>
    </div>
    <div class="sim-05-grade-2d_slider_container" style="margin-bottom:0;">
      <label style="color:#27ae60;">Spostamento v (Y):</label>
      <input type="range" id="sim-05-grade-2d_sliderV" min="-30" max="30" value="5">
      <span id="sim-05-grade-2d_slValV" class="sim-05-grade-2d_slider_val">5</span>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Exp2D(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const specC = root.querySelector('#sim-05-grade-2d_CanvasSpec');
    const spaceC = root.querySelector('#sim-05-grade-2d_CanvasSpace');
    const sctx = specC.getContext('2d');
    const spctx = spaceC.getContext('2d');

    let u = 10;
    let v = 5;
    const N = 220;

    function updateMetrics() {
      const r = Math.sqrt(u*u + v*v);
      let angle = Math.atan2(v, u) * (180 / Math.PI);
      if (angle < 0) angle += 360;

      root.querySelector('#sim-05-grade-2d_valU').textContent = u;
      root.querySelector('#sim-05-grade-2d_valV').textContent = v;
      root.querySelector('#sim-05-grade-2d_valR').textContent = r.toFixed(2);
      root.querySelector('#sim-05-grade-2d_valAng').textContent = angle.toFixed(1) + '°';

      root.querySelector('#sim-05-grade-2d_sliderU').value = u;
      root.querySelector('#sim-05-grade-2d_sliderV').value = v;
      root.querySelector('#sim-05-grade-2d_slValU').textContent = u;
      root.querySelector('#sim-05-grade-2d_slValV').textContent = v;
    }

    function render() {
      updateMetrics();

      // 1. Desenhar Espectro
      sctx.fillStyle = '#fafaf7';
      sctx.fillRect(0, 0, N, N);

      sctx.strokeStyle = '#e4dcc8';
      sctx.lineWidth = 1;
      sctx.beginPath();
      sctx.moveTo(N/2, 0); sctx.lineTo(N/2, N);
      sctx.moveTo(0, N/2); sctx.lineTo(N, N/2);
      sctx.stroke();

      sctx.fillStyle = '#8a8371';
      sctx.beginPath();
      sctx.arc(N/2, N/2, 2.5, 0, 2*Math.PI);
      sctx.fill();

      let ptX = N/2 + u;
      let ptY = N/2 - v;

      sctx.strokeStyle = '#b9770e88';
      sctx.lineWidth = 1.5;
      sctx.beginPath();
      sctx.moveTo(N/2, N/2);
      sctx.lineTo(ptX, ptY);
      sctx.stroke();

      let symX = N/2 - u;
      let symY = N/2 + v;
      sctx.fillStyle = '#c0392b88';
      sctx.beginPath();
      sctx.arc(symX, symY, 4, 0, 2*Math.PI);
      sctx.fill();

      sctx.fillStyle = '#2980b9';
      sctx.strokeStyle = '#ffffff';
      sctx.lineWidth = 1.5;
      sctx.beginPath();
      sctx.arc(ptX, ptY, 5.5, 0, 2*Math.PI);
      sctx.fill();
      sctx.stroke();

      // 2. Desenhar Onda Espacial 2D Resultante
      const imgData = spctx.createImageData(N, N);
      const data = imgData.data;
      const freqScale = 2 * Math.PI / N;

      for (let y = 0; y < N; y++) {
        const ny = y - N/2;
        for (let x = 0; x < N; x++) {
          const nx = x - N/2;
          const val = Math.cos(freqScale * (u * nx + v * (-ny)));
          const intensity = Math.floor((val + 1) * 127.5);

          const idx = (y * N + x) * 4;
          data[idx]     = intensity;
          data[idx + 1] = intensity;
          data[idx + 2] = intensity;
          data[idx + 3] = 255;
        }
      }
      spctx.putImageData(imgData, 0, 0);
    }

    root.querySelector('#sim-05-grade-2d_sliderU').addEventListener('input', function() {
      u = parseInt(this.value, 10);
      render();
    });

    root.querySelector('#sim-05-grade-2d_sliderV').addEventListener('input', function() {
      v = parseInt(this.value, 10);
      render();
    });

    specC.addEventListener('mousedown', function(e) {
      const rect = specC.getBoundingClientRect();
      const clickX = e.clientX - rect.left;
      const clickY = e.clientY - rect.top;

      let newU = Math.round(clickX - N/2);
      let newV = Math.round(N/2 - clickY);

      u = Math.max(-30, Math.min(30, newU));
      v = Math.max(-30, Math.min(30, newV));

      render();
    });

    render();
  }

  function tryInitSim05Exp2D(){
    var root = document.getElementById('sim-05-grade-2d');
    if (root) initSim05Exp2D(root); else setTimeout(tryInitSim05Exp2D, 200);
  }
  tryInitSim05Exp2D();
})();
</script>
""")

**Figura 5.4:** Simulatore interattivo della sintesi di Fourier 2D. Modifica la posizione orizzontale ($u$) e verticale ($v$) del coefficiente nello spettro di frequenze centrato e osserva come la distanza dal centro determina la frequenza spaziale (spessore) e l


<figure id="fig-05-sim-05-grade-2d">
  <img src="imagens/fig-05-sim-05-grade-2d.png" alt=" Simulatore interattivo della sintesi di Fourier 2D. Modifica la posizione orizzontale ($u$) e verticale ($v$) del coefficiente nello spettro di frequenze centrato e osserva come la distanza dal centro determina la frequenza spaziale (spessore) e l'angolo determina l'orientamento dell'onda sinusoidale generata. " style="max-width:80%" />
  <figcaption><strong>Figura 5.4:</strong>  Simulatore interattivo della sintesi di Fourier 2D. Modifica la posizione orizzontale ($u$) e verticale ($v$) del coefficiente nello spettro di frequenze centrato e osserva come la distanza dal centro determina la frequenza spaziale (spessore) e l'angolo determina l'orientamento dell'onda sinusoidale generata. </figcaption>
</figure>

### 5.3.4 Definizione Matematica

Si consideri un'immagine $f(x,y)$ con dimensioni $M \times N$. La sua **Trasformata Discreta di Fourier 2D** (DFT) è definita da:

<a id="eq-05-dft"></a>
$$
F(u,v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)} \tag{5.1}
$$


dove $u = 0, 1, \ldots, M-1$ e $v = 0, 1, \ldots, N-1$ rappresentano le frequenze discrete nelle direzioni orizzontale e verticale, rispettivamente. Il termine esponenziale corrisponde a una sinusoide bidimensionale, la cui frequenza e orientazione sono determinate dagli indici $(u,v)$.

La **Trasformata Discreta Inversa di Fourier 2D** (IDFT) ricostruisce l'immagine originale a partire dai suoi coefficienti:

<a id="eq-05-idft"></a>
$$
f(x,y) = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} F(u,v)\, e^{j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)} \tag{5.2}
$$


Le Equazioni [Equação 5.1](#eq-05-dft) e [Equação 5.2](#eq-05-idft) mostrano che la DFT e la IDFT formano una coppia di trasformazioni: la prima converte l'immagine nel dominio della frequenza, mentre la seconda ricostruisce esattamente l'immagine originale a partire dai suoi coefficienti.

> ### 📝 5.3.4.1 Sul simbolo $j$
>
> Il termine $j$ denota l'**unità immaginaria**, definita da $j^2 = -1$. In ingegneria e nell'elaborazione dei segnali, si adotta $j$ invece di $i$ per evitare conflitti con la notazione della corrente elettrica. Il suo utilizzo nell'esponenziale complessa, governato dalla formula di Eulero ($e^{j\theta} = \cos\theta + j\sin\theta$), consente di rappresentare in modo compatto l'ampiezza e la fase di ciascuna frequenza spaziale presente nell'immagine.

> ### 📝 Che cos'è il componente DC?
>
> Il coefficiente $F(0,0)$, denominato **componente DC** (*Direct Current*), è uguale alla somma delle intensità di tutti i pixel dell'immagine (vedi [Figura 5.5](#fig-05-espectro-conceitual)):
>
> $$
> F(0,0)=MN\,\bar{f},
> $$
>
> dove $\bar{f}$ è l'intensità media dell'immagine. Per questo motivo, il componente DC rappresenta il livello medio di intensità e, nella maggior parte delle immagini naturali, possiede la maggiore ampiezza dello spettro.
>
> Gli altri coefficienti rappresentano variazioni attorno a tale media. Dopo l'applicazione del **FFT Shift**, il componente DC viene spostato al centro dello spettro, concentrando le basse frequenze nella regione centrale e le alte frequenze ai bordi.

In [10]:
# @title { display-mode: "form" }
from IPython.display import HTML

HTML("""
<div style="font-family:sans-serif; max-width:880px; margin:0 auto; padding:10px;">
<div style="text-align:center; font-size:12px; font-weight:bold; color:#374151; margin-bottom:8px;">
  Anatomia dello Spettro di Fourier 2D (dopo fftshift)
</div>
<svg viewBox="0 0 640 320" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#f9fafb;">
  <!-- Fundo gradiente radial simulado -->
  <defs>
    <radialGradient id="specGrad" cx="50%" cy="50%" r="50%">
      <stop offset="0%" style="stop-color:#1e3a5f;stop-opacity:1"/>
      <stop offset="20%" style="stop-color:#1a5276;stop-opacity:1"/>
      <stop offset="50%" style="stop-color:#0d2137;stop-opacity:1"/>
      <stop offset="100%" style="stop-color:#050e1a;stop-opacity:1"/>
    </radialGradient>
    <radialGradient id="brightCenter" cx="50%" cy="50%" r="15%">
      <stop offset="0%" style="stop-color:#ffffff;stop-opacity:1"/>
      <stop offset="60%" style="stop-color:#f0c040;stop-opacity:0.9"/>
      <stop offset="100%" style="stop-color:#1a5276;stop-opacity:0"/>
    </radialGradient>
  </defs>
  <rect x="20" y="10" width="380" height="300" fill="url(#specGrad)" rx="6"/>
  <rect x="20" y="10" width="380" height="300" fill="url(#brightCenter)" rx="6"/>
  <!-- Cruzes de alta energia (bordas horizontais/verticais) -->
  <line x1="210" y1="10" x2="210" y2="310" stroke="#4a9eda" stroke-width="1.5" opacity="0.4"/>
  <line x1="20" y1="160" x2="400" y2="160" stroke="#4a9eda" stroke-width="1.5" opacity="0.4"/>
  <!-- Círculos de frequência -->
  <circle cx="210" cy="160" r="30" fill="none" stroke="#f0c040" stroke-width="1" stroke-dasharray="4,3" opacity="0.7"/>
  <circle cx="210" cy="160" r="70" fill="none" stroke="#7dd3fc" stroke-width="1" stroke-dasharray="4,3" opacity="0.5"/>
  <circle cx="210" cy="160" r="120" fill="none" stroke="#93c5fd" stroke-width="0.8" stroke-dasharray="4,3" opacity="0.3"/>
  <!-- Ponto DC -->
  <circle cx="210" cy="160" r="6" fill="#ffffff"/>
  <!-- Rótulos no espectro -->
  <text x="210" y="148" font-size="9" fill="#fff" text-anchor="middle" font-weight="bold">DC</text>
  <text x="210" y="205" font-size="8" fill="#f0c040" text-anchor="middle">basse freq.</text>
  <text x="210" y="245" font-size="8" fill="#7dd3fc" text-anchor="middle">medie freq.</text>
  <text x="330" y="110" font-size="8" fill="#93c5fd" text-anchor="middle">alte freq.</text>
  <text x="210" y="295" font-size="9" fill="#cbd5e1" text-anchor="middle" font-style="italic">Spettro di Magnitudine |F(u,v)| — scala log</text>
  <!-- Painel direito: explicações -->
  <rect x="420" y="10" width="200" height="300" fill="#ffffff" rx="6" stroke="#e5e7eb"/>
  <text x="520" y="35" font-size="10" fill="#1e293b" text-anchor="middle" font-weight="bold">Regioni dello Spettro</text>
  <!-- DC -->
  <circle cx="440" cy="65" r="7" fill="#ffffff" stroke="#f0c040" stroke-width="2"/>
  <text x="455" y="61" font-size="9" fill="#374151" font-weight="bold">DC (0,0)</text>
  <text x="455" y="73" font-size="8" fill="#6b7280">Media globale dei pixel</text>
  <!-- Baixas -->
  <rect x="433" y="95" width="14" height="14" rx="2" fill="#f0c040" opacity="0.7"/>
  <text x="455" y="105" font-size="9" fill="#374151" font-weight="bold">Basse frequenze</text>
  <text x="455" y="116" font-size="8" fill="#6b7280">Forma, sfondo, illuminazione</text>
  <!-- Médias -->
  <rect x="433" y="135" width="14" height="14" rx="2" fill="#7dd3fc" opacity="0.7"/>
  <text x="455" y="145" font-size="9" fill="#374151" font-weight="bold">Medie frequenze</text>
  <text x="455" y="156" font-size="8" fill="#6b7280">Texture, pattern</text>
  <!-- Altas -->
  <rect x="433" y="175" width="14" height="14" rx="2" fill="#1e3a5f" stroke="#93c5fd" stroke-width="1"/>
  <text x="455" y="185" font-size="9" fill="#374151" font-weight="bold">Alte frequenze</text>
  <text x="455" y="196" font-size="8" fill="#6b7280">Bordi, rumore, dettagli</text>
  <!-- Seta de eixos -->
  <text x="440" y="235" font-size="8" fill="#6b7280">u → freq. orizzontale</text>
  <text x="440" y="248" font-size="8" fill="#6b7280">v → freq. verticale</text>
  <line x1="440" y1="265" x2="600" y2="265" stroke="#d1d5db" stroke-width="0.8"/>
  <text x="520" y="280" font-size="8" fill="#9ca3af" text-anchor="middle">Visualizzazione in scala log</text>
  <text x="520" y="292" font-size="8" fill="#9ca3af" text-anchor="middle">log(1 + |F|) comprime l'intervallo</text>
</svg>
</div>
""")

**Figura 5.5:** Diagramma concettuale dello spettro di Fourier 2D centrato.


<figure id="fig-05-espectro-conceitual">
  <img src="imagens/fig-05-espectro-conceitual.png" alt=" Diagramma concettuale dello spettro di Fourier 2D centrato. " style="max-width:80%" />
  <figcaption><strong>Figura 5.5:</strong>  Diagramma concettuale dello spettro di Fourier 2D centrato. </figcaption>
</figure>

### 5.3.5 Magnitudine e Fase

Ogni coefficiente della Trasformata Discreta di Fourier (DFT) è un numero complesso e può essere scritto come

$$
F(u,v)=R(u,v)+j\,I(u,v),
$$

dove $R(u,v)$ e $I(u,v)$ corrispondono, rispettivamente, alle parti **reale** e **immaginaria** del coefficiente. Dall'Equazione [Equação 5.1](#eq-05-dft) si ottiene

$$
R(u,v)=
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)
\cos\!\left(
2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)
\right),
$$

e

$$
I(u,v)=
-
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)
\sin\!\left(
2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)
\right).
$$

Da questa rappresentazione si definiscono due grandezze fondamentali:

- **Magnitudine**, che indica l'intensità della componente di frequenza,

$$
|F(u,v)|=\sqrt{R(u,v)^2+I(u,v)^2};
$$

- **Fase**, che determina l'allineamento (o lo spostamento) spaziale della componente,

$$
\phi(u,v)=\operatorname{atan2}\!\left(I(u,v),\,R(u,v)\right).
$$

Pertanto, ogni coefficiente può anche essere scritto nella sua forma polare,

$$
F(u,v)=|F(u,v)|\,e^{j\phi(u,v)}.
$$

Lo spettro di Fourier può quindi essere visualizzato mediante due immagini distinte: lo **spettro di magnitudine**, generalmente utilizzato per analizzare la distribuzione delle frequenze, e lo **spettro di fase**, che descrive l'organizzazione spaziale delle componenti sinusoidali.

Sebbene lo spettro di magnitudine sia il più utilizzato per l'ispezione visiva, la fase contiene gran parte delle informazioni strutturali dell'immagine. La combinazione di magnitudine e fase consente di ricostruire esattamente l'immagine originale tramite la IDFT.

### 5.3.6 Cosa trasportano l'ampiezza e la fase?

Una dimostrazione classica consiste nel combinare l'ampiezza di un'immagine con la fase di un'altra e ricostruire il risultato. Questo esperimento evidenzia che:

- **La fase** preserva la struttura spaziale dell'immagine, inclusa la posizione degli oggetti, i loro contorni e la loro geometria. Piccole alterazioni nella fase possono provocare grandi cambiamenti visivi.
- **L'ampiezza** controlla come l'energia è distribuita tra le frequenze spaziali, influenzando principalmente il contrasto e la tessitura.

Quando un'immagine viene ricostruita con l'ampiezza di A e la fase di B, il risultato tende a **somigliare più a B che ad A**, evidenziando che la fase è il componente principale responsabile dell'organizzazione spaziale della scena. Tuttavia, l'ampiezza rimane importante, poiché modula il contrasto delle strutture ricostruite. Pertanto, una ricostruzione fedele dipende dalla combinazione coerente tra ampiezza e fase.

Un esempio di questo comportamento è presentato in [Figura 5.6](#fig-05-dft-intro).

> ### 📝 Analogia con l'Audio: Limitazioni e Precauzioni
>
> La fase di un segnale svolge ruoli distinti nell'audio e nelle immagini:
>
> - **Audio stereo o multicanale:** la fase relativa tra i canali è fondamentale per la percezione della posizione delle sorgenti sonore, tramite le differenze interaurali di tempo (ITD, *Interaural Time Differences*).
> - **Audio monaurale:** la fase assoluta esercita una scarsa influenza percettiva diretta.
> - **Immagini (DFT):** la fase è il fattore principale responsabile dell'organizzazione spaziale della scena, mentre l'ampiezza modula il contrasto e la distribuzione dell'energia tra le frequenze.
>
> In entrambi i domini, l'**ampiezza** è legata all'intensità delle componenti di frequenza: nell'audio, influenza il timbro e l'intensità percepita; nelle immagini, influenza il contrasto e la tessitura.

In [11]:
 
import numpy as np, cv2, os  # [pdi] passthrough: garante imports desta trilha
# ── Esperimento: L'Importanza della Fase ─────────────────────────────────────
# ── Caricamento dell'immagine ────────────────────────────────────────────────
url     = "https://upload.wikimedia.org/wikipedia/commons/2/25/GAZI.MD.AHAD_11.jpg"
caminho = "imagens/coins.jpg"

if not os.path.exists(caminho):
    os.makedirs("imagens", exist_ok=True)
    img_obj = mm.read(url, pil=True)
    mm.write(img_obj, caminho)
else:
    img_obj = mm.read(caminho, pil=True)

img_color = np.array(img_obj)
img_gray  = mm.gray(img_color)

img_a = cv2.resize(img_gray, (400, 400))

# Creare un'immagine B sintetica (motivo geometrico)
img_b = np.zeros((400, 400), dtype=np.uint8)
cv2.rectangle(img_b, (100, 100), (300, 300), 255, -1)
cv2.circle(img_b, (200, 200), 150, 128, 10)

FA = np.fft.fft2(img_a)
FB = np.fft.fft2(img_b)

# Scambio di Fase
rec_A_mag_B_fase = np.real(np.fft.ifft2(np.abs(FA) * np.exp(1j * np.angle(FB))))
rec_B_mag_A_fase = np.real(np.fft.ifft2(np.abs(FB) * np.exp(1j * np.angle(FA))))

mm.show(
    [img_a, img_b, rec_A_mag_B_fase, rec_B_mag_A_fase],
    titles=["Immagine A", "Immagine B", "Mag(A) + Fase(B)", "Mag(B) + Fase(A)"],
    cols=4, figsize=(16, 4)
)

print("💡 La fase preserva bordi e contorni; la magnitudine controlla contrasto e")
print("texture. Nell'audio stereo, la fase influisce sulla localizzazione spaziale; nelle")
print("immagini, determina l'organizzazione della scena.")
# [pdi:state-io] auto-gerado — não editar à mão
import os as _pdi_os; _pdi_os.makedirs("tmp/state", exist_ok=True)
mm.write(img_gray, "tmp/state/img_gray_20.png")
# [pdi:state-io:end]


<Figure size 2400x600 with 4 Axes>

**Figura 5.6:** Esperimento di cambio di fase: Immagine A (monete) e Immagine B (motivo geometrico) ricostruite con magnitudini e fasi scambiate. Il risultato mostra che la struttura visiva è **molto più sensibile alla fase** che alla magnitudine: quando la fase di B viene mantenuta, l


💡 La fase preserva bordi e contorni; la magnitudine controlla contrasto e
texture. Nell'audio stereo, la fase influisce sulla localizzazione spaziale; nelle
immagini, determina l'organizzazione della scena.


## 5.4 Teorema della Convoluzione e Strategie di Filtraggio

Il **Teorema della Convoluzione** stabilisce una relazione fondamentale tra il dominio spaziale e quello delle frequenze:

<a id="eq-05-conv-teorema"></a>
$$
f(x,y) \circledast h(x,y) \;\overset{\mathcal{F}}{\longleftrightarrow}\; F(u,v)\,H(u,v) \tag{5.3}
$$


dove $\circledast$ rappresenta la **convoluzione circolare discreta**. Pertanto, la convoluzione tra un'immagine $f(x,y)$ e un filtro $h(x,y)$ può essere sostituita dalla moltiplicazione dei loro spettri.

In pratica, per ottenere lo stesso risultato della convoluzione lineare eseguita nel dominio spaziale, si applica il **zero-padding** prima della Trasformata Rapida di Fourier (FFT), evitando artefatti ai bordi dell'immagine.

Tuttavia, la filtrazione nel dominio delle frequenze non è sempre l'alternativa più efficiente. Per filtri come quello gaussiano e il filtro della media (*Box Filter*), la proprietà di **separabilità** consente di ridurre significativamente il costo computazionale della convoluzione nel dominio spaziale.

### 5.4.1 *Kernel* Separabile vs. Non Separabile

Un ***kernel* separabile** può essere scritto come il prodotto esterno di due vettori unidimensionali,

$$
H = v\,h^T,
$$

consentendo di sostituire la convoluzione bidimensionale con due convoluzioni unidimensionali consecutive: una nella direzione orizzontale e una in quella verticale.

Un ***kernel* non separabile**, invece, non ammette tale decomposizione e, pertanto, la sua convoluzione deve essere eseguita direttamente sull'intorno bidimensionale.

In pratica, per un *kernel* di dimensione $K \times K$, la convoluzione diretta richiede $K^2$ moltiplicazioni per pixel, mentre un *kernel* separabile ne richiede solo $2K$, riducendo significativamente il costo computazionale.

### 5.4.2 Analisi dell'Efficienza Computazionale

Si consideri un'immagine di dimensioni $M \times N$ e un filtro quadrato di dimensione $K \times K$. La [Tabela 5.2](#tbl-05-fft-complexity-expanded) confronta la complessità delle principali strategie di filtraggio.

<a id="tbl-05-fft-complexity-expanded"></a>

**Tabela 5.2:** Confronto della complessità della convoluzione diretta, separabile e tramite Trasformata Rapida di Fourier (FFT).

| Metodo di filtraggio | Complessità asintotica | Dipendenza da $K$ | Applicazione tipica |
| --- | --- | --- | --- |
| **Spatiale non separabile** | $\mathcal{O}(MNK^2)$ | Quadratica | *Kernel* piccoli e non separabili |
| **Spatiale separabile** | $\mathcal{O}(MNK)$ | Lineare | Filtri Gaussiano e della media |
| **Tramite FFT** | $\mathcal{O}(MN\log(MN))$ | Indipendente da $K$ | *Kernel* grandi |


Per *kernel* piccoli, la convoluzione spaziale, specialmente quando il filtro è separabile, risulta solitamente più efficiente grazie al basso costo delle operazioni. All'aumentare della dimensione del *kernel*, il filtraggio tramite FFT diventa più vantaggioso, poiché il suo costo è praticamente indipendente dalla dimensione del filtro.

### 5.4.3 Discussione dei risultati sperimentali

Il grafico ottenuto nella prova con l'immagine delle monete ($2560 \times 1920$), presentato nella [Figura 5.7](#fig-05-conv-eficiencia), conferma il comportamento previsto dall'analisi della complessità computazionale.

1. **Convoluzione non separabile ($\mathcal{O}(MNK^2)$)**  
La convoluzione diretta presenta una crescita quadratica con la dimensione del *kernel*. Per valori piccoli di $K$, il costo è basso, ma aumenta rapidamente man mano che il *kernel* cresce, diventando impraticabile per applicazioni in tempo reale.

2. **Filtraggio tramite FFT ($\mathcal{O}(MN \log(MN))$)**  
Il costo della FFT dipende solo dalla dimensione dell'immagine, essendo indipendente da $K$. Pertanto, le sue prestazioni rimangono approssimativamente costanti al variare del *kernel*, rendendola vantaggiosa per filtri grandi o non separabili.

3. **Convoluzione separabile ($\mathcal{O}(MNK)$)**  
La scomposizione del *kernel* in due filtri monodimensionali riduce significativamente il costo computazionale. In pratica, questo approccio tende ad essere il più efficiente per filtri separabili, specialmente in implementazioni ottimizzate.

In generale, la scelta del metodo dipende dalla dimensione e dalla struttura del *kernel*. I filtri separabili sono più efficienti nel dominio spaziale, mentre la FFT diventa più vantaggiosa per *kernel* grandi o per convoluzioni multiple nel dominio della frequenza.

<a id="eq-05-filter-comparison"></a>
$$
g = \mathcal{F}^{-1}\bigl[\mathcal{F}(f)\cdot \mathcal{F}(h)\bigr]
\quad \text{(FFT)}
\qquad
g = f \circledast h
\quad \text{(convoluzione diretta)}
\qquad
g = (f \circledast v) \circledast h^T
\quad \text{(separabile)} \tag{5.4}
$$


dove:

* $f(x,y)$ rappresenta l'immagine di ingresso;
* $h(x,y)$ è il *kernel* bidimensionale del filtro;
* $v$ e $h^T$ sono, rispettivamente, i vettori verticale e orizzontale che compongono il *kernel* separabile.

In [12]:
%%writefile tmp/fig_05_conv_eficiencia.cpp
#define MM_OUT "tmp/fig_05_conv_eficiencia.png"
// Compile: g++ -std=c++17 -o program program.cpp $(pkg-config --cflags --libs opencv4)
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"
#include <filesystem>

#ifndef MM_OUT
#endif

int main() {
    // Trilha C++: curvas de CUSTO relativo (contagem de operacoes) — a forma das
    // curvas (K^2 vs 2K vs log N) e o que importa; a trilha py mede tempos reais.
    std::vector<double> K = {3, 7, 11, 15, 21, 31, 41, 51};
    double N2 = 256.0 * 256.0;

    std::vector<double> t_nao_sep;
    std::vector<double> t_sep;
    std::vector<double> t_fft;

    for (double k : K) {
        t_nao_sep.push_back(N2 * k * k / 1e6);
        t_sep.push_back(N2 * 2.0 * k / 1e6);
        t_fft.push_back(N2 * std::log2(N2) / 1e6);
    }

    std::vector<std::vector<double>> xs = {K, K, K};
    std::vector<std::vector<double>> ys = {t_nao_sep, t_sep, t_fft};
    std::vector<std::string> labels = {
        "Nao Separavel  O(N^2 K^2)", 
        "Separavel  O(N^2 . 2K)", 
        "Via FFT  O(N^2 log N)"
    };

    mm::Image chart = mm::lineChart(
        xs, ys, labels,
        {}, // cores vazias - usa padrão
        "Custo relativo: Espacial vs Frequencia",
        "Tamanho do kernel  K", 
        "Operacoes  (x10^6)"
    );

    mm::show(std::vector<mm::Image>{chart}, MM_OUT, 
             std::vector<std::string>{"Comparacao de complexidade"}, 1);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(chart, "tmp/fig_05_conv_eficiencia_0.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_conv_eficiencia.cpp


In [13]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_conv_eficiencia.cpp -o tmp/fig_05_conv_eficiencia -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_conv_eficiencia \
  && test -f "tmp/fig_05_conv_eficiencia.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_conv_eficiencia.png"

[1] Comparacao de complexidade


In [14]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_conv_eficiencia_0.png"),
        ],
        titles=[
            'Comparacao de complexidade',
        ],
        cols=1,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_conv_eficiencia_0.png (ver a versao Python)")

<Figure size 750x750 with 1 Axes>

**Figura 5.7:** Comparação de eficiência: Convolução Não Separável (Espacial 2D), Separável (Espacial 1D) e via FFT.


> ### ❗ 5.4.4 Il problema della convoluzione circolare (*wrap-around*)
>
> La Trasformata Discreta di Fourier (DFT) assume che l'immagine sia **estesa periodicamente nello spazio**, cioè che i suoi bordi si ripetano indefinitamente.
>
> In questa condizione, la moltiplicazione nel dominio della frequenza corrisponde a una **convoluzione circolare** nel dominio spaziale. Di conseguenza, regioni opposte dell'immagine (alto e basso, sinistra e destra) iniziano a interagire artificialmente, come illustrato nella [Figura 5.8](#fig-05-padding-error)..
>
> L'applicazione del *zero-padding* prima della FFT riduce questo effetto estendendo l'immagine con valori nulli ai bordi, avvicinando il risultato alla convoluzione lineare. Questo comportamento può essere interpretato alla luce del Teorema della Convoluzione, presentato nella [Figura 5.9](#fig-05-conv-teorema)..

In [15]:
%%writefile tmp/fig_05_padding_error.cpp
#define MM_OUT "tmp/fig_05_padding_error.png"
//| label: fig-05-padding-error
//| fig-cap: "Sem *padding*, um deslocamento severo faz a imagem vazar para o lado oposto (convolução circular)."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <complex>
#include <cmath>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    // ── Definir M e N ─────────────────────────────────────────────────────────────
    int M = img_gray.h;  // linha adicionada
    int N = img_gray.w;

    // Simulação de um filtro de deslocamento brutal
    cv::Mat H_shift(M, N, CV_64FC2, cv::Scalar(0, 0));
    for (int u = 0; u < M; u++) {
        for (int v = 0; v < N; v++) {
            double phase = -2.0 * M_PI * (u * 120.0 / M + v * 120.0 / N);
            H_shift.at<cv::Vec2d>(u, v) = {std::cos(phase), std::sin(phase)};
        }
    }

    // Filtragem SEM padding (causa o wrap-around)
    // F_img = np.fft.fft2(img_gray) → FFT da imagem em ponto flutuante
    cv::Mat img_gray_f;
    cv::Mat img_gray_cv = img_gray;  // conversão implícita
    img_gray_cv.convertTo(img_gray_f, CV_64F);
    cv::Mat F_img;
    cv::dft(img_gray_f, F_img, cv::DFT_COMPLEX_OUTPUT);

    // Multiplicação no domínio da frequência
    cv::Mat F_product;
    cv::mulSpectrums(F_img, H_shift, F_product, 0);

    // Inversa com escala e saída real
    cv::Mat img_vazada;
    cv::idft(F_product, img_vazada, cv::DFT_SCALE | cv::DFT_REAL_OUTPUT);

    // Normalização para visualização
    cv::Mat img_vazada_vis;
    cv::normalize(img_vazada, img_vazada_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Exibição
    mm::Image img_vazada_vis_img = img_vazada_vis;
    mm::show({img_gray, img_vazada_vis_img},
             MM_OUT,
             {"Original", "Filtragem s/ Padding (Vazamento)"}, 2);
    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_05_padding_error_0.png");
mm::write(img_vazada_vis, "tmp/fig_05_padding_error_1.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_padding_error.cpp


In [16]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_padding_error.cpp -o tmp/fig_05_padding_error -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_padding_error \
  && test -f "tmp/fig_05_padding_error.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_padding_error.png"

[1] Original
[2] Filtragem s/ Padding (Vazamento)


In [17]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_padding_error_0.png"),
            mm.read("tmp/fig_05_padding_error_1.png"),
        ],
        titles=[
            'Original',
            'Filtragem s/ Padding (Vazamento)',
        ],
        cols=2,
        figsize=(10, 4),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_padding_error_0.png (ver a versao Python)")

<Figure size 1500x600 with 2 Axes>

**Figura 5.8:** Sem *padding*, um deslocamento severo faz a imagem vazar para o lado oposto (convolução circular).


In [18]:
%%writefile tmp/fig_05_conv_teorema.cpp
#define MM_OUT "tmp/fig_05_conv_teorema.png"
//| label: fig-05-conv-teorema
//| fig-cap: "Teorema della Convoluzione: filtrare nel dominio spaziale (Gaussiana) equivale a moltiplicare lo spettro per $H(u,v)$ in frequenza. Le uscite coincidono visivamente."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    // Stessa sfocatura attraverso due percorsi: spazio vs. frequenza.
    int M = img_gray.h;
    int N = img_gray.w;

    cv::Mat H = mm::gaussFilter(M, N, 30);      // H(u,v): trasferimento passa-basso gaussiano
    mm::Image f_freq = mm::freqFilter(img_gray, H);    // convoluzione tramite moltiplicazione in frequenza

    // Applica la stessa sfocatura spaziale (Gaussiana 31x31 con sigma=5)
    cv::Mat img_mat(img_gray.h, img_gray.w, CV_8UC1, img_gray.data.data());
    cv::Mat blurred;
    cv::GaussianBlur(img_mat, blurred, cv::Size(31, 31), 5);
    mm::Image f_esp(blurred);

    mm::show(
        std::vector<mm::Image>{img_gray, f_esp, f_freq},
        MM_OUT,
        {"Originale", "Spazio: Gaussiana", "Frequenza: H(u,v).F(u,v)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_05_conv_teorema_0.png");
mm::write(f_esp, "tmp/fig_05_conv_teorema_1.png");
mm::write(f_freq, "tmp/fig_05_conv_teorema_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_conv_teorema.cpp


In [19]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_conv_teorema.cpp -o tmp/fig_05_conv_teorema -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_conv_teorema \
  && test -f "tmp/fig_05_conv_teorema.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_conv_teorema.png"

[1] Originale
[2] Spazio: Gaussiana
[3] Frequenza: H(u,v).F(u,v)


In [20]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_conv_teorema_0.png"),
            mm.read("tmp/fig_05_conv_teorema_1.png"),
            mm.read("tmp/fig_05_conv_teorema_2.png"),
        ],
        titles=[
            'Original',
            'Espaco: Gaussiana',
            'Frequencia: H(u,v).F(u,v)',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_conv_teorema_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figura 5.9:** Teorema da Convolução: filtrar no domínio do espaço (Gaussiana) equivale a multiplicar o espectro por $H(u,v)$ na frequência. As saídas coincidem visualmente.


> ### 📝 5.5 Sulla differenza numerica
>
> La differenza residua dell'ordine di $10^{-13}$ non viola il Teorema della Convoluzione, ma riflette **limitazioni computazionali** inerenti all'aritmetica a virgola mobile (doppia precisione, ~$10^{-16}$) e all'**ordine delle operazioni** tra i due metodi:
>
> - **Convoluzione spaziale:** somma ponderata di vicini con arrotondamenti successivi.
> - **Convoluzione in frequenza:** coinvolge tre trasformate FFT e una moltiplicazione complessa, soggetta a errori di troncamento e quantizzazione.
>
> Pertanto, l'uguaglianza teorica è esatta, ma l'implementazione numerica produce una differenza praticamente nulla (errore relativo < $10^{-12}$), confermando il teorema entro la precisione della macchina.

## 5.6 Filtri nel Dominio delle Frequenze

Un filtro nel dominio delle frequenze può essere interpretato come una **funzione di trasferimento applicata allo spettro dell'immagine**. In questa rappresentazione, ogni coefficiente di frequenza viene moltiplicato per un valore compreso tra 0 e 1, che determina la sua attenuazione o preservazione. La forma di questa funzione definisce l'effetto visivo del filtro.

**Taglio netto e *ringing*.** I filtri ideali con transizione istantanea a una frequenza di taglio $D_0$ producono discontinuità nel dominio delle frequenze. Questa discontinuità si riflette nel dominio spaziale come oscillazioni in prossimità dei bordi, note come *ringing*. Questo effetto è associato alla convoluzione con funzioni di supporto infinito nello spazio, come la funzione *sinc*, come illustrato nella [Figura 5.10](#fig-05-conv-teorema-zoom).

**Filtri con transizione graduale.** Alternative come i filtri Gaussiano e Butterworth attenuano la transizione tra regioni di passaggio e di reiezione, riducendo il *ringing*. Di contro, tale attenuazione implica un confine di separazione meno definito tra frequenze preservate e attenuate.

In [21]:
%%writefile tmp/fig_05_conv_teorema_zoom.cpp
#define MM_OUT "tmp/fig_05_conv_teorema_zoom.png"
//| label: fig-05-conv-teorema-zoom
//| fig-cap: "A Dualidade Perigosa: o corte abrupto na Frequência (cilindro Ideal) vira obrigatoriamente uma *sinc* no espaço. Suas ondulações causam o *ringing* fantasma nas bordas da imagem."
//| echo: true
//| output: true

// Filtro Ideal na frequência (cilindro) e sua resposta espacial (sinc 2D).
#include <opencv2/opencv.hpp>
#include "morph.hpp"
#include <vector>
#include <string>
#include <filesystem>

int main() {
    int N = 128;
    cv::Mat H_freq = mm::idealFilter(N, N, 20);   // 1 dentro do raio 20, 0 fora
    mm::Image h_space = mm::spatialKernel(H_freq); // ifft2(H) -> ondulações da sinc (ringing)

    mm::show(
        {H_freq, h_space},
        MM_OUT,
        {
            "Frequencia: filtro Ideal (cilindro)",
            "Espaco: ondulacoes da sinc (causa do ringing)",
        },
        2
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(H_freq, "tmp/fig_05_conv_teorema_zoom_0.png");
mm::write(h_space, "tmp/fig_05_conv_teorema_zoom_1.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_conv_teorema_zoom.cpp


In [22]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_conv_teorema_zoom.cpp -o tmp/fig_05_conv_teorema_zoom -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_conv_teorema_zoom \
  && test -f "tmp/fig_05_conv_teorema_zoom.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_conv_teorema_zoom.png"

[1] Frequencia: filtro Ideal (cilindro)
[2] Espaco: ondulacoes da sinc (causa do ringing)


In [23]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_conv_teorema_zoom_0.png"),
            mm.read("tmp/fig_05_conv_teorema_zoom_1.png"),
        ],
        titles=[
            'Frequencia: filtro Ideal (cilindro)',
            'Espaco: ondulacoes da sinc (causa do ringing)',
        ],
        cols=2,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_conv_teorema_zoom_0.png (ver a versao Python)")

<Figure size 1500x750 with 2 Axes>

**Figura 5.10:** A Dualidade Perigosa: o corte abrupto na Frequência (cilindro Ideal) vira obrigatoriamente uma *sinc* no espaço. Suas ondulações causam o *ringing* fantasma nas bordas da imagem.


### 5.6.1 Filtri Passa-Basso

I filtri passa-basso attenuano le componenti ad alta frequenza, producendo un appiattimento dell'immagine e una riduzione del rumore. Dopo la centralizzazione dello spettro (FFT Shift), la distanza di ciascun punto dal centro è data da:

<a id="eq-05-dist-centro"></a>
$$
D(u,v) = \sqrt{\left(u - \tfrac{M}{2}\right)^2 + \left(v - \tfrac{N}{2}\right)^2} \tag{5.5}
$$


**Filtro Ideale (LPFI):**
<a id="eq-05-lpf-ideal"></a>
$$
H_{\text{ideal}}(u,v) =
\begin{cases}
1, & D(u,v) \leq D_0 \\
0, & D(u,v) > D_0
\end{cases} \tag{5.6}
$$


Il taglio netto a $D_0$ introduce discontinuità nel dominio delle frequenze, causando oscillazioni nel dominio spaziale note come *ringing*. Questo effetto è associato alla convoluzione con funzioni a supporto infinito.

**Filtro Gaussiano (LPFG):**
<a id="eq-05-lpf-gauss"></a>
$$
H_{\text{gauss}}(u,v) = e^{-D^2(u,v)/(2\sigma^2)} \tag{5.7}
$$


La regolarità della funzione gaussiana nel dominio delle frequenze evita discontinuità, eliminando il *ringing* e producendo una transizione graduale tra frequenze preservate e attenuate.

**Filtro di Butterworth (LPFB) di ordine $n$:**
<a id="eq-05-lpf-butterworth"></a>
$$
H_{\text{BW}}(u,v) = \frac{1}{1 + \left[D(u,v)/D_0\right]^{2n}} \tag{5.8}
$$


Il parametro $n$ controlla la gradualità della transizione tra l'attenuazione e il passaggio delle frequenze. Valori piccoli producono transizioni morbide, mentre valori grandi avvicinano il comportamento al filtro ideale, con un maggiore rischio di *ringing*. Un esempio comparativo è mostrato in [Figura 5.11](#fig-05-filtros-passa-baixa).

In [24]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div style="font-family:sans-serif;max-width:680px;margin:0 auto;padding:10px;">
<div style="text-align:center;font-size:12px;font-weight:bold;color:#374151;margin-bottom:8px;">
  Profili dei Filtri Passa-Basso — confronto visivo (D₀ = 30)
</div>
<svg viewBox="0 0 640 200" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#fff;">
  <defs>
    <marker id="ah" markerWidth="6" markerHeight="6" refX="3" refY="3" orient="auto">
      <path d="M0,0 L6,3 L0,6 Z" fill="#9ca3af"/>
    </marker>
  </defs>
  <!-- Grid -->
  <line x1="60" y1="20" x2="60" y2="170" stroke="#e5e7eb" stroke-width="0.8"/>
  <line x1="60" y1="170" x2="610" y2="170" stroke="#e5e7eb" stroke-width="0.8"/>
  <!-- Eixos -->
  <line x1="60" y1="170" x2="605" y2="170" stroke="#9ca3af" stroke-width="1" marker-end="url(#ah)"/>
  <line x1="60" y1="175" x2="60" y2="15" stroke="#9ca3af" stroke-width="1" marker-end="url(#ah)"/>
  <text x="612" y="174" font-size="9" fill="#6b7280">D(u,v)</text>
  <text x="63" y="14" font-size="9" fill="#6b7280">H</text>
  <!-- Rótulos eixo Y -->
  <text x="52" y="35" font-size="8" fill="#6b7280" text-anchor="end">1.0</text>
  <text x="52" y="102" font-size="8" fill="#6b7280" text-anchor="end">0.5</text>
  <text x="52" y="173" font-size="8" fill="#6b7280" text-anchor="end">0.0</text>
  <line x1="57" y1="33" x2="63" y2="33" stroke="#9ca3af" stroke-width="0.8"/>
  <line x1="57" y1="100" x2="63" y2="100" stroke="#9ca3af" stroke-width="0.8"/>
  <!-- D0 marker -->
  <line x1="210" y1="30" x2="210" y2="175" stroke="#d1d5db" stroke-width="0.8" stroke-dasharray="3,3"/>
  <text x="210" y="184" font-size="8" fill="#9ca3af" text-anchor="middle">D₀</text>
  <!-- Filtro Ideal (vermelho) -->
  <polyline points="60,33 210,33 210,170 610,170" fill="none" stroke="#D85A30" stroke-width="2"/>
  <!-- Filtro Gaussiano (verde) -->
  <path d="M60,33 C100,33 130,40 160,60 S210,110 250,140 S320,168 610,170" fill="none" stroke="#1D9E75" stroke-width="2"/>
  <!-- Filtro Butterworth n=2 (azul) -->
  <path d="M60,33 C130,33 165,45 195,75 S225,130 250,148 S310,168 610,170" fill="none" stroke="#534AB7" stroke-width="2"/>
  <!-- Butterworth n=5 (roxo claro) -->
  <path d="M60,33 C170,33 195,40 208,70 S215,140 225,158 S260,170 610,170" fill="none" stroke="#9333ea" stroke-width="1.5" stroke-dasharray="5,3"/>
  <!-- Legenda -->
  <rect x="430" y="25" width="170" height="100" fill="#f9fafb" stroke="#e5e7eb" rx="4"/>
  <line x1="440" y1="45" x2="465" y2="45" stroke="#D85A30" stroke-width="2"/>
  <text x="470" y="49" font-size="9" fill="#374151">Ideale (taglio perfetto)</text>
  <text x="470" y="60" font-size="8" fill="#9ca3af">→ ringing sui bordi</text>
  <line x1="440" y1="78" x2="465" y2="78" stroke="#1D9E75" stroke-width="2"/>
  <text x="470" y="82" font-size="9" fill="#374151">Gaussiano</text>
  <text x="470" y="93" font-size="8" fill="#9ca3af">→ senza ringing</text>
  <line x1="440" y1="106" x2="465" y2="106" stroke="#534AB7" stroke-width="2"/>
  <text x="470" y="110" font-size="9" fill="#374151">Butterworth n=2</text>
  <line x1="440" y1="118" x2="453" y2="118" stroke="#9333ea" stroke-width="1.5" stroke-dasharray="4,2"/>
  <text x="470" y="122" font-size="9" fill="#374151">Butterworth n=5</text>
  <!-- Zona de transição -->
  <text x="240" y="85" font-size="8" fill="#6b7280" font-style="italic">zona di</text>
  <text x="240" y="96" font-size="8" fill="#6b7280" font-style="italic">transizione</text>
</svg>
<div style="font-size:10px;color:#6b7280;margin-top:6px;text-align:center;">
  All'aumentare dell'ordine di Butterworth, il profilo si avvicina al filtro Ideale — e il ringing aumenta.
</div>
</div>
""")

**Figura 5.11:** Filtri passa-basso.


<figure id="fig-05-filtros-passa-baixa">
  <img src="imagens/fig-05-filtros-passa-baixa.png" alt=" Filtri passa-basso. " style="max-width:80%" />
  <figcaption><strong>Figura 5.11:</strong>  Filtri passa-basso. </figcaption>
</figure>

### 5.6.2 Filtri Passa-Alto e Passa-Banda

**I filtri passa-alto** possono essere ottenuti da un filtro passa-basso complementare, definito come:

$$
H_{\text{HP}}(u,v) = 1 - H_{\text{LP}}(u,v)
$$

Questo tipo di filtro preserva le componenti ad alta frequenza, evidenziando bordi e dettagli, mentre attenua le regioni a variazione graduale.

**I filtri passa-banda** preservano solo una banda intermedia di frequenze, limitata da due raggi $D_L$ e $D_H$:

$$
H_{\text{BP}}(u,v) =
H_{\text{LP}}^{(D_H)}(u,v)\cdot
\left[1 - H_{\text{LP}}^{(D_L)}(u,v)\right]
$$

Questo tipo di filtraggio è utile quando si desidera rimuovere simultaneamente le componenti a bassa e ad alta frequenza, preservando solo le strutture di scala intermedia.

Un'applicazione importante è la rimozione del **rumore periodico**, in cui i pattern regolari appaiono come picchi localizzati nello spettro di magnitudine. Questi picchi possono essere attenuati mediante filtri *notch* (reietta-banda), posizionati specificamente sulle frequenze indesiderate.

Esempi di filtri nel dominio della frequenza sono presentati nel simulatore della [Figura 5.12](#fig-05-sim-05-filtros), [Figura 5.13](#fig-05-filtros-freq) e [Figura 5.14](#fig-05-filtros-passa-alta)..

In [25]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-05-filtros" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-05-filtros * { box-sizing: border-box; }
  #sim-05-filtros canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-filtros button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-filtros button:hover { background: #e8dfcf; }
  #sim-05-filtros .sim05_sf_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  
  .sf_legend { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 8px; margin-bottom: 14px; }
  .sf_leg_item { display: flex; align-items: flex-start; gap: 10px; padding: 10px 12px; border-radius: 10px; border: 1px solid #e9e3d3; cursor: pointer; background: #fafaf7; transition: opacity .15s; }
  .sf_leg_item.sf_off { opacity: .35; }
  .sf_leg_swatch { width: 32px; min-width: 32px; height: 3px; margin-top: 8px; border-radius: 2px; }
  .sf_leg_name { font-size: 12.5px; font-weight: 700; }
  .sf_leg_desc { font-size: 10.5px; color: #8a8371; line-height: 1.4; margin-top: 2px; }
  
  .sf_controls { display: flex; align-items: center; gap: 12px; flex-wrap: wrap; margin-bottom: 14px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sf_ctrl_lbl { font-size: 11.5px; color: #5e5a4a; white-space: nowrap; font-weight: 600; }
  .sf_ctrl_val { font-size: 12px; font-weight: 700; min-width: 25px; color: #26241d; font-family: monospace; }
  .sf_radio_grp { display: flex; gap: 12px; }
  .sf_radio_grp label { display: flex; align-items: center; gap: 6px; font-size: 11.5px; color: #5e5a4a; cursor: pointer; font-weight: 600; }
  
  .sf_charts { display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 14px; }
  .sf_card { background: #fafaf7; border-radius: 12px; padding: 12px; border: 1px solid #e9e3d3; }
  .sf_card_lbl { font-size: 10.5px; color: #5e5a4a; margin-bottom: 8px; font-weight: 700; text-transform: uppercase; letter-spacing: .04em; }
  .sf_stats { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 14px; }
  .sf_pill { font-size: 10.5px; padding: 4px 10px; border-radius: 8px; background: #fafaf7; color: #5e5a4a; border: 1px solid #e9e3d3; font-weight: 600; }
  .sf_pill b { color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎛️ Simulatore: Filtri nel Dominio della Frequenza</span>
  <span class="sim05_sf_pill">Passa-Basso / Passa-Alto</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <div class="sf_legend" id="sf_legend"></div>

  <div class="sf_controls">
    <span class="sf_ctrl_lbl">Frequenza di taglio D₀</span>
    <input type="range" id="sf_d0" min="5" max="100" value="30" step="1" style="flex:1;min-width:120px;max-width:240px;accent-color:#2980b9;cursor:pointer;">
    <span class="sf_ctrl_val" id="sf_d0v">30</span>
    <span class="sf_ctrl_lbl" style="margin-left:8px">Tipo di filtro</span>
    <div class="sf_radio_grp">
      <label><input type="radio" name="sf_ft" value="lp" checked style="cursor:pointer;"> passa-basso</label>
      <label><input type="radio" name="sf_ft" value="hp" style="cursor:pointer;"> passa-alto</label>
    </div>
  </div>

  <div class="sf_charts">
    <div class="sf_card">
      <div class="sf_card_lbl">Risposta H(D)</div>
      <canvas id="sf_c1" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Spettro filtrato |F · H|</div>
      <canvas id="sf_c2" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Segnale 1D — originale vs filtrato</div>
      <canvas id="sf_c3" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Energia trattenuta per banda (%)</div>
      <canvas id="sf_c4" style="width:100%;display:block"></canvas>
    </div>
  </div>

  <div class="sf_stats" id="sf_stats"></div>

</div>
</div>

<script>
(function(){
  function initSim05Filtros(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    var SF = [
      {key:'ideal', name:'Ideal',          desc:'Corte perfeito em D₀ — causa ringing',       color:'#c0392b', dash:null },
      {key:'gauss', name:'Gaussiano',      desc:'Transição suave — sem ringing',                 color:'#27ae60', dash:[6,3]},
      {key:'bw2',   name:'Butterworth n=2', desc:'Compromisso: suave com banda controlável',       color:'#2980b9', dash:[4,2]},
      {key:'bw5',   name:'Butterworth n=5', desc:'Aproxima o ideal mantendo transição suave',    color:'#b9770e', dash:[2,2]},
    ];

    var sf_D0 = 30, sf_hp = false;
    var sf_on = {ideal:true, gauss:true, bw2:true, bw5:true};

    function sf_H(D, key, d0, hp){
      var h;
      if(key === 'ideal')      h = D <= d0 ? 1 : 0;
      else if(key === 'gauss') h = Math.exp(-D*D/(2*d0*d0));
      else if(key === 'bw2')   h = 1/(1+Math.pow(D/d0,4));
      else                     h = 1/(1+Math.pow(D/d0,10));
      return hp ? 1-h : h;
    }

    function sf_setup(id, h){
      var c = root.querySelector('#' + id);
      var w = c.parentElement.clientWidth - 24;
      if(w < 100) w = 280;
      c.width  = w;
      c.height = h || 180;
      return {c:c, ctx:c.getContext('2d'), w:c.width, h:c.height};
    }

    function sf_axes(ctx, w, h, pad, xmax, ymin, ymax, xlabel, ylabel){
      var l=pad.l, r=pad.r, t=pad.t, b=pad.b;
      ctx.clearRect(0,0,w,h);

      ctx.strokeStyle='rgba(0,0,0,.06)'; ctx.lineWidth=0.5;
      var nx=4, ny=4;
      for(var i=0; i<=nx; i++){
        var x = l + (w-l-r) * i / nx;
        ctx.beginPath(); ctx.moveTo(x, t); ctx.lineTo(x, h-b); ctx.stroke();
      }
      for(var j=0; j<=ny; j++){
        var y = t + (h-t-b) * j / ny;
        ctx.beginPath(); ctx.moveTo(l, y); ctx.lineTo(w-r, y); ctx.stroke();
      }

      ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=1;
      ctx.beginPath(); ctx.moveTo(l, t); ctx.lineTo(l, h-b); ctx.lineTo(w-r, h-b); ctx.stroke();

      ctx.fillStyle='#8a8371'; ctx.font='10px monospace'; ctx.textAlign='center';
      for(var i=0; i<=nx; i++){
        var x = l + (w-l-r) * i / nx;
        var val = Math.round(xmax * i / nx);
        ctx.fillText(val, x, h-b+12);
      }
      ctx.textAlign='right';
      for(var j=0; j<=ny; j++){
        var y = t + (h-t-b) * j / ny;
        var val = ymax - (ymax-ymin) * j / ny;
        ctx.fillText(val.toFixed(2), l-4, y+3);
      }

      ctx.fillStyle='#5e5a4a'; ctx.font='10.5px Inter,sans-serif'; ctx.textAlign='center';
      ctx.fillText(xlabel, l+(w-l-r)/2, h-2);
      ctx.save(); ctx.translate(11, t+(h-t-b)/2); ctx.rotate(-Math.PI/2);
      ctx.fillText(ylabel, 0, 0); ctx.restore();

      var d0x = l + (sf_D0/xmax)*(w-l-r);
      if(d0x > l && d0x < w-r){
        ctx.strokeStyle='#b9770e'; ctx.lineWidth=1; ctx.setLineDash([4,3]);
        ctx.beginPath(); ctx.moveTo(d0x, t); ctx.lineTo(d0x, h-b); ctx.stroke();
        ctx.setLineDash([]);
        ctx.fillStyle='#b9770e'; ctx.font='10px monospace'; ctx.textAlign='center';
        ctx.fillText('D₀', d0x, t-2);
      }

      return {
        toX: function(v){ return l + (v/xmax)*(w-l-r); },
        toY: function(v){ return (h-b) - (v-ymin)/(ymax-ymin)*(h-t-b); }
      };
    }

    function sf_line(ctx, pts, color, dash, fill){
      if(!pts.length) return;
      ctx.strokeStyle = color; ctx.lineWidth = 2;
      ctx.setLineDash(dash || []);
      if(fill){
        ctx.fillStyle = color.replace(')', ', 0.12)').replace('rgb', 'rgba');
        ctx.beginPath();
        ctx.moveTo(pts[0].x, pts[0].baseY);
        pts.forEach(function(p){ ctx.lineTo(p.x, p.y); });
        ctx.lineTo(pts[pts.length-1].x, pts[pts.length-1].baseY);
        ctx.closePath(); ctx.fill();
      }
      ctx.beginPath();
      pts.forEach(function(p, i){ i===0 ? ctx.moveTo(p.x, p.y) : ctx.lineTo(p.x, p.y); });
      ctx.stroke();
      ctx.setLineDash([]);
    }

    function sf_drawProfile(){
      var s = sf_setup('sf_c1'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var ax = sf_axes(ctx, s.w, s.h, pad, 300, 0, 1, 'D(u,v)', 'H(D)');
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var d=0; d<=300; d+=2){
          pts.push({x: ax.toX(d), y: ax.toY(sf_H(d, f.key, sf_D0, sf_hp)), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, false);
      });
    }

    function sf_drawSpectrum(){
      var s = sf_setup('sf_c2'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var ax = sf_axes(ctx, s.w, s.h, pad, 300, 0, 1, 'D(u,v)', '|F·H|');
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var d=0; d<=300; d+=2){
          var y = Math.max(0, Math.exp(-d*d/(2*80*80)) * sf_H(d, f.key, sf_D0, sf_hp));
          pts.push({x: ax.toX(d), y: ax.toY(y), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, true);
      });
    }

    function sf_drawSignal(){
      var s = sf_setup('sf_c3'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var N = 128;
      var all = [];
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        for(var t=0; t<N; t++){
          var v = 0;
          for(var fr=1; fr<80; fr++) v += sf_H(fr, f.key, sf_D0, sf_hp) * Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
          all.push(v/7);
        }
      });
      for(var t=0; t<N; t++){
        var v = 0;
        for(var fr=1; fr<60; fr++) v += Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
        all.push(v/7);
      }
      var ymin = Math.min.apply(null, all)*1.15, ymax = Math.max.apply(null, all)*1.15;
      if(ymax - ymin < 0.1){ymin = -0.5; ymax = 0.5;}
      var ax = sf_axes(ctx, s.w, s.h, pad, N, ymin, ymax, 'amostras', 'amp');

      var orig = [];
      for(var t=0; t<N; t++){
        var v = 0;
        for(var fr=1; fr<60; fr++) v += Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
        orig.push({x: ax.toX(t), y: ax.toY(v/7), baseY: ax.toY(0)});
      }
      ctx.globalAlpha = 0.4;
      sf_line(ctx, orig, '#8a8371', [4,3], false);
      ctx.globalAlpha = 1;

      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var t=0; t<N; t++){
          var v = 0;
          for(var fr=1; fr<80; fr++) v += sf_H(fr, f.key, sf_D0, sf_hp) * Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
          pts.push({x: ax.toX(t), y: ax.toY(v/7), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, false);
      });
    }

    function sf_drawEnergy(){
      var s = sf_setup('sf_c4'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:32};
      var bands = [[0,20],[20,40],[40,60],[60,80],[80,100]];
      var labels = ['0–20','20–40','40–60','60–80','80–100'];
      var active = SF.filter(function(f){ return sf_on[f.key]; });
      if(active.length === 0) return;

      ctx.clearRect(0, 0, s.w, s.h);
      ctx.strokeStyle = 'rgba(0,0,0,.06)'; ctx.lineWidth = 0.5;
      for(var j=0; j<=4; j++){
        var y = pad.t + (s.h-pad.t-pad.b) * j / 4;
        ctx.beginPath(); ctx.moveTo(pad.l, y); ctx.lineTo(s.w-pad.r, y); ctx.stroke();
      }
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 1;
      ctx.beginPath(); ctx.moveTo(pad.l, pad.t); ctx.lineTo(pad.l, s.h-pad.b);
      ctx.lineTo(s.w-pad.r, s.h-pad.b); ctx.stroke();

      ctx.fillStyle = '#8a8371'; ctx.font = '10px monospace'; ctx.textAlign = 'right';
      for(var j=0; j<=4; j++){
        var y = pad.t + (s.h-pad.t-pad.b) * j / 4;
        ctx.fillText((100 - j*25) + '%', pad.l-4, y+3);
      }

      var bw = (s.w - pad.l - pad.r) / bands.length;
      var gw = bw * 0.12, fw = (bw - gw*(active.length+1)) / active.length;
      if(fw < 2) fw = 2;

      bands.forEach(function(band, bi){
        var energy = function(key){
          var e = 0, n = 0;
          for(var d = band[0]; d < band[1]; d++){ e += Math.pow(sf_H(d, key, sf_D0, sf_hp), 2); n++; }
          return n > 0 ? Math.round(e/n * 100) : 0;
        };
        var bx = pad.l + bi * bw;
        active.forEach(function(f, fi){
          var val = energy(f.key);
          var x = bx + gw*(fi+1) + fw*fi;
          var barH = (val/100) * (s.h - pad.t - pad.b);
          var y = (s.h - pad.b) - barH;
          ctx.fillStyle = f.color + 'bb';
          ctx.fillRect(x, y, fw, barH);
          ctx.strokeStyle = f.color; ctx.lineWidth = 0.5;
          ctx.strokeRect(x, y, fw, barH);
        });
        ctx.fillStyle = '#8a8371'; ctx.font = '10px monospace'; ctx.textAlign = 'center';
        ctx.fillText(labels[bi], bx + bw/2, s.h - pad.b + 14);
      });

      ctx.fillStyle = '#5e5a4a'; ctx.font = '10.5px Inter,sans-serif'; ctx.textAlign = 'center';
      ctx.save(); ctx.translate(11, pad.t + (s.h-pad.t-pad.b)/2); ctx.rotate(-Math.PI/2);
      ctx.fillText('energia (%)', 0, 0); ctx.restore();
      ctx.fillText('banda de frequência', pad.l + (s.w-pad.l-pad.r)/2, s.h - 1);
    }

    function sf_buildLegend(){
      var el = root.querySelector('#sf_legend');
      el.innerHTML = '';
      SF.forEach(function(f){
        var d = document.createElement('div');
        d.className = 'sf_leg_item' + (sf_on[f.key] ? '' : ' sf_off');
        d.style.borderColor = sf_on[f.key] ? f.color : '#e9e3d3';
        var swatchStyle = 'background:' + f.color;
        if(f.dash){
          var seg = f.dash[0], gap = f.dash[1];
          swatchStyle = 'background:repeating-linear-gradient(90deg,' + f.color + ' 0 ' + seg + 'px,transparent ' + seg + 'px ' + (seg+gap) + 'px)';
        }
        d.innerHTML =
          '<div class="sf_leg_swatch" style="' + swatchStyle + '"></div>' +
          '<div><div class="sf_leg_name" style="color:' + f.color + '">' + f.name + '</div>' +
          '<div class="sf_leg_desc">' + f.desc + '</div></div>';
        d.addEventListener('click', function(){
          sf_on[f.key] = !sf_on[f.key]; sf_buildLegend(); sf_draw();
        });
        el.appendChild(d);
      });
    }

    function sf_stats(){
      var el = root.querySelector('#sf_stats');
      el.innerHTML = SF.filter(function(f){ return sf_on[f.key]; }).map(function(f){
        var h50 = sf_H(sf_D0, f.key, sf_D0, sf_hp).toFixed(2);
        var en = Math.round(function(){
          var s = 0;
          for(var i=0; i<200; i++) s += Math.pow(sf_H(i*0.5, f.key, sf_D0, sf_hp), 2);
          return s/200;
        }() * 100);
        return '<div class="sf_pill" style="border-color:' + f.color + '55">' +
          '<b style="color:' + f.color + '">' + f.name + '</b>' +
          ' H(D₀)=<b>' + h50 + '</b> &middot; energia=<b>' + en + '%</b></div>';
      }).join('');
    }

    function sf_draw(){
      sf_drawProfile();
      sf_drawSpectrum();
      sf_drawSignal();
      sf_drawEnergy();
      sf_stats();
    }

    root.querySelector('#sf_d0').addEventListener('input', function(){
      sf_D0 = +this.value;
      root.querySelector('#sf_d0v').textContent = sf_D0;
      sf_draw();
    });

    root.querySelectorAll('input[name="sf_ft"]').forEach(function(r){
      r.addEventListener('change', function(e){
        sf_hp = e.target.value === 'hp';
        sf_draw();
      });
    });

    sf_buildLegend();
    sf_draw();
    window.addEventListener('resize', sf_draw);
  }

  function tryInitSim05Filtros(){
    var root = document.getElementById('sim-05-filtros');
    if (root) initSim05Filtros(root); else setTimeout(tryInitSim05Filtros, 200);
  }
  tryInitSim05Filtros();
})();
</script>
""")

**Figura 5.12:** Simulatore interattivo di filtri nel dominio della frequenza.


<figure id="fig-05-sim-05-filtros">
  <img src="imagens/fig-05-sim-05-filtros.png" alt=" Simulatore interattivo di filtri nel dominio della frequenza. " style="max-width:80%" />
  <figcaption><strong>Figura 5.12:</strong>  Simulatore interattivo di filtri nel dominio della frequenza. </figcaption>
</figure>

In [26]:
%%writefile tmp/fig_05_filtros_freq.cpp
#define MM_OUT "tmp/fig_05_filtros_freq.png"
//| label: fig-05-filtros-freq
//| fig-cap: "Comparação entre filtros passa-baixa: Ideal (D₀=30), Gaussiano (D₀=30) e Butterworth (D₀=30, n=2). Perfis de H(u,v) ao longo de uma linha central e imagens filtradas correspondentes."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"

// Função auxiliar para converter H (CV_64F) para visualização
mm::Image H_vis(const cv::Mat& H) {
    cv::Mat H_uint8;
    cv::normalize(H * 255.0, H_uint8, 0, 255, cv::NORM_MINMAX, CV_8U);
    return mm::Image(H_uint8);
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    int M = img_gray.h;
    int N = img_gray.w;
    double D0 = 30;
    int n_bw = 2;

    // ── Funções de transferência (H centrado) via construtores de morph.hpp ───────
    //   Ideal:       1 se D<=D0, senão 0
    //   Gaussiano:   exp(-D^2/(2 D0^2))
    //   Butterworth: 1 / (1 + (D/D0)^(2n))
    cv::Mat H_ideal = mm::idealFilter(M, N, D0);
    cv::Mat H_gauss = mm::gaussFilter(M, N, D0);
    cv::Mat H_bw    = mm::butterFilter(M, N, D0, n_bw);

    // ── Imagens filtradas via FFT ────────────────────────────────────────────────
    mm::Image img_ideal = mm::freqFilter(img_gray, H_ideal);
    mm::Image img_gauss = mm::freqFilter(img_gray, H_gauss);
    mm::Image img_bw    = mm::freqFilter(img_gray, H_bw);

    // ── Perfis de H(u,v) na linha central, desenhados com cv::line ──────────────
    int linha = M / 2;
    int Wp = 512, Hp = 288;
    cv::Mat perfil(Hp, Wp, CV_8UC3, cv::Scalar(255, 255, 255));

    // Função lambda para desenhar um perfil
    auto traca = [&](const cv::Mat& row, cv::Scalar cor) {
        cv::Point ant(-1, -1);
        bool first = true;
        for (int x = 0; x < N; ++x) {
            double val = row.at<double>(x);
            int px = (int)std::round(x * (double)(Wp - 1) / (N - 1));
            int py = (int)std::round(10 + (1.0 - val) * (Hp - 20));
            if (ant.x >= 0) {
                cv::line(perfil, ant, cv::Point(px, py), cor, 2, cv::LINE_AA);
            }
            ant = cv::Point(px, py);
        }
    };

    traca(H_ideal.row(linha), cv::Scalar(216, 90, 48));
    traca(H_gauss.row(linha), cv::Scalar(29, 158, 117));
    traca(H_bw.row(linha),    cv::Scalar(83, 74, 183));

    // Preparar imagens para exibição
    std::vector<mm::Image> imagens = {
        img_gray, img_ideal, img_gauss, img_bw,
        H_vis(H_ideal), H_vis(H_gauss), H_vis(H_bw),
        mm::Image(perfil)
    };

    std::vector<std::string> titulos = {
        "Original", "LPF Ideal", "LPF Gaussiano", "LPF Butterworth (n=2)",
        "H Ideal", "H Gaussiano", "H Butterworth", "Perfis H(u,v)"
    };

    mm::show(imagens, MM_OUT, titulos, 4);

    return 0;
}

Overwriting tmp/fig_05_filtros_freq.cpp


In [27]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_filtros_freq.cpp -o tmp/fig_05_filtros_freq -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_filtros_freq \
  && test -f "tmp/fig_05_filtros_freq.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_filtros_freq.png"

[1] Original
[2] LPF Ideal
[3] LPF Gaussiano
[4] LPF Butterworth (n=2)
[5] H Ideal
[6] H Gaussiano
[7] H Butterworth
[8] Perfis H(u,v)


In [28]:
try:
    mm.show(mm.read("tmp/fig_05_filtros_freq.png"), figsize=(16, 9))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_filtros_freq.png (ver a versao Python)")

<Figure size 2400x1350 with 1 Axes>

**Figura 5.13:** Comparação entre filtros passa-baixa: Ideal (D₀=30), Gaussiano (D₀=30) e Butterworth (D₀=30, n=2). Perfis de H(u,v) ao longo de uma linha central e imagens filtradas correspondentes.


In [29]:
%%writefile tmp/fig_05_filtros_passa_alta.cpp
#define MM_OUT "tmp/fig_05_filtros_passa_alta.png"
#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    //| label: fig-05-filtros-passa-alta
    //| fig-cap: "Filtro passa-alta Gaussiano. (a) Original; (b) Filtro passa-alta (D₀=30) - as bordas das moedas e fundo texturizado são realçados."

    // Filtro passa-alta = complemento do passa-baixa Gaussiano (1 - H_gauss),
    // construído com mm.gaussFilter(..., highpass=True) e aplicado via FFT.
    int M = img_gray.h;
    int N = img_gray.w;
    double D0 = 30;
    cv::Mat H_alta = mm::gaussFilter(M, N, D0, true);
    mm::Image img_alta = mm::freqFilter(img_gray, H_alta);

    mm::show(
        std::vector<mm::Image>{img_gray, img_alta},
        MM_OUT,
        std::vector<std::string>{"Original", "Passa-alta Gaussiano ($D_0=30$)"},
        2
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_05_filtros_passa_alta_0.png");
mm::write(img_alta, "tmp/fig_05_filtros_passa_alta_1.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_filtros_passa_alta.cpp


In [30]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_filtros_passa_alta.cpp -o tmp/fig_05_filtros_passa_alta -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_filtros_passa_alta \
  && test -f "tmp/fig_05_filtros_passa_alta.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_filtros_passa_alta.png"

[1] Original
[2] Passa-alta Gaussiano ($D_0=30$)


In [31]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_filtros_passa_alta_0.png"),
            mm.read("tmp/fig_05_filtros_passa_alta_1.png"),
        ],
        titles=[
            'Original',
            'Passa-alta Gaussiano ($D_0=30$)',
        ],
        cols=2,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_filtros_passa_alta_0.png (ver a versao Python)")

<Figure size 1500x750 with 2 Axes>

**Figura 5.14:** Filtro passa-alta Gaussiano. (a) Original; (b) Filtro passa-alta (D₀=30) - as bordas das moedas e fundo texturizado são realçados.


### 5.6.3 Rimozione del Rumore Periodico

Il rumore periodico — associato a interferenze elettriche, pattern regolari dei sensori o artefatti di scansione — appare nello spettro di Fourier come **picchi puntuali simmetrici rispetto al centro**.

Il filtro **reietta-banda (*notch*)** attenua selettivamente queste frequenze, preservando le altre componenti dell'immagine. Un esempio di applicazione è presentato nella [Figura 5.15](#fig-05-ruido-periodico)..

In [32]:
%%writefile tmp/fig_05_ruido_periodico.cpp
#define MM_OUT "tmp/fig_05_ruido_periodico.png"
//| label: fig-05-ruido-periodico
//| fig-cap: "Remoção de ruído periódico via filtro *notch* no domínio da frequência: (a) imagem com ruído senoidal, (b) espectro mostrando os picos do ruído, (c) máscara *notch* centrada nos picos, (d) imagem restaurada."
//| echo: true
//| output: true

// Ruído senoidal 2D -> espectro -> máscara notch nos 4 picos simétricos -> restauração.
#include <opencv2/opencv.hpp>
#include <iostream>
#include <cmath>
#include <vector>
#include <string>
#include <cstdio>
#include "morph.hpp"

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    // img_gray já está disponível (injetado automaticamente)

    int h_img = img_gray.h;
    int w_img = img_gray.w;

    // Criar meshgrid X, Y equivalente
    cv::Mat X(h_img, w_img, CV_64F), Y(h_img, w_img, CV_64F);
    for (int y = 0; y < h_img; y++) {
        for (int x = 0; x < w_img; x++) {
            X.at<double>(y, x) = x;
            Y.at<double>(y, x) = y;
        }
    }

    double u0 = 20, v0 = 20;  // frequências exatas do ruído

    // Converter img_gray para CV_64F
    cv::Mat img_gray64f;
    cv::Mat img_gray_mat = img_gray;  // conversão implícita para cv::Mat
    img_gray_mat.convertTo(img_gray64f, CV_64F);

    // Calcular ruído senoidal
    cv::Mat ruido(h_img, w_img, CV_64F);
    for (int y = 0; y < h_img; y++) {
        for (int x = 0; x < w_img; x++) {
            ruido.at<double>(y, x) = 40 * sin(2 * M_PI * (u0 * X.at<double>(y, x) / w_img + v0 * Y.at<double>(y, x) / h_img));
        }
    }

    // Imagem ruidosa = clip(img_gray + ruido, 0, 255)
    cv::Mat img_ruidosa64f = img_gray64f + ruido;
    cv::Mat img_ruidosa;
    cv::threshold(img_ruidosa64f, img_ruidosa, 0, 255, cv::THRESH_TRUNC);
    cv::threshold(img_ruidosa, img_ruidosa, 0, 0, cv::THRESH_TOZERO);
    img_ruidosa.convertTo(img_ruidosa, CV_8U);

    mm::Image img_ruidosa_mm = img_ruidosa;  // converter para mm::Image se necessário

    // Espectro de magnitude (log) da imagem ruidosa
    mm::Image mag_vis = mm::spectrumMag(img_ruidosa_mm);

    // Máscara notch: 1.0 em todo o plano, disco de 0.0 em cada um dos 4 picos
    cv::Mat mascara = cv::Mat::ones(h_img, w_img, CV_64F);
    double r_notch = 8;
    int cy = h_img / 2, cx = w_img / 2;

    // Criar meshgrid yy, xx com indexing="ij"
    cv::Mat yy(h_img, w_img, CV_64F), xx(h_img, w_img, CV_64F);
    for (int y = 0; y < h_img; y++) {
        for (int x = 0; x < w_img; x++) {
            yy.at<double>(y, x) = y;
            xx.at<double>(y, x) = x;
        }
    }

    // Para cada par (dy, dx) nos 4 picos simétricos
    std::vector<std::pair<double, double>> picos = {{v0, u0}, {-v0, -u0}, {v0, -u0}, {-v0, u0}};
    for (auto& p : picos) {
        double dy = p.first, dx = p.second;
        for (int y = 0; y < h_img; y++) {
            for (int x = 0; x < w_img; x++) {
                double dist = sqrt(pow(yy.at<double>(y, x) - (cy + dy), 2) + pow(xx.at<double>(y, x) - (cx + dx), 2));
                if (dist <= r_notch) {
                    mascara.at<double>(y, x) = 0.0;
                }
            }
        }
    }

    // Visualização da máscara
    cv::Mat mascara_vis = mascara * 255;
    mascara_vis.convertTo(mascara_vis, CV_8U);

    // Filtragem: aplica a máscara centrada como filtro no domínio da frequência
    mm::Image img_rest_vis = mm::freqFilter(img_ruidosa_mm, mascara);

    // PSNR entre original e restaurada
    cv::Mat img_gray_mat2 = img_gray;  // re-converter para cv::Mat
    cv::Mat img_rest_mat = img_rest_vis;  // converter mm::Image para cv::Mat
    double psnr = cv::PSNR(img_gray_mat2, img_rest_mat);
    printf("PSNR (original vs restaurada): %.2f dB\n", psnr);

    // Mostrar resultados
    std::vector<mm::Image> imagens = {img_ruidosa_mm, mag_vis, mascara_vis, img_rest_vis};
    std::vector<std::string> titulos = {
        "Com ruído periódico",
        "Espectro (log)",
        "Máscara notch",
        "Restaurada (PSNR=" + std::to_string(psnr).substr(0, 4) + " dB)"
    };

    // Corrigir título com formatação .1f
    char titulo_psnr[100];
    sprintf(titulo_psnr, "Restaurada (PSNR=%.1f dB)", psnr);
    titulos[3] = titulo_psnr;

    mm::show(imagens, MM_OUT, titulos, 4);

    return 0;
}

Overwriting tmp/fig_05_ruido_periodico.cpp


In [33]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_ruido_periodico.cpp -o tmp/fig_05_ruido_periodico -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_ruido_periodico \
  && test -f "tmp/fig_05_ruido_periodico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_ruido_periodico.png"

PSNR (original vs restaurada): 7.46 dB
[1] Com ruído periódico
[2] Espectro (log)
[3] Máscara notch
[4] Restaurada (PSNR=7.5 dB)


In [34]:
try:
    mm.show(mm.read("tmp/fig_05_ruido_periodico.png"), figsize=(16, 4))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_ruido_periodico.png (ver a versao Python)")

<Figure size 2400x600 with 1 Axes>

**Figura 5.15:** Remoção de ruído periódico via filtro *notch* no domínio da frequência: (a) imagem com ruído senoidal, (b) espectro mostrando os picos do ruído, (c) máscara *notch* centrada nos picos, (d) imagem restaurada.


### 📌 Sintesi — Filtri Spettrali

| Filtro | Effetto visivo | Artefatto | Uso |
|:---|:---|:---|:---|
| Passa-basso ideale | Attenuazione intensa | *Ringing* | Illustrativo |
| Passa-basso Gaussiano | Attenuazione graduale | Non presenta *ringing* | Attenuazione generale |
| Passa-basso Butterworth | Attenuazione controllata | *Ringing* (ordini elevati) | Compromesso tra attenuazione e selettività |
| Passa-alto | Enfatizzazione dei bordi | Amplificazione del rumore | Rilevamento dei contorni |
| *Notch* | Rimozione selettiva delle frequenze | Possibili distorsioni locali | Rimozione del rumore periodico |

La progettazione dei filtri nel dominio della frequenza consiste nella definizione di maschere spettrali. Tuttavia, effetti nel dominio spaziale, come *ringing* e sfocatura, emergono direttamente da queste scelte nello spettro.

## 5.7 *Wavelet* e Multirisoluzione

La Trasformata di Fourier decompone il segnale in frequenze **globali**: ogni coefficiente $F(u,v)$ riceve contributi dall'intera immagine, senza informazioni esplicite sulla localizzazione spaziale di tali frequenze. Pertanto, strutture localizzate, come i bordi, sono rappresentate in modo distribuito nello spettro.

Le ***wavelet* (ondine)** superano questa limitazione utilizzando funzioni base **localizzate nello spazio**, che possono essere traslate e scalate. Queste funzioni possiedono **supporto compatto**, ossia sono diverse da zero solo in una regione finita del dominio, consentendo una rappresentazione simultanea in termini di **frequenza e localizzazione spaziale**.

### 5.7.1 Il Limite della Trasformata di Fourier: localizzazione spaziale

La Trasformata di Fourier descrive con precisione **quali frequenze sono presenti** in un segnale, ma non rappresenta esplicitamente **dove queste frequenze si verificano nello spazio**.

Nell'esperimento presentato nella [Figura 5.16](#fig-05-fracasso-fourier), due immagini con strutture localizzate in posizioni diverse producono spettri di magnitudine praticamente identici. Ciò accade perché la rappresentazione di Fourier è globale: ogni coefficiente riceve un contributo dall'intera immagine.

Di conseguenza, lo spettro di magnitudine non rappresenta esplicitamente la localizzazione dei bordi o di altre strutture, ma solo la distribuzione delle frequenze presenti. Questa limitazione ha motivato lo sviluppo di rappresentazioni multirisoluzione, come la Trasformata *Wavelet* Discreta (DWT), in grado di descrivere simultaneamente la frequenza e la localizzazione spaziale delle strutture dell'immagine.

In [35]:
%%writefile tmp/fig_05_fracasso_fourier.cpp
#define MM_OUT "tmp/fig_05_fracasso_fourier.png"
#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

//| label: fig-05-fracasso-fourier
//| fig-cap: "Fourier global é cego para a posição. Os espectros não dizem onde as bordas estão."
//| echo: true
//| output: true

int main() {
    // Cria os dois sinais binários: cruzes em posições diferentes
    cv::Mat img_sinal1 = cv::Mat::zeros(128, 128, CV_8UC1);
    img_sinal1(cv::Rect(20, 0, 5, 128)).setTo(1);
    img_sinal1(cv::Rect(0, 100, 128, 5)).setTo(1);

    cv::Mat img_sinal2 = cv::Mat::zeros(128, 128, CV_8UC1);
    img_sinal2(cv::Rect(90, 0, 5, 128)).setTo(1);
    img_sinal2(cv::Rect(0, 30, 128, 5)).setTo(1);

    // Calcula a magnitude do espectro de Fourier (log(1+|F|)) para cada sinal
    mm::Image mag1 = mm::spectrumMag(mm::Image(img_sinal1));
    mm::Image mag2 = mm::spectrumMag(mm::Image(img_sinal2));

    // Exibe os sinais e seus espectros
    mm::show({mm::Image(img_sinal1), mag1, mm::Image(img_sinal2), mag2},
             MM_OUT,
             {"Sinal A", "Espectro A", "Sinal B (Deslocado)", "Espectro B"},
             4);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_sinal1, "tmp/fig_05_fracasso_fourier_0.png");
mm::write(mag1, "tmp/fig_05_fracasso_fourier_1.png");
mm::write(img_sinal2, "tmp/fig_05_fracasso_fourier_2.png");
mm::write(mag2, "tmp/fig_05_fracasso_fourier_3.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_fracasso_fourier.cpp


In [36]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_fracasso_fourier.cpp -o tmp/fig_05_fracasso_fourier -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_fracasso_fourier \
  && test -f "tmp/fig_05_fracasso_fourier.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_fracasso_fourier.png"

[1] Sinal A
[2] Espectro A
[3] Sinal B (Deslocado)
[4] Espectro B


In [37]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_fracasso_fourier_0.png"),
            mm.read("tmp/fig_05_fracasso_fourier_1.png"),
            mm.read("tmp/fig_05_fracasso_fourier_2.png"),
            mm.read("tmp/fig_05_fracasso_fourier_3.png"),
        ],
        titles=[
            'Sinal A',
            'Espectro A',
            'Sinal B (Deslocado)',
            'Espectro B',
        ],
        cols=4,
        figsize=(14, 4),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_fracasso_fourier_0.png (ver a versao Python)")

<Figure size 2100x600 with 4 Axes>

**Figura 5.16:** Fourier global é cego para a posição. Os espectros não dizem onde as bordas estão.


### 5.7.2 Trasformata *Wavelet* Discreta 2D

La Trasformata *Wavelet* Discreta (DWT) applica, separatamente nelle direzioni orizzontale e verticale, due filtri complementari: un **passa-basso** $h$ (approssimazione) e un **passa-alto** $g$ (dettagli), seguiti da sottocampionamento per un fattore di 2 in ciascuna dimensione. Questo processo produce quattro sottobande, i cui nomi indicano la combinazione dei filtri applicati in ciascuna direzione (L = *Low-pass*, passa-basso; H = *High-pass*, passa-alto). Le caratteristiche di ciascuna sottobanda sono riassunte nella [Tabela 5.3](#tbl-05-dwt-subbandas).

$$
\text{DWT}(f)=\{\underbrace{\text{LL}}_{\text{approssimazione}},\;
\underbrace{\text{LH}}_{\text{dettagli orizzontali}},\;
\underbrace{\text{HL}}_{\text{dettagli verticali}},\;
\underbrace{\text{HH}}_{\text{dettagli diagonali}}\}.
$$

<a id="tbl-05-dwt-subbandas"></a>

**Tabela 5.3:** Sottobande prodotte dalla Trasformata *Wavelet* Discreta 2D (DWT), indicando i filtri applicati in ciascuna direzione e il contenuto predominante di ciascuna componente.

| Sottobanda | Filtri applicati | Contenuto visivo |
|:---|:---:|:---|
| **LL** | basso × basso | Approssimazione dell'immagine (versione smussata e ridotta) |
| **LH** | basso × alto | Bordi orizzontali e variazioni verticali |
| **HL** | alto × basso | Bordi verticali e variazioni orizzontali |
| **HH** | alto × alto | Dettagli diagonali e trame |


La decomposizione può essere applicata ricorsivamente sulla sottobanda LL, generando una rappresentazione multirisoluzione. Dopo $J$ livelli, si ottiene una struttura con $3J+1$ sottobande, in cui ogni nuovo livello riduce la risoluzione della componente di approssimazione.

> ### 📝 Collegamento con le CNN
>
> La decomposizione multirisoluzione delle *wavelet* possiede una relazione concettuale con le rappresentazioni gerarchiche utilizzate nelle reti neurali convoluzionali (CNN). In entrambi i casi, successive fasi di filtraggio e riduzione della risoluzione producono descrizioni sempre più astratte dell'immagine. Tuttavia, le ***wavelet* utilizzano filtri matematicamente definiti e ricostruibili**, mentre le **CNN apprendono i propri filtri durante l'addestramento**.

### 5.7.3 Famiglie di *Wavelet*

Diverse famiglie di *wavelet* presentano compromessi distinti tra **supporto spaziale**, regolarità e capacità di compressione. Il **supporto** corrisponde all'estensione della funzione *wavelet* nel dominio spaziale: quanto più piccolo è il supporto, tanto più localizzata è la funzione; quanto più grande, tanto più liscia tende a essere la sua rappresentazione, sebbene con un costo computazionale maggiore. La [Tabela 5.4](#tbl-05-wavelet-familias) confronta alcune delle famiglie più utilizzate.

<a id="tbl-05-wavelet-familias"></a>

**Tabela 5.4:** Confronto tra famiglie di *wavelet*, evidenziando la lunghezza del supporto, il numero di momenti nulli, la simmetria e le applicazioni tipiche.

| *Wavelet* | Lunghezza del supporto | Momenti nulli | Simmetria | Uso tipico |
|:---|:---:|:---:|:---:|:---|
| Haar | 2 | 1 | Asimmetrica | Introduzione e analisi di base |
| Daubechies db4 | 8 | 4 | Asimmetrica | Compressione e analisi generale |
| Symlet sym4 | 8 | 4 | Quasi simmetrica | Ricostruzione di segnali |
| Biortogonale 5/3 | 5/3 | 2/2 | Simmetrica | JPEG 2000 senza perdita |
| Biortogonale 9/7 | 9/7 | 4/4 | Simmetrica | JPEG 2000 con perdita |


I **momenti nulli** misurano la capacità della *wavelet* di rappresentare regioni uniformi dell'immagine con pochi coefficienti diversi da zero. Una *wavelet* con $p$ momenti nulli annulla esattamente i polinomi di grado fino a $p-1$. Di conseguenza, quanto maggiore è il numero di momenti nulli, tanto maggiore tende a essere l'efficienza di compressione nelle regioni omogenee, sebbene ciò implichi generalmente funzioni con supporto più esteso.

La [Figura 5.17](#fig-05-wavelet-functions) presenta le funzioni di base (*wavelet*) $\psi(t)$ nel dominio spaziale. Queste funzioni possiedono **supporto compatto**, cioè sono diverse da zero solo in una regione finita del dominio, contrariamente alle sinusoidi della Trasformata di Fourier, che si estendono su tutto il dominio.

In [38]:
%%writefile tmp/fig_05_wavelet_functions.cpp
#define MM_OUT "tmp/fig_05_wavelet_functions.png"
//| label: fig-05-wavelet-functions
//| fig-cap: "Funções da *Wavelet* (ψ). Note como elas rapidamente decaem para zero (suporte compacto), ao contrário das senoides infinitas de Fourier."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>


int main() {
    // psi via mm.wavefun (algoritmo em cascata) — suporte compacto: decai a zero.
    std::vector<double> x_h, phi_h, psi_h;
    mm::wavefun("haar", 6, x_h, phi_h, psi_h);
    std::vector<double> x_d, phi_d, psi_d;
    mm::wavefun("db4", 6, x_d, phi_d, psi_d);

    std::vector<std::vector<double>> xs_h = {x_h};
    std::vector<std::vector<double>> ys_h = {psi_h};
    std::vector<std::string> labels_h = {"psi Haar"};
    std::vector<cv::Scalar> colors_h = {cv::Scalar(180, 60, 40)};
    mm::Image chart_h = mm::lineChart(xs_h, ys_h, labels_h, colors_h,
                                      "Ondaleta Haar (psi)", "t");

    std::vector<std::vector<double>> xs_d = {x_d};
    std::vector<std::vector<double>> ys_d = {psi_d};
    std::vector<std::string> labels_d = {"psi Daubechies 4"};
    std::vector<cv::Scalar> colors_d = {cv::Scalar(60, 140, 40)};
    mm::Image chart_d = mm::lineChart(xs_d, ys_d, labels_d, colors_d,
                                      "Ondaleta Daubechies 4 (psi)", "t");

    std::vector<mm::Image> charts = {chart_h, chart_d};
    std::vector<std::string> titles = {"Haar", "Daubechies 4"};
    mm::show(charts, MM_OUT, titles, 2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(chart_h, "tmp/fig_05_wavelet_functions_0.png");
mm::write(chart_d, "tmp/fig_05_wavelet_functions_1.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_wavelet_functions.cpp


In [39]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_wavelet_functions.cpp -o tmp/fig_05_wavelet_functions -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_wavelet_functions \
  && test -f "tmp/fig_05_wavelet_functions.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_wavelet_functions.png"

[1] Haar
[2] Daubechies 4


In [40]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_wavelet_functions_0.png"),
            mm.read("tmp/fig_05_wavelet_functions_1.png"),
        ],
        titles=[
            'Haar',
            'Daubechies 4',
        ],
        cols=2,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_wavelet_functions_0.png (ver a versao Python)")

<Figure size 1500x750 with 2 Axes>

**Figura 5.17:** Funções da *Wavelet* (ψ). Note como elas rapidamente decaem para zero (suporte compacto), ao contrário das senoides infinitas de Fourier.


Il diagrama della [Figura 5.18](#fig-05-wavelet-diagrama) illustra l’analisi multirisoluzione eseguita dalla DWT, in cui la sottobanda di approssimazione (LL) viene successivamente decomposta, formando una rappresentazione gerarchica su due livelli.

In [41]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div style="font-family:sans-serif;max-width:680px;margin:0 auto;padding:10px;">
<div style="text-align:center;font-size:12px;font-weight:bold;color:#374151;margin-bottom:8px;">
  Decomposizione Wavelet 2D — Struttura Multirisoluzione (2 livelli)
</div>
<svg viewBox="0 0 640 260" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#f9fafb;">
  <!-- Imagem original -->
  <rect x="20" y="80" width="100" height="100" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5" rx="3"/>
  <text x="70" y="126" font-size="10" fill="#1e40af" text-anchor="middle" font-weight="bold">f(x,y)</text>
  <text x="70" y="140" font-size="9" fill="#1e40af" text-anchor="middle">M × N</text>
  <!-- Seta 1 -->
  <line x1="120" y1="130" x2="165" y2="130" stroke="#6b7280" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="142" y="124" font-size="8" fill="#6b7280" text-anchor="middle">DWT</text>
  <!-- Bloco Nível 1: 4 subbandas -->
  <rect x="165" y="55" width="80" height="75" fill="#fef3c7" stroke="#f59e0b" stroke-width="1.2" rx="2"/>
  <text x="205" y="87" font-size="8" fill="#92400e" text-anchor="middle" font-weight="bold">LL₁</text>
  <text x="205" y="98" font-size="7" fill="#92400e" text-anchor="middle">appross.</text>
  <text x="205" y="109" font-size="7" fill="#92400e" text-anchor="middle">M/2 × N/2</text>
  <rect x="245" y="55" width="80" height="75" fill="#dcfce7" stroke="#22c55e" stroke-width="1.2" rx="2"/>
  <text x="285" y="87" font-size="8" fill="#166534" text-anchor="middle" font-weight="bold">LH₁</text>
  <text x="285" y="98" font-size="7" fill="#166534" text-anchor="middle">orizz.</text>
  <rect x="165" y="130" width="80" height="75" fill="#fce7f3" stroke="#ec4899" stroke-width="1.2" rx="2"/>
  <text x="205" y="162" font-size="8" fill="#9d174d" text-anchor="middle" font-weight="bold">HL₁</text>
  <text x="205" y="173" font-size="7" fill="#9d174d" text-anchor="middle">vert.</text>
  <rect x="245" y="130" width="80" height="75" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.2" rx="2"/>
  <text x="285" y="162" font-size="8" fill="#5b21b6" text-anchor="middle" font-weight="bold">HH₁</text>
  <text x="285" y="173" font-size="7" fill="#5b21b6" text-anchor="middle">diag.</text>
  <!-- Rótulo nível 1 -->
  <text x="245" y="248" font-size="9" fill="#6b7280" text-anchor="middle">Livello 1 — M/2 × N/2 ciascuno</text>
  <!-- Seta LL₁ → Nível 2 -->
  <line x1="205" y1="55" x2="205" y2="45" stroke="#6b7280" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="205" y1="45" x2="400" y2="45" stroke="#6b7280" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="400" y1="45" x2="400" y2="55" stroke="#6b7280" stroke-width="1" marker-end="url(#arr)" stroke-dasharray="3,2"/>
  <text x="302" y="40" font-size="8" fill="#6b7280" text-anchor="middle">DWT su LL₁</text>
  <!-- Bloco Nível 2: subbandas de LL₁ -->
  <rect x="360" y="55" width="50" height="45" fill="#fef3c7" stroke="#f59e0b" stroke-width="1.2" rx="2"/>
  <text x="385" y="75" font-size="8" fill="#92400e" text-anchor="middle" font-weight="bold">LL₂</text>
  <text x="385" y="88" font-size="7" fill="#92400e" text-anchor="middle">M/4×N/4</text>
  <rect x="410" y="55" width="50" height="45" fill="#dcfce7" stroke="#22c55e" stroke-width="1.2" rx="2"/>
  <text x="435" y="80" font-size="8" fill="#166534" text-anchor="middle" font-weight="bold">LH₂</text>
  <rect x="360" y="100" width="50" height="45" fill="#fce7f3" stroke="#ec4899" stroke-width="1.2" rx="2"/>
  <text x="385" y="125" font-size="8" fill="#9d174d" text-anchor="middle" font-weight="bold">HL₂</text>
  <rect x="410" y="100" width="50" height="45" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.2" rx="2"/>
  <text x="435" y="125" font-size="8" fill="#5b21b6" text-anchor="middle" font-weight="bold">HH₂</text>
  <text x="435" y="168" font-size="8" fill="#6b7280" text-anchor="middle">Livello 2</text>
  <!-- Legenda direita -->
  <rect x="490" y="55" width="140" height="130" fill="#fff" stroke="#e5e7eb" rx="4"/>
  <text x="560" y="73" font-size="9" fill="#374151" text-anchor="middle" font-weight="bold">Legenda</text>
  <rect x="500" y="82" width="12" height="12" fill="#fef3c7" stroke="#f59e0b"/>
  <text x="518" y="92" font-size="8" fill="#374151">LL — Approssimazione</text>
  <rect x="500" y="100" width="12" height="12" fill="#dcfce7" stroke="#22c55e"/>
  <text x="518" y="110" font-size="8" fill="#374151">LH — Bordi orizz.</text>
  <rect x="500" y="118" width="12" height="12" fill="#fce7f3" stroke="#ec4899"/>
  <text x="518" y="128" font-size="8" fill="#374151">HL — Bordi vert.</text>
  <rect x="500" y="136" width="12" height="12" fill="#ede9fe" stroke="#8b5cf6"/>
  <text x="518" y="146" font-size="8" fill="#374151">HH — Dettagli diag.</text>
  <text x="560" y="168" font-size="8" fill="#6b7280" text-anchor="middle">Ogni livello: ½ della</text>
  <text x="560" y="178" font-size="8" fill="#6b7280" text-anchor="middle">risoluzione precedente</text>
  <defs>
    <marker id="arr" markerWidth="6" markerHeight="6" refX="3" refY="3" orient="auto">
      <path d="M0,0 L6,3 L0,6 Z" fill="#6b7280"/>
    </marker>
  </defs>
</svg>
</div>
""")

**Figura 5.18:** Diagramma della decomposizione *wavelet* 2D su due livelli.


<figure id="fig-05-wavelet-diagrama">
  <img src="imagens/fig-05-wavelet-diagrama.png" alt=" Diagramma della decomposizione *wavelet* 2D su due livelli. " style="max-width:80%" />
  <figcaption><strong>Figura 5.18:</strong>  Diagramma della decomposizione *wavelet* 2D su due livelli. </figcaption>
</figure>

Il simulatore [Figura 5.19](#fig-05-sim-05-wavelet) consente di esplorare in modo interattivo la Trasformata *Wavelet* Discreta 2D (DWT) utilizzando la *wavelet* di Haar. La decomposizione in sottobande evidenzia la separazione tra la componente di approssimazione e le componenti di dettaglio dell'immagine.

I diversi modelli di ingresso permettono di osservare il comportamento direzionale dei filtri. In immagini con bordi orizzontali e verticali, le sottobande LH e HL evidenziano, rispettivamente, le variazioni verticali e orizzontali dell'intensità. Nelle regioni a variazione graduale, la maggior parte dell'energia si concentra nella sottobanda di approssimazione LL, mentre le sottobande di dettaglio presentano coefficienti prossimi allo zero.

L'analisi multirisoluzione può essere osservata anche aumentando il numero di livelli di decomposizione. In tal caso, solo la sottobanda $\text{LL}_1$ viene nuovamente decomposta, generando le sottobande $\text{LL}_2$, $\text{LH}_2$, $\text{HL}_2$ e $\text{HH}_2$, che formano il secondo livello della rappresentazione gerarchica.

In pattern costituiti da regioni omogenee di grande estensione, come una sfumatura graduale o una scacchiera composta da blocchi di grandi dimensioni, l'energia rimane prevalentemente concentrata nella sottobanda LL. Nella sfumatura, ciò avviene perché le differenze tra pixel adiacenti sono piccole. Nella scacchiera, invece, i pixel possiedono praticamente la stessa intensità all'interno di ciascun blocco, cosicché solo i confini tra blocchi producono coefficienti non nulli nelle sottobande di dettaglio. Poiché tali confini occupano solo una piccola frazione dell'immagine, il loro contributo all'energia totale rimane ridotto.

Per rendere possibile l'analisi visiva di queste variazioni sottili, il simulatore incorpora un controllo del guadagno di contrasto dei dettagli (variabile da 1 a 8). Questo parametro funziona come un fattore di amplificazione lineare applicato esclusivamente ai coefficienti delle sottobande di dettaglio (LH, HL e HH) prima della loro visualizzazione a schermo. Negli scenari di transizione graduale (come il gradiente) o di uniformità locale (come l'interno dei blocchi della scacchiera), le differenze numeriche calcolate dal filtro passa-alto di Haar producono coefficienti molto prossimi allo zero, il che renderebbe i quadranti corrispondenti scuri e impercettibili a occhio nudo. Moltiplicando questi valori per il guadagno, il simulatore recupera visivamente le strutture ad alta frequenza nascoste ed enfatizza l'orientamento dei bordi residui.

Il grafico dell'energia per sottobanda quantifica tale distribuzione tra la componente di approssimazione e le componenti di dettaglio, dimostrando che il guadagno visivo non altera la metrica originale dell'energia. Nelle immagini naturali, la maggior parte dell'energia si concentra nella sottobanda LL, mentre le sottobande LH, HL e HH rappresentano principalmente bordi, texture e altre variazioni locali dell'intensità.

In [42]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-05-wavelet" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-05-wavelet * { box-sizing: border-box; }
  #sim-05-wavelet canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; image-rendering: pixelated; }
  #sim-05-wavelet select { font-size: 11px; padding: 6px 10px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; font-weight: 600; cursor: pointer; outline: none; }
  #sim-05-wavelet input[type=range] { cursor: pointer; accent-color: #2980b9; }
  #sim-05-wavelet .sim04_w_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_w_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🌊 Simulatore: Decomposizione Wavelet 2D</span>
  <span class="sim04_w_pill">Trasformata di Haar</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles Superiores -->
  <div style="display:flex; flex-wrap:wrap; gap:14px; align-items:center; margin-bottom:14px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    <div style="display:flex; flex-direction:column; gap:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Motivo Sintetico</label>
      <select id="wsim_pattern">
        <option value="combined" selected>Combinato (forme + texture)</option>
        <option value="shapes">Forme (bordi h/v)</option>
        <option value="texture">Texture (alta frequenza)</option>
        <option value="gradient">Gradiente morbido</option>
      </select>
    </div>
    
    <div style="display:flex; flex-direction:column; gap:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Livelli di Decomposizione</label>
      <div style="display:flex; gap:12px; height:32px; align-items:center;">
        <label style="display:flex; align-items:center; gap:5px; font-size:11.5px; color:#5e5a4a; font-weight:600; cursor:pointer;"><input type="radio" name="wsim_lv" value="1" style="cursor:pointer;"> 1 livello</label>
        <label style="display:flex; align-items:center; gap:5px; font-size:11.5px; color:#5e5a4a; font-weight:600; cursor:pointer;"><input type="radio" name="wsim_lv" value="2" checked style="cursor:pointer;"> 2 livelli</label>
      </div>
    </div>
    
    <div style="display:flex; flex-direction:column; gap:4px; flex:1; min-width:160px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Guadagno di Contrasto: <span id="wsim_gainv" style="font-weight:700; color:#2980b9;">3.0</span></label>
      <input type="range" id="wsim_gain" min="1" max="8" step="0.5" value="3" style="width:100%; height:4px;">
    </div>
  </div>

  <!-- Imagem Original vs Mosaico Wavelet -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(240px, 1fr)); gap:14px; margin-bottom:14px;">
    <div class="sim04_w_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700;">Immagine Originale</div>
      <canvas id="wsim_orig" style="width:100%; display:block; margin:0 auto;"></canvas>
    </div>
    <div class="sim04_w_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700;">Decomposizione Wavelet (Mosaico)</div>
      <canvas id="wsim_mosaic" style="width:100%; display:block; margin:0 auto; background:#ffffff;"></canvas>
    </div>
  </div>

  <!-- Energia por Subbanda -->
  <div class="sim04_w_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700; text-align:left;">Energia per Sottobanda (%) — Somma Preservata (Parseval)</div>
    <canvas id="wsim_energy" style="width:100%; display:block; margin:0 auto;"></canvas>
  </div>

  <!-- Legendas Explicativas -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(220px, 1fr)); gap:8px;">
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#26241d,#fafaf7);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">LL — Approssimazione</div><div style="font-size:10.5px; color:#8a8371;">Versione attenuata e ridotta dell'immagine</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">LH — Dettaglio Orizzontale</div><div style="font-size:10.5px; color:#8a8371;">Evidenzia i bordi orizzontali (variazione verticale)</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">HL — Dettaglio Verticale</div><div style="font-size:10.5px; color:#8a8371;">Evidenzia i bordi verticali (variazione orizzontale)</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">HH — Dettaglio Diagonale</div><div style="font-size:10.5px; color:#8a8371;">Texture e angoli (variazione in entrambe le direzioni)</div></div>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04Wavelet(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    var wsim_N = 128;

    function wsim_genImage(type){
      var img = [];
      for(var y=0; y<wsim_N; y++){
        var row = [];
        for(var x=0; x<wsim_N; x++){
          var v = 0;
          if(type==='gradient'){
            v = 255*(0.5*x/wsim_N + 0.5*y/wsim_N);
          } else if(type==='shapes'){
            v = 40 + 30*Math.sin(x/30);
            if(x>18&&x<58&&y>18&&y<58) v = 220;
            var cx=95, cy=95, r=24;
            if((x-cx)*(x-cx)+(y-cy)*(y-cy) < r*r) v = 195;
            if(x>70&&x<74) v = 235;
          } else if(type==='texture'){
            var period=8;
            v = ((Math.floor(x/period)+Math.floor(y/period))%2===0) ? 200 : 55;
          } else {
            v = 55 + 35*(x/wsim_N) + 15*Math.sin(y/12);
            if(x>12&&x<50&&y>12&&y<50) v = 225;
            var cx2=95, cy2=38, r2=17;
            if((x-cx2)*(x-cx2)+(y-cy2)*(y-cy2) < r2*r2) v = 205;
            if(y>82 && y<122){
              var p=6;
              v = ((Math.floor(x/p)+Math.floor(y/p))%2===0) ? 185 : 65;
            }
            if(Math.abs(x-y) < 2) v = 240;
          }
          row.push(Math.max(0,Math.min(255,v)));
        }
        img.push(row);
      }
      return img;
    }

    function wsim_dwt2(m){
      var h = m.length, w = m[0].length;
      var halfH = h / 2, halfW = w / 2;
      
      var LL = [], LH = [], HL = [], HH = [];
      for (var r = 0; r < halfH; r++) {
        LL.push(new Array(halfW));
        LH.push(new Array(halfW));
        HL.push(new Array(halfW));
        HH.push(new Array(halfW));
      }

      for(var r=0; r<halfH; r++){
        for(var c=0; c<halfW; c++){
          var a = m[2*r][2*c];
          var b = m[2*r][2*c+1];
          var g = m[2*r+1][2*c];
          var d = m[2*r+1][2*c+1];
          
          LL[r][c] = (a + b + g + d) / 2.0;
          LH[r][c] = (a - b + g - d) / 2.0;
          HL[r][c] = (a + b - g - d) / 2.0;
          HH[r][c] = (a - b - g + d) / 2.0;
        }
      }
      return {LL:LL, LH:LH, HL:HL, HH:HH};
    }

    function wsim_divCol(t){
      t = Math.max(-1, Math.min(1, t));
      if(t>=0) {
        // Interpola de branco (255,255,255) até azul forte (41,128,185)
        return [
          Math.round(255 + t*(41 - 255)),
          Math.round(255 + t*(128 - 255)),
          Math.round(255 + t*(185 - 255))
        ];
      }
      var s = -t;
      // Interpola de branco (255,255,255) até vermelho forte (192,57,43)
      return [
        Math.round(255 + s*(192 - 255)),
        Math.round(255 + s*(57 - 255)),
        Math.round(255 + s*(43 - 255))
      ];
    }

    function wsim_tileCanvas(mat, mode, gain){
      var d = mat.length;
      var cnv = document.createElement('canvas');
      cnv.width = d; cnv.height = d;
      var cctx = cnv.getContext('2d');
      var idata = cctx.createImageData(d,d);
      if(mode==='gray'){
        var mn=Infinity, mx=-Infinity;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){ var v=mat[r][c]; if(v<mn)mn=v; if(v>mx)mx=v; }
        var range=(mx-mn)||1;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){
          var v=(mat[r][c]-mn)/range*255;
          var idx=(r*d+c)*4;
          idata.data[idx]=v; idata.data[idx+1]=v; idata.data[idx+2]=v; idata.data[idx+3]=255;
        }
      } else {
        var maxAbs=0;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){ var av=Math.abs(mat[r][c]); if(av>maxAbs) maxAbs=av; }
        maxAbs = (maxAbs/gain) || 1;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){
          var t = mat[r][c]/maxAbs;
          var rgb = wsim_divCol(t);
          var idx=(r*d+c)*4;
          idata.data[idx]=rgb[0]; idata.data[idx+1]=rgb[1]; idata.data[idx+2]=rgb[2]; idata.data[idx+3]=255;
        }
      }
      cctx.putImageData(idata,0,0);
      return cnv;
    }

    function wsim_energySum(mat){
      var s=0;
      for(var r=0; r<mat.length; r++) for(var c=0; c<mat[0].length; c++) s += mat[r][c]*mat[r][c];
      return s;
    }

    var wsim_pattern='combined', wsim_level=2, wsim_gain=3;
    var wsim_currentImg = wsim_genImage(wsim_pattern);

    function wsim_setupSquare(id, cap){
      var c = root.querySelector('#' + id);
      var parentW = c.parentElement.clientWidth - 24;
      var w = Math.min(parentW, cap || 360);
      if(w<80) w = 240;
      c.width = w; c.height = w;
      return {c:c, ctx:c.getContext('2d'), size:w};
    }

    function wsim_drawOriginal(){
      var s = wsim_setupSquare('wsim_orig', 360);
      s.ctx.imageSmoothingEnabled = false;
      var tile = wsim_tileCanvas(wsim_currentImg, 'gray', wsim_gain);
      s.ctx.drawImage(tile, 0, 0, s.size, s.size);
    }

    function wsim_drawMosaic(){
      var s = wsim_setupSquare('wsim_mosaic', 360);
      var ctx = s.ctx, full = s.size, half = full/2;
      ctx.imageSmoothingEnabled = false;
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, full, full);

      var c1 = wsim_dwt2(wsim_currentImg);

      function place(mat, mode, x, y, w, h){
        var tile = wsim_tileCanvas(mat, mode, wsim_gain);
        ctx.drawImage(tile, x, y, w, h);
      }

      if(wsim_level===1){
        place(c1.LL, 'gray', 0, 0, half, half);
        place(c1.LH, 'div', half, 0, half, half);
        place(c1.HL, 'div', 0, half, half, half);
        place(c1.HH, 'div', half, half, half, half);
      } else {
        var c2 = wsim_dwt2(c1.LL);
        var q = half/2;
        place(c2.LL, 'gray', 0, 0, q, q);
        place(c2.LH, 'div', q, 0, q, q);
        place(c2.HL, 'div', 0, q, q, q);
        place(c2.HH, 'div', q, q, q, q);
        
        place(c1.LH, 'div', half, 0, half, half);
        place(c1.HL, 'div', 0, half, half, half);
        place(c1.HH, 'div', half, half, half, half);
      }

      ctx.strokeStyle = '#e4dcc8';
      ctx.lineWidth = 1.5;
      ctx.beginPath();
      ctx.moveTo(half,0); ctx.lineTo(half,full);
      ctx.moveTo(0,half); ctx.lineTo(full,half);
      ctx.stroke();
      if(wsim_level===2){
        ctx.lineWidth = 1;
        ctx.beginPath();
        ctx.moveTo(half/2,0); ctx.lineTo(half/2,half);
        ctx.moveTo(0,half/2); ctx.lineTo(half,half/2);
        ctx.stroke();
      }

      ctx.font = '700 10.5px monospace';
      function lbl(t,x,y){
        ctx.fillStyle = '#26241d';
        ctx.fillText(t, x+5, y+14);
      }
      if(wsim_level===1){
        lbl('LL', 0,0); lbl('LH', half,0); lbl('HL',0,half); lbl('HH', half,half);
      } else {
        lbl('LL₂', 0,0); lbl('LH₂', half/2,0); lbl('HL₂',0,half/2); lbl('HH₂', half/2, half/2);
        lbl('LH₁', half,0); lbl('HL₁',0,half); lbl('HH₁', half,half);
      }
    }

    function wsim_drawEnergy(){
      var s = root.querySelector('#wsim_energy');
      var w = s.parentElement.clientWidth - 24;
      if(w<100) w = 280;
      s.width = w; s.height = 160;
      var ctx = s.getContext('2d');
      ctx.clearRect(0,0,w,s.height);

      var c1 = wsim_dwt2(wsim_currentImg);
      var total = wsim_energySum(wsim_currentImg);
      var bars, labels;
      if(wsim_level===1){
        bars = [wsim_energySum(c1.LL), wsim_energySum(c1.LH), wsim_energySum(c1.HL), wsim_energySum(c1.HH)];
        labels = ['LL','LH','HL','HH'];
      } else {
        var c2 = wsim_dwt2(c1.LL);
        bars = [wsim_energySum(c2.LL), wsim_energySum(c2.LH), wsim_energySum(c2.HL), wsim_energySum(c2.HH),
                wsim_energySum(c1.LH), wsim_energySum(c1.HL), wsim_energySum(c1.HH)];
        labels = ['LL₂','LH₂','HL₂','HH₂','LH₁','HL₁','HH₁'];
      }
      var pcts = bars.map(function(b){ return b/total*100; });

      var pad = {l:34, r:10, t:12, b:24};
      var plotH = s.height - pad.t - pad.b;
      var plotW = w - pad.l - pad.r;

      ctx.strokeStyle = 'rgba(0,0,0,.06)'; ctx.lineWidth=0.5;
      for(var j=0;j<=4;j++){
        var y = pad.t + plotH*j/4;
        ctx.beginPath(); ctx.moveTo(pad.l,y); ctx.lineTo(w-pad.r,y); ctx.stroke();
      }
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth=1;
      ctx.beginPath(); ctx.moveTo(pad.l,pad.t); ctx.lineTo(pad.l,s.height-pad.b); ctx.lineTo(w-pad.r,s.height-pad.b); ctx.stroke();

      ctx.fillStyle = '#8a8371'; ctx.font='9.5px monospace'; ctx.textAlign='right';
      for(var j=0;j<=4;j++){
        var y = pad.t + plotH*j/4;
        ctx.fillText(Math.round(100 - j*25) + '%', pad.l-4, y+3);
      }

      var bw = plotW/pcts.length;
      var barW = bw*0.6;
      ctx.textAlign='center';
      pcts.forEach(function(p, i){
        var x = pad.l + i*bw + (bw-barW)/2;
        var barH = (p/100)*plotH;
        var y = (s.height-pad.b) - barH;
        var isLL = labels[i].indexOf('LL') === 0;
        ctx.fillStyle = isLL ? '#2980b9' : '#c0392b';
        ctx.fillRect(x,y,barW,barH);
        ctx.strokeStyle = isLL ? '#1b4f72' : '#78281f';
        ctx.lineWidth=0.5;
        ctx.strokeRect(x,y,barW,barH);
        if(barH>16){
          ctx.fillStyle='#ffffff'; ctx.font='bold 9.5px monospace';
          ctx.fillText(Math.round(p) + '%', x+barW/2, y+13);
        }
        ctx.fillStyle='#8a8371'; ctx.font='10px monospace';
        ctx.fillText(labels[i], x+barW/2, s.height-pad.b+14);
      });
    }

    function wsim_redraw(){
      wsim_currentImg = wsim_genImage(wsim_pattern);
      wsim_drawOriginal();
      wsim_drawMosaic();
      wsim_drawEnergy();
    }

    root.querySelector('#wsim_pattern').addEventListener('change', function(e){
      wsim_pattern = e.target.value; wsim_redraw();
    });
    root.querySelectorAll('input[name="wsim_lv"]').forEach(function(r){
      r.addEventListener('change', function(e){ wsim_level = +e.target.value; wsim_drawMosaic(); wsim_drawEnergy(); });
    });
    root.querySelector('#wsim_gain').addEventListener('input', function(e){
      wsim_gain = +e.target.value;
      root.querySelector('#wsim_gainv').textContent = wsim_gain.toFixed(1);
      wsim_drawMosaic();
    });

    wsim_redraw();
    window.addEventListener('resize', wsim_redraw);
  }

  function tryInitSim04Wavelet(){
    var root = document.getElementById('sim-05-wavelet');
    if (root) initSim04Wavelet(root); else setTimeout(tryInitSim04Wavelet, 200);
  }
  tryInitSim04Wavelet();
})();
</script>
""")

**Figura 5.19:** Simulazione della decomposizione *wavelet* 2D.


<figure id="fig-05-sim-05-wavelet">
  <img src="imagens/fig-05-sim-05-wavelet.png" alt=" Simulazione della decomposizione *wavelet* 2D. " style="max-width:80%" />
  <figcaption><strong>Figura 5.19:</strong>  Simulazione della decomposizione *wavelet* 2D. </figcaption>
</figure>

### 5.7.4 Analisi Multirisoluzione con la DWT 2D

La Trasformata *Wavelet* Discreta 2D (DWT) decompone un'immagine in componenti di approssimazione e dettaglio, organizzate in modo gerarchico su diverse scale e orientazioni. Poiché le sottobande di dettaglio in immagini naturali spesso presentano coefficienti a basso contrasto, gli esempi pratici che seguono utilizzano un pattern geometrico sintetico generato in Python. Questo approccio replica il comportamento del simulatore della [Figura 5.19](#fig-05-sim-05-wavelet), rendendo visivamente espliciti gli effetti del filtraggio spaziale e della decomposizione multirisoluzione.

#### 5.7.4.1 Scomposizione a Mosaico su Più Livelli

La [Figura 5.20](#fig-05-dwt-subbandas) illustra la struttura gerarchica della DWT su due livelli utilizzando la *wavelet* di Haar. Il processo si basa sull'applicazione combinata di filtri passa-basso e passa-alto nelle direzioni orizzontale e verticale, seguiti da un sottocampionamento con fattore 2.

Nel primo livello, l'immagine originale genera la sottobanda di approssimazione ($LL_1$) e le componenti di dettaglio orizzontale ($LH_1$), verticale ($HL_1$) e diagonale ($HH_1$). Nell'analisi multirisoluzione, la sottobanda $LL_1$ viene nuovamente filtrata e sottocampionata, producendo il secondo livello di scomposizione ($LL_2$, $LH_2$, $HL_2$ e $HH_2$).

Per consentire l'interpretazione visiva delle componenti di dettaglio, il codice estrae il valore assoluto dei loro coefficienti e applica una normalizzazione lineare (*min-max*) per occupare l'intera gamma dinamica dei toni di grigio [0, 255]. Questa operazione trasforma le regioni omogenee (coefficienti nulli) in nero ed evidenzia in bianco i bordi e le texture estratte a ciascuna scala e orientazione.

In [43]:
%%writefile tmp/fig_05_dwt_subbandas.cpp
#define MM_OUT "tmp/fig_05_dwt_subbandas.png"
//| label: fig-05-dwt-subbandas
//| fig-cap: "Decomposição *wavelet* 2D de 2 níveis com *wavelet* Haar: subbandas LL, LH, HL, HH em cada nível. As subbandas de detalhe revelam estruturas orientadas em diferentes escalas utilizando um padrão sintético."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <iostream>
#include <cmath>
#include <vector>
#include <string>
#include "morph.hpp"

// Função para gerar a imagem sintética (mesmo padrão 'combined' do simulador)
cv::Mat gerar_imagem_sintetica(int N = 256) {
    cv::Mat img = cv::Mat::zeros(N, N, CV_64F);
    for (int y = 0; y < N; y++) {
        for (int x = 0; x < N; x++) {
            double v = 55 + 35 * (static_cast<double>(x) / N) + 15 * std::sin(y / 24.0);
            // Quadrado
            if (24 < x && x < 100 && 24 < y && y < 100) {
                v = 225;
            }
            // Círculo
            double cx = 190, cy = 76, r = 34;
            if (std::pow(x - cx, 2) + std::pow(y - cy, 2) < r * r) {
                v = 205;
            }
            // Textura periódica (inferior)
            if (y > 164 && y < 244) {
                int p = 12;
                v = (((x / p + y / p) % 2) == 0) ? 185 : 65;
            }
            // Linha diagonal
            if (std::abs(x - y) < 4) {
                v = 240;
            }
            img.at<double>(y, x) = std::max(0.0, std::min(255.0, v));
        }
    }
    cv::Mat img_uint8;
    img.convertTo(img_uint8, CV_8U);
    return img_uint8;
}

// Normaliza subbanda para visualização [0,255]
cv::Mat sb_vis(const cv::Mat& sb) {
    cv::Mat abs_sb, norm_sb, uint8_sb;
    cv::absdiff(sb, cv::Scalar(0), abs_sb);
    cv::normalize(abs_sb, norm_sb, 0, 255, cv::NORM_MINMAX);
    norm_sb.convertTo(uint8_sb, CV_8U);
    return uint8_sb;
}

int main() {
    // Substitui a imagem escura de moedas pelo padrão sintético claro
    cv::Mat img_gray = gerar_imagem_sintetica(256);
    cv::Mat img_float;
    img_gray.convertTo(img_float, CV_64F);

    // ── Decomposição wavelet 2 níveis ─────────────────────────────────────────────
    std::string wavelet = "haar";

    // Nível 1
    mm::Subbands s1 = mm::dwt2(img_float, wavelet);
    cv::Mat LL1 = s1.LL, LH1 = s1.LH, HL1 = s1.HL, HH1 = s1.HH;

    // Nível 2 (aplicado sobre LL1)
    mm::Subbands s2 = mm::dwt2(LL1, wavelet);
    cv::Mat LL2 = s2.LL, LH2 = s2.LH, HL2 = s2.HL, HH2 = s2.HH;

    std::cout << "Forma original     : " << img_gray.rows << "x" << img_gray.cols << std::endl;
    std::cout << "LL1 (nível 1)      : " << LL1.rows << "x" << LL1.cols 
              << "  |  LH1/HL1/HH1: " << LH1.rows << "x" << LH1.cols << std::endl;
    std::cout << "LL2 (nível 2)      : " << LL2.rows << "x" << LL2.cols 
              << "    |  LH2/HL2/HH2: " << LH2.rows << "x" << LH2.cols << std::endl;

    std::vector<mm::Image> imgs_dwt = {
        mm::Image(img_gray), mm::Image(sb_vis(LL1)), mm::Image(sb_vis(LH1)), 
        mm::Image(sb_vis(HL1)), mm::Image(sb_vis(HH1)),
        mm::Image(sb_vis(LL2)), mm::Image(sb_vis(LH2)), mm::Image(sb_vis(HL2)), 
        mm::Image(sb_vis(HH2))
    };
    std::vector<std::string> titles_dwt = {
        "Original",
        "LL₁ (aprox.)", "LH₁ (horiz.)", "HL₁ (vert.)", "HH₁ (diag.)",
        "LL₂ (aprox.)", "LH₂ (horiz.)", "HL₂ (vert.)", "HH₂ (diag.)"
    };

    mm::show(imgs_dwt, MM_OUT, titles_dwt, 5);

    return 0;
}

Overwriting tmp/fig_05_dwt_subbandas.cpp


In [44]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dwt_subbandas.cpp -o tmp/fig_05_dwt_subbandas -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dwt_subbandas \
  && test -f "tmp/fig_05_dwt_subbandas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dwt_subbandas.png"

Forma original     : 256x256
LL1 (nível 1)      : 128x128  |  LH1/HL1/HH1: 128x128
LL2 (nível 2)      : 64x64    |  LH2/HL2/HH2: 64x64
[1] Original
[2] LL₁ (aprox.)
[3] LH₁ (horiz.)
[4] HL₁ (vert.)
[5] HH₁ (diag.)
[6] LL₂ (aprox.)
[7] LH₂ (horiz.)
[8] HL₂ (vert.)
[9] HH₂ (diag.)


In [45]:
try:
    mm.show(mm.read("tmp/fig_05_dwt_subbandas.png"), figsize=(16, 7))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_dwt_subbandas.png (ver a versao Python)")

<Figure size 2400x1050 with 1 Axes>

**Figura 5.20:** Decomposição *wavelet* 2D de 2 níveis com *wavelet* Haar: subbandas LL, LH, HL, HH em cada nível. As subbandas de detalhe revelam estruturas orientadas em diferentes escalas utilizando um padrão sintético.


#### 5.7.4.2 Il Compromesso tra Localizzazione e Morbidezza

La scelta della funzione di base (*wavelet*) influenza direttamente il modo in cui le caratteristiche dell'immagine vengono distribuite e codificate dai coefficienti della DWT. La [Figura 5.21](#fig-05-dwt-wavelets) confronta i risultati pratici ottenuti applicando quattro famiglie distinte sul pattern geometrico sintetico: `haar`, `db4`, `sym4` e `bior2.2`.

Poiché possiede un supporto corto e una forma a funzione a gradino, la *wavelet* di Haar produce coefficienti altamente localizzati in corrispondenza delle discontinuità spaziali, generando bordi sottili e nitidi nelle sottobande di dettaglio. Al contrario, famiglie come Daubechies (`db4`) e Symlets (`sym4`), che presentano un supporto maggiore (filtri più lunghi) e un numero più elevato di momenti nulli, generano risposte più morbide e distribuite attorno alle transizioni, il che può introdurre lievi oscillazioni o effetti di sfocatura sui confini netti.

Questo comportamento evidenzia il classico compromesso (*trade-off*) dell'analisi multirisoluzione: supporti più piccoli favoriscono la localizzazione spaziale esatta dei bordi, mentre supporti più ampi e un maggior numero di momenti nulli tendono a produrre rappresentazioni più sparse e morbide. Questa morbidezza e la capacità di attenuazione delle alte frequenze garantiscono una maggiore efficienza nella compattazione dell'energia, caratteristiche fondamentali per applicazioni di compressione dei dati e rimozione del rumore (*denoising*).

In [46]:
%%writefile tmp/fig_05_dwt_wavelets.cpp
#define MM_OUT "tmp/fig_05_dwt_wavelets.png"
//| label: fig-05-dwt-wavelets
//| fig-cap: "Comparação entre famílias de *wavelets*: Haar, db4, sym4 e bior2.2. Subbanda LL₁ (aproximação) e HH₁ (diagonal) para cada escolha, ilustrando o compromisso entre compactação e suavidade com base no padrão sintético."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include <functional>
#include "morph.hpp"
#include <iostream>

// Função auxiliar para visualizar subbandas (normalizar para 0-255)
mm::Image sb_vis(const cv::Mat& sb) {
    cv::Mat normalized;
    // Normalizar para visualização
    double minVal, maxVal;
    cv::minMaxLoc(sb, &minVal, &maxVal);
    if (maxVal - minVal > 0) {
        sb.convertTo(normalized, CV_8U, 255.0 / (maxVal - minVal), -minVal * 255.0 / (maxVal - minVal));
    } else {
        normalized = cv::Mat::zeros(sb.size(), CV_8U);
    }
    return mm::Image(normalized);
}

// Função para gerar imagem sintética (fallback)
cv::Mat gerar_imagem_sintetica(int N = 256) {
    cv::Mat img(N, N, CV_64F);
    for (int y = 0; y < N; y++) {
        for (int x = 0; x < N; x++) {
            double v = 55 + 35 * ((double)x / N) + 15 * std::sin(y / 24.0);
            if (24 < x && x < 100 && 24 < y && y < 100) v = 225;
            int cx = 190, cy = 76, r = 34;
            if ((x - cx) * (x - cx) + (y - cy) * (y - cy) < r * r) v = 205;
            if (y > 164 && y < 244) {
                int p = 12;
                v = (((x / p + y / p) % 2 == 0) ? 185 : 65);
            }
            if (std::abs(x - y) < 4) v = 240;
            v = std::max(0.0, std::min(255.0, v));
            img.at<double>(y, x) = v;
        }
    }
    cv::Mat img8u;
    img.convertTo(img8u, CV_8U);
    return img8u;
}

int main() {
    // Garante que img_gray e img_float utilizem o mesmo padrão sintético claro
    cv::Mat img_gray;

    // Fallback: gerar imagem sintética (não temos acesso a variáveis de outros blocos)
    img_gray = gerar_imagem_sintetica(256);

    cv::Mat img_float;
    img_gray.convertTo(img_float, CV_64F);

    std::vector<std::string> wavelets_comp = {"haar", "db4", "sym4", "bior2.2"};
    std::vector<mm::Image> imgs_comp;
    std::vector<std::string> titles_comp;

    for (const auto& wname : wavelets_comp) {
        // pywt.dwt2 -> mm::dwt2
        mm::Subbands sub = mm::dwt2(img_float, wname);
        cv::Mat LL = sub.LL;
        cv::Mat LH = sub.LH;
        cv::Mat HL = sub.HL;
        cv::Mat HH = sub.HH;

        imgs_comp.push_back(sb_vis(LL));
        imgs_comp.push_back(sb_vis(HH));
        titles_comp.push_back(wname + " — LL₁");
        titles_comp.push_back(wname + " — HH₁");
    }

    mm::show(imgs_comp, MM_OUT, titles_comp, 4);

    return 0;
}

Overwriting tmp/fig_05_dwt_wavelets.cpp


In [47]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dwt_wavelets.cpp -o tmp/fig_05_dwt_wavelets -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dwt_wavelets \
  && test -f "tmp/fig_05_dwt_wavelets.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dwt_wavelets.png"

[1] haar — LL₁
[2] haar — HH₁
[3] db4 — LL₁
[4] db4 — HH₁
[5] sym4 — LL₁
[6] sym4 — HH₁
[7] bior2.2 — LL₁
[8] bior2.2 — HH₁


In [48]:
try:
    mm.show(mm.read("tmp/fig_05_dwt_wavelets.png"), figsize=(14, 8))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_dwt_wavelets.png (ver a versao Python)")

<Figure size 2100x1200 with 1 Axes>

**Figura 5.21:** Comparação entre famílias de *wavelets*: Haar, db4, sym4 e bior2.2. Subbanda LL₁ (aproximação) e HH₁ (diagonal) para cada escolha, ilustrando o compromisso entre compactação e suavidade com base no padrão sintético.


#### 5.7.4.3 Limiarizzazione dei Coefficienti e Compressione

Una delle principali applicazioni della Trasformata *Wavelet* Discreta (DWT) è la compressione dei dati, favorita dalla capacità di rappresentazione **sparsa** dei coefficienti. La [Figura 5.22](#fig-05-dwt-reconstrucao) illustra l'effetto della limiarizzazione netta (*hard thresholding*), tecnica in cui i coefficienti di dettaglio con magnitudine inferiore a una soglia $T$ vengono integralmente azzerati prima del processo di sintesi eseguito dalla Trasformata *Wavelet* Discreta Inversa (IDWT).

Man mano che la soglia $T$ viene aumentata, un volume crescente di coefficienti ad alta frequenza viene azzerato. Poiché concentrano meno energia, la rimozione di queste componenti riduce considerevolmente la quantità di informazione necessaria per rappresentare l'immagine, mantenendo intatta la componente di approssimazione globale (la sottobanda $LL$ più profonda) per preservare la struttura macro. Visivamente, questo scarto di coefficienti si manifesta attraverso la scomparsa progressiva delle trame fini e la levigatura delle transizioni brusche di intensità.

La fedeltà dell'immagine ricostruita rispetto all'originale è quantificata dalla metrica del **Picco del Rapporto Segnale-Rumore (PSNR, *Peak Signal-to-Noise Ratio*)**, espressa in decibel (dB). Valori più elevati di PSNR indicano una distorsione minore e una maggiore prossimità matematica al segnale originale. L'esperimento pratico evidenzia il decadimento graduale del PSNR all'aumentare dell'aggressività della limiarizzazione, consentendo di valutare numericamente la soglia ottimale per il bilanciamento tra compressione e degrado visivo.

In [49]:
%%writefile tmp/fig_05_dwt_reconstrucao.cpp
#define MM_OUT "tmp/fig_05_dwt_reconstrucao.png"
// Compile with: g++ -std=c++17 -O2 snippet.cpp -o snippet $(pkg-config --cflags --libs opencv4) -I/path/to/morph/include
#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include <cstdint>
#include "morph.hpp"

// Helper function to generate synthetic image (same as Python fallback)
cv::Mat gerar_imagem_sintetica(int N = 256) {
    cv::Mat img(N, N, CV_64F);
    for (int y = 0; y < N; ++y) {
        for (int x = 0; x < N; ++x) {
            double v = 55 + 35 * (double(x) / N) + 15 * std::sin(double(y) / 24);
            if (24 < x && x < 100 && 24 < y && y < 100) v = 225;
            int cx = 190, cy = 76, r = 34;
            if ((x - cx) * (x - cx) + (y - cy) * (y - cy) < r * r) v = 205;
            if (y > 164 && y < 244) {
                int p = 12;
                v = (((x / p + y / p) % 2 == 0)) ? 185 : 65;
            }
            if (std::abs(x - y) < 4) v = 240;
            img.at<double>(y, x) = std::max(0.0, std::min(255.0, v));
        }
    }
    cv::Mat img8;
    img.convertTo(img8, CV_8U);
    return img8;
}

cv::Mat dwt_threshold_reconstruct(const cv::Mat& img, const std::string& wavelet = "db4", int nivel = 2, double threshold = 0.0) {
    cv::Mat img64;
    img.convertTo(img64, CV_64F);

    // Perform decomposition using mm::wavedec2
    mm::WaveDec2 c = mm::wavedec2(img64, wavelet, nivel);

    // Apply thresholding to detail coefficients
    mm::WaveDec2 c_t;
    c_t.LL = c.LL.clone();  // LL not thresholded
    c_t.detail.resize(c.detail.size());

    for (size_t j = 0; j < c.detail.size(); ++j) {
        c_t.detail[j].resize(3);
        for (int k = 0; k < 3; ++k) {
            c_t.detail[j][k] = mm::wave_threshold(c.detail[j][k], threshold, "hard");
        }
    }

    // Reconstruct via mm::waverec2
    cv::Mat rec = mm::waverec2(c_t, wavelet);

    // Crop to original dimensions and clip to [0, 255]
    rec = rec(cv::Rect(0, 0, img.cols, img.rows)).clone();
    cv::Mat rec8;
    rec.convertTo(rec8, CV_8U);
    return rec8;
}

int main() {
    // Ensure img_gray uses the same clear synthetic pattern
    cv::Mat img_gray = gerar_imagem_sintetica(256);

    std::vector<int> thresholds = {0, 10, 30, 60, 100};
    std::vector<mm::Image> imgs_thr;
    std::vector<std::string> titles_thr;

    imgs_thr.push_back(mm::Image(img_gray));
    titles_thr.push_back("Original");

    for (int t : thresholds) {
        cv::Mat rec = dwt_threshold_reconstruct(img_gray, "db4", 2, double(t));
        double psnr = cv::PSNR(img_gray, rec);
        imgs_thr.push_back(mm::Image(rec));
        titles_thr.push_back("T=" + std::to_string(t) + "  PSNR=" + 
                            std::to_string(psnr).substr(0, std::to_string(psnr).find('.') + 2) + " dB");
    }

    mm::show(imgs_thr, MM_OUT, titles_thr, 3);

    return 0;
}

Overwriting tmp/fig_05_dwt_reconstrucao.cpp


In [50]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dwt_reconstrucao.cpp -o tmp/fig_05_dwt_reconstrucao -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dwt_reconstrucao \
  && test -f "tmp/fig_05_dwt_reconstrucao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dwt_reconstrucao.png"

[1] Original
[2] T=0  PSNR=361.2 dB
[3] T=10  PSNR=42.4 dB
[4] T=30  PSNR=32.5 dB
[5] T=60  PSNR=28.0 dB
[6] T=100  PSNR=23.1 dB


In [51]:
try:
    mm.show(mm.read("tmp/fig_05_dwt_reconstrucao.png"), figsize=(14, 10))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_dwt_reconstrucao.png (ver a versao Python)")

<Figure size 2100x1500 with 1 Axes>

**Figura 5.22:** Reconstrução *wavelet* com limiarização de coeficientes (*hard thresholding*): à medida que o limiar aumenta, mais detalhes são zerados, produzindo imagens progressivamente mais suaves. Métrica PSNR quantifica a perda de qualidade sobre o padrão sintético.


### Sintesi — Fourier vs. *Wavelet*: quando utilizzare ciascun approccio?

La [Tabela 5.5](#tbl-05-fourier-wavelet) sintetizza le principali differenze strutturali e operative tra la Trasformata Discreta di Fourier (DFT) e la Trasformata *Wavelet* Discreta (DWT).

<a id="tbl-05-fourier-wavelet"></a>

**Tabela 5.5:** Confronto tra la Trasformata Discreta di Fourier (DFT) e la Trasformata *Wavelet* Discreta (DWT), evidenziandone le principali caratteristiche e applicazioni.

| Criterio | Fourier (DFT) | *Wavelet* (DWT) |
|:---|:---|:---|
| **Funzioni di base** | Sinusoidi di supporto infinito | Funzioni di supporto compatto |
| **Localizzazione spaziale** | Non esplicita (globale) | Esplicita (locale) |
| **Filtraggio spettrale** | Eccellente per il controllo fine delle frequenze | Basata su sottobande (scale) |
| **Compressione delle immagini** | Base della DCT (JPEG tradizionale) | Base della DWT (JPEG 2000) |
| **Analisi multiscala** | No | Sì |
| **Rimozione del rumore periodico** | Altamente efficiente | Poco indicata |
| **Segnali non stazionari** | Limitata | Altamente efficiente |


In termini pratici, la DFT si consolida come lo strumento ideale per l'analisi spettrale pura, la progettazione di filtri selettivi nel dominio della frequenza e l'attenuazione di rumori periodici e armonici. D'altro canto, la DWT eccelle in scenari che richiedono la rigorosa preservazione della localizzazione spaziale delle caratteristiche associata al loro contenuto frequenziale, distinguendosi nella compressione dei dati, nell'analisi multirisoluzione e nell'elaborazione di transizioni brusche. Pertanto, entrambe le trasformate devono essere comprese come tecniche perfettamente complementari, che tracciano percorsi distinti e specifici per la risoluzione di problemi nell'ambito dell'elaborazione digitale di immagini e visione artificiale (PDI-VC).

> ### 📝 Analogie con l'Audio: Limitazioni e Precauzioni
>
> Nel tracciare analogie tra l'elaborazione delle immagini e l'audio, è importante considerare le differenze fondamentali:
>
> * Nei sistemi audio stereo/multicanale, la fase tra i canali è cruciale per la percezione della localizzazione spaziale (differenze interaurali di fase e di tempo).
>
> * Nei sistemi monoaurali, la fase ha un'influenza percettiva limitata — l'orecchio umano è relativamente insensibile alla fase assoluta di componenti sinusoidali isolate.
>
> * Nelle immagini, la fase della DFT è sempre fondamentale per la localizzazione spaziale delle strutture, indipendentemente dal fatto che l'immagine sia monocromatica o a colori.
>
> L'analogia tra la fase nell'audio e la fase nelle immagini deve essere utilizzata con cautela, evidenziando che, sebbene entrambe trasportino informazioni sull'organizzazione spaziale/temporale del segnale, i meccanismi percettivi sono fondamentalmente differenti.

## 5.8 Compressione delle Immagini

Mentre le *wavelet* stabiliscono il fondamento teorico dello standard JPEG 2000, lo standard JPEG tradizionale si basa sulla **Trasformata Discreta dei Coseni (DCT, *Discrete Cosine Transform*)**. Nonostante le differenze strutturali, entrambi gli approcci condividono lo stesso principio fondamentale: compattare l'energia dell'immagine in un numero ridotto di coefficienti e scartare le componenti di minore rilevanza con impatto visivo minimo.

L'obiettivo centrale della compressione è ridurre il volume di dati necessario per l'archiviazione o la trasmissione di un'immagine. Tale processo è reso possibile dall'identificazione e dall'eliminazione di **ridondanze** strutturali e percettive.

### 5.8.1 Tassonomia delle Ridondanze

Lo sviluppo di algoritmi di compressione si fonda sull'identificazione e sull'eliminazione di tre categorie principali di ridondanza, sintetizzate nella [Tabela 5.6](#tbl-05-redundancias).

<a id="tbl-05-redundancias"></a>

**Tabela 5.6:** Categorie di ridondanza nelle immagini digitali e i rispettivi meccanismi di sfruttamento.

| Tipo | Definizione | Approccio di Sfruttamento |
|:---|:---|:---|
| **Spaziale (interpixel)** | Elevata correlazione e dipendenza statistica tra pixel adiacenti. | DCT, DWT e codifica predittiva. |
| **Spettrale (intercanale)** | Correlazione statistica tra i canali di colore di una stessa immagine. | Trasformazioni dello spazio colore (es: RGB in $YC_bC_r$). |
| **Psicovisuale** | Insensibilità del sistema visivo umano (SVH) alle variazioni ad alta frequenza e a basso contrasto. | Processi di quantizzazione selettiva dei coefficienti. |


A seconda della preservazione dell'informazione originale dopo il processo di decodifica, i metodi di compressione si dividono in due classi fondamentali:

* **Senza perdita (*lossless*):** Garantisce una ricostruzione bit per bit identica all'immagine originale. Viene impiegata in scenari in cui l'integrità dei dati è strettamente critica, come nelle immagini mediche, nella diagnostica per immagini e nell'archiviazione di documenti testuali.
* **Con perdita (*lossy*):** Ammette l'introduzione di una distorsione controllata nel segnale in cambio di tassi di compressione sostanzialmente più elevati. È l'approccio standard per le fotografie di consumo e lo *streaming* video, ecosistemi nei quali il SVH tollera piccole attenuazioni dell'alta frequenza senza percezione di degrado della qualità visiva.

### 5.8.2 Trasformata del Coseno Discreta (DCT-II 2D)

La **Trasformata del Coseno Discreta** (DCT) costituisce l'operazione centrale dello standard JPEG. A differenza della DFT, che utilizza una base complessa, la DCT si basa su funzioni trigonometriche puramente reali. Per un blocco immagine $f(x,y)$ di dimensioni $N \times N$, la **DCT-II 2D** mappa il segnale spaziale nel dominio delle frequenze spaziali, generando la matrice dei coefficienti $C(u,v)$ tramite:

<a id="eq-05-dct"></a>
$$
C(u,v) = \alpha(u)\,\alpha(v) \sum_{x=0}^{N-1}\sum_{y=0}^{N-1} f(x,y)\,
\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]
\cos\!\left[\frac{\pi(2y+1)v}{2N}\right] \tag{5.9}
$$


dove i fattori di normalizzazione ortogonale sono dati da $\alpha(0) = \sqrt{1/N}$ e $\alpha(k) = \sqrt{2/N}$ per $k > 0$.

Ogni coefficiente $C(u,v)$ quantifica il contributo — o "peso" — di una specifica frequenza spaziale all'interno di quel blocco. Il termine $C(0,0)$ è denominato **componente DC** e rappresenta l'intensità media del blocco (frequenza nulla). I restanti coefficienti, chiamati **componenti AC** (*Alternating Current*), corrispondono a frequenze spaziali progressivamente più elevate.

### 5.8.3 Le Funzioni di Base della DCT

Da una prospettiva geometrica, la [Equação 5.9](#eq-05-dct) realizza la proiezione del blocco di pixel su un insieme di funzioni ortogonali. Per il caso standard del JPEG ($N=8$), il blocco spaziale viene decomposto in una combinazione lineare di **64 funzioni di base** bidimensionali, denotate da $B_{u,v}(x,y)$ e generate dal prodotto di funzioni cosinusoidali:

$$B_{u,v}(x,y) = \cos\left[ \frac{\pi (2x+1)u}{16} \right] \cos\left[ \frac{\pi (2y+1)v}{16} \right]$$

In questo modo, l'operazione inversa può essere interpretata come la ricostruzione esatta del blocco originale tramite la somma ponderata di queste 64 matrici di base, dove ogni coefficiente $C(u,v)$ funge da peso analitico della rispettiva componente armonica.

La **frequenza spaziale** indicata dagli indici $(u,v)$ determina il numero di cicli di oscillazione lungo le dimensioni orizzontali e verticali del blocco. Come illustrato nella [Figura 5.23](#fig-05-dct-basis) — il cui codice isola ciascuna base applicando la trasformazione inversa su impulsi unitari —, queste 64 funzioni sono organizzate in una matrice $8 \times 8$. L'angolo superiore sinistro ($u=0, v=0$) mostra il pattern uniforme a frequenza nulla (DC), mentre il progredire verso destra (asse $u$) o verso il basso (asse $v$) mappa variazioni armoniche progressivamente maggiori, rappresentando transizioni rapide, bordi e trame nelle orientazioni orizzontali, verticali e diagonali.

> ### 📝 DCT vs DFT: Vantaggio della Compattazione dell'Energia
>
> Sia la DCT che la DFT mappano un blocco spaziale $N \times N$ in una matrice di coefficienti della stessa dimensione. Tuttavia, per immagini naturali, la DCT presenta una maggiore efficienza nella **compattazione dell'energia** alle basse frequenze. Ciò avviene perché la DCT assume implicitamente una simmetria pari del segnale ai bordi del blocco, il che equivale a un'estensione periodica continua, minimizzando l'effetto di dispersione spettrale (*ringing*). Di conseguenza, la maggior parte dei coefficienti AC decade rapidamente verso valori prossimi allo zero, ottimizzando il *pipeline* di compressione senza introdurre una degradazione visiva percepibile.

In [52]:
%%writefile tmp/fig_05_dct_basis.cpp
#define MM_OUT "tmp/fig_05_dct_basis.png"
//| label: fig-05-dct-basis
//| fig-cap: "O Alfabeto Visual do JPEG: As 64 funções de base da DCT-II. O coeficiente DC fica no topo esquerdo (suave). Ao descer e avançar à direita, a oscilação espacial aumenta drasticamente."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
    // Cada base é a IDCT de um único coeficiente unitário — montadas num mosaico 8×8.
    const int tile = 32;
    cv::Mat montagem(8 * tile, 8 * tile, CV_8UC1);
    for (int i = 0; i < 8; i++) {
        for (int j = 0; j < 8; j++) {
            cv::Mat coef = cv::Mat::zeros(8, 8, CV_64F);
            coef.at<double>(i, j) = 1.0;
            cv::Mat base = mm::idct2(coef);
            base = mm::idct2(coef);
            cv::normalize(base, base, 0, 255, cv::NORM_MINMAX, CV_8U);
            cv::Mat base_resized;
            cv::resize(base, base_resized, cv::Size(tile, tile), 0, 0, cv::INTER_NEAREST);
            base_resized.copyTo(montagem(cv::Rect(j * tile, i * tile, tile, tile)));
        }
    }

    mm::Image montagem_img = montagem;
    std::vector<mm::Image> images = {montagem_img};
    std::vector<std::string> titles = {"As 64 bases da DCT-II 8×8 (DC no topo-esquerdo)"};
    mm::show(images, MM_OUT, titles, 1);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(montagem, "tmp/fig_05_dct_basis_0.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_dct_basis.cpp


In [53]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dct_basis.cpp -o tmp/fig_05_dct_basis -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dct_basis \
  && test -f "tmp/fig_05_dct_basis.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dct_basis.png"

[1] As 64 bases da DCT-II 8×8 (DC no topo-esquerdo)


In [54]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_dct_basis_0.png"),
        ],
        titles=[
            'As 64 bases da DCT-II 8×8 (DC no topo-esquerdo)',
        ],
        cols=1,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_dct_basis_0.png (ver a versao Python)")

<Figure size 750x750 with 1 Axes>

**Figura 5.23:** O Alfabeto Visual do JPEG: As 64 funções de base da DCT-II. O coeficiente DC fica no topo esquerdo (suave). Ao descer e avançar à direita, a oscilação espacial aumenta drasticamente.


### 5.8.4 Concentrazione di Energia e Ricostruzione Progressiva

Prima dell'applicazione della DCT, i pixel del blocco di intensità vengono di routine traslati (sottraendo $128$ per immagini a 8 bit) al fine di centrare il segnale attorno allo zero, eliminando componenti continue superflue. Calcolando la DCT sul blocco risultante, la proprietà di **compattazione dell'energia** diventa evidente: la quasi totalità della varianza e dell'informazione dell'immagine originale si concentra nel coefficiente DC ($C(0,0)$) e nei primi armonici AC a bassa frequenza.

La [Figura 5.24](#fig-05-dct-bloco) dimostra questo fenomeno mediante una ricostruzione progressiva per troncamento brusco. Invece di utilizzare tutti i 64 coefficienti, l'algoritmo conserva solo i primi $k$ componenti — selezionati in base a una scansione che privilegia le basse frequenze spaziali — e azzera i rimanenti.

La sintesi inversa (**IDCT**) eseguita con solo una frazione dei coefficienti (come il 15% o il 30%) è già in grado di recuperare le strutture e l'illuminazione macro del blocco originale di pixel. Man mano che gli armonici a frequenze più elevate vengono progressivamente reintegrati, i dettagli fini e le transizioni rapide vengono ripristinati. Questo comportamento valida il principio della compressione percettiva: le alte frequenze scartate possiedono poca energia e la loro assenza, in condizioni normali, genera un impatto visivo secondario sulla percezione dell'osservatore.

In [55]:
%%writefile tmp/fig_05_dct_bloco.cpp
#define MM_OUT "tmp/fig_05_dct_bloco.png"
//| label: fig-05-dct-bloco
//| fig-cap: "DCT 2D em bloco 8×8: coefficienti e ricostruzione progressiva."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <cmath>
#include <cstdio>
#include <vector>
#include <string>
#include <algorithm>
#include "morph.hpp"

static cv::Mat dct2_block(const cv::Mat& bloco) {
    // DCT-II 2D ortogonale (separabile).
    cv::Mat temp, result;
    cv::dct(bloco, temp);
    return temp.clone();
}

static cv::Mat idct2_block(const cv::Mat& coefs) {
    // IDCT-II 2D ortogonale.
    cv::Mat result;
    cv::idct(coefs, result);
    return result;
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    // ── Blocco 8×8 centrato sull'immagine ─────────────────────────────────────
    int cy = img_gray.h / 2;
    int cx = img_gray.w / 2;

    cv::Mat img_mat(img_gray.h, img_gray.w, CV_8UC1, img_gray.data.data());
    cv::Mat bloco_f = img_mat(cv::Rect(cx, cy, 8, 8)).clone();
    bloco_f.convertTo(bloco_f, CV_64F);
    bloco_f -= 128.0;

    // DCT 2D del blocco
    cv::Mat C = dct2_block(bloco_f);

    printf("Coefficienti DCT del blocco 8×8:\n");
    // Stampare i coefficienti arrotondati
    for (int i = 0; i < 8; i++) {
        for (int j = 0; j < 8; j++) {
            printf("%4d ", (int)std::round(C.at<double>(i, j)));
        }
        printf("\n");
    }

    double dc_energy = C.at<double>(0, 0) * C.at<double>(0, 0);
    double total_energy = 0.0;
    for (int i = 0; i < 8; i++)
        for (int j = 0; j < 8; j++)
            total_energy += C.at<double>(i, j) * C.at<double>(i, j);

    printf("\nEnergia DC     : %.1f\n", dc_energy);
    printf("Energia totale : %.1f\n", total_energy);
    printf("Frazione nel DC: %.1f%% ← concentrazione di energia\n", 
           (dc_energy / total_energy) * 100.0);

    // ── Ricostruzione progressiva ──────────────────────────────────────────────
    std::vector<mm::Image> imgs_rec;
    std::vector<std::string> titles_rec;

    cv::Mat bloco_orig_f;
    bloco_f.convertTo(bloco_orig_f, CV_8U);
    bloco_orig_f += 128;
    cv::Mat bloco_show;
    cv::normalize(bloco_orig_f, bloco_show, 0, 255, cv::NORM_MINMAX, CV_8U);
    imgs_rec.push_back(mm::Image(bloco_show));
    titles_rec.push_back("Blocco originale\n(8×8 pixel)");

    std::vector<int> keeps = {1, 4, 10, 20, 40, 64};
    for (int keep : keeps) {
        cv::Mat C_trunc = cv::Mat::zeros(8, 8, CV_64F);

        // Costruire l'ordine zig-zag semplicemente usando la somma u+v
        std::vector<std::pair<int,int>> indices;
        for (int u = 0; u < 8; u++)
            for (int v = 0; v < 8; v++)
                indices.push_back({u, v});

        std::sort(indices.begin(), indices.end(), 
                  [](const std::pair<int,int>& a, const std::pair<int,int>& b) {
                      return (a.first + a.second) < (b.first + b.second);
                  });

        for (int i = 0; i < keep && i < (int)indices.size(); i++) {
            int u = indices[i].first;
            int v = indices[i].second;
            C_trunc.at<double>(u, v) = C.at<double>(u, v);
        }

        cv::Mat rec = idct2_block(C_trunc);
        rec += 128.0;

        cv::Mat rec_8u;
        rec.convertTo(rec_8u, CV_8U);

        imgs_rec.push_back(mm::Image(rec_8u));
        char buf[64];
        snprintf(buf, sizeof(buf), "%d coef.\n(%.0f%% del totale)", keep, 
                 (keep / 64.0) * 100.0);
        titles_rec.push_back(buf);
    }

    mm::show(imgs_rec, MM_OUT, titles_rec, 4);

    return 0;
}

Overwriting tmp/fig_05_dct_bloco.cpp


In [56]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dct_bloco.cpp -o tmp/fig_05_dct_bloco -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dct_bloco \
  && test -f "tmp/fig_05_dct_bloco.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dct_bloco.png"

Coefficienti DCT del blocco 8×8:
 192  -48    1    8    0   -1    0    0 
 -96   54    7  -10    0    0    0    0 
  14  -14    8    0    0    0    0    0 
   0    0  -11    1    0    0    0    0 
   9  -11    0    0    1    0    1    0 
   0    0    0    0    0    0    0    0 
   0    0    0    0   -1    0    0    0 
   0    0    0    0    0    0    0    0 

Energia DC     : 36816.0
Energia totale : 52253.0
Frazione nel DC: 70.5% ← concentrazione di energia
[1] Blocco originale
(8×8 pixel)
[2] 1 coef.
(2% del totale)
[3] 4 coef.
(6% del totale)
[4] 10 coef.
(16% del totale)
[5] 20 coef.
(31% del totale)
[6] 40 coef.
(62% del totale)
[7] 64 coef.
(100% del totale)


In [57]:
try:
    mm.show(mm.read("tmp/fig_05_dct_bloco.png"), figsize=(12, 7))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_dct_bloco.png (ver a versao Python)")

<Figure size 1800x1050 with 1 Axes>

**Figura 5.24:** DCT 2D em bloco 8×8: coeficientes e reconstrução progressiva.


### 5.8.5 Il *Pipeline* di Compressione JPEG

Lo standard JPEG opera dividendo l'immagine in blocchi disgiunti di $8 \times 8$ pixel, elaborati tramite una sequenza di trasformazioni spaziali, percettive e statistiche. Il *pipeline* completo di codifica è strutturato in sei fasi principali:

$$
\text{RGB} \xrightarrow{\text{(1) } YC_bC_r} \xrightarrow{\text{(2) Sottocampionamento}} \xrightarrow{\text{(3) Blocchi } 8 \times 8} \xrightarrow{\text{(4) DCT}} \xrightarrow{\text{(5) Quantizzazione}} \xrightarrow{\text{(6) Codifica Entropica}}
$$

La [Tabela 5.7](#tbl-05-pipeline-jpeg) dettaglia la funzione analitica e il fondamento percettivo che giustifica ciascuna di queste fasi.

<a id="tbl-05-pipeline-jpeg"></a>

**Tabela 5.7:** Fasi del *pipeline* di compressione JPEG e i rispettivi fondamenti di progetto.

| Fase | Operazione | Fondamento Percettivo e Statistico |
|:---:|:---|:---|
| **1** | Conversione $RGB \rightarrow YC_bC_r$ | Separa la luminanza ($Y$) dalla crominanza ($C_b, C_r$). Il sistema visivo umano (SVH) presenta maggiore sensibilità alle variazioni di luminosità che di colore. |
| **2** | Sottocampionamento della crominanza (es: 4:2:0) | Riduce la risoluzione spaziale dei canali colore della metà, scartando dati ridondanti con impatto visivo trascurabile. |
| **3–4** | Centratura e applicazione della DCT $8 \times 8$ | Trasla i pixel nell'intervallo $[-128, 127]$ e compatta l'energia spettrale del blocco nei coefficienti a bassa frequenza. |
| **5** | Quantizzazione lineare selettiva | Divide ciascun coefficiente $C(u,v)$ per l'elemento corrispondente della matrice $Q(u,v)$, applicando un arrotondamento a interi. Costituisce la principale fonte di compressione con perdita. |
| **6** | Scansione a zig-zag e codifica | Ordina i coefficienti quantizzati per massimizzare le sequenze nulle consecutive, ottimizzando la codifica a lunghezza di corsa (RLE) e la codifica di Huffman. |


La **matrice di quantizzazione** $Q(u,v)$ è il meccanismo centrale di controllo del compromesso tra tasso di compressione e qualità visiva. Nell'algoritmo pratico della [Figura 5.25](#fig-05-jpeg-pipeline), il fattore di qualità stabilito dall'utente (scala da 1 a 100) viene convertito in uno scalare che parametrizza la severità della matrice $Q$. Valori ridotti di qualità espandono i divisori di $Q(u,v)$, forzando il troncamento di massa dei coefficienti AC a zero. Quando questa eliminazione è eccessiva, la discontinuità ai confini dei blocchi adiacenti non viene attenuata nella ricostruzione, generando i cosiddetti **artefatti a blocchi** (*blocking artifacts*).

#### La Logica della Scansione a Zig-Zag

L'efficienza del codificatore entropico successivo alla quantizzazione dipende direttamente dall'ordinamento dei dati. Poiché la DCT concentra l'energia vitale nel vertice superiore sinistro della matrice (basse frequenze) e spinge i coefficienti nulli verso le estremità opposte, la lettura lineare per righe o colonne frammenterebbe le sequenze di zeri.

L'ordinamento a zig-zag risolve questa limitazione percorrendo la matrice diagonalmente in ordine crescente di frequenza spaziale. Questa mappatura raggruppa i coefficienti significativi all'inizio del vettore e concentra i coefficienti nulli in un'unica sequenza continua alla fine dell'arrangiamento, consentendo all'algoritmo RLE di codificare grandi blocchi di dati in modo compatto ed efficiente.

> ### 📝 5.9 Cos'è l'RLE?
>
> **RLE** (*Run-Length Encoding*) è una tecnica di compressione senza perdita che codifica sequenze consecutive di valori identici — specialmente **zeri** — come una coppia (conteggio, valore). Nel JPEG, dopo la scansione a zig-zag, i coefficienti quantizzati sono organizzati in modo che gli zeri si concentrino alla fine del vettore. L'RLE comprime quindi questa lunga corsa di zeri con estrema efficienza, ottimizzando l'archiviazione e la trasmissione dell'immagine compressa.

In [58]:
%%writefile tmp/fig_05_jpeg_pipeline.cpp
#define MM_OUT "tmp/fig_05_jpeg_pipeline.png"
//| label: fig-05-jpeg-pipeline
//| fig-cap: "*Pipeline* JPEG simplificado aplicado à imagem clássica do *Cameraman*: DCT em blocos 8×8, quantização com diferentes fatores de qualidade e reconstrução via IDCT. Os artefatos de bloco (*blocking artifacts*) tornam-se visualmente evidentes em fatores de qualidade reduzidos ($Q=10$ e $Q=25$)."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cstdio>
#include "morph.hpp"


int main() {
    // Pipeline JPEG (DCT 8x8 -> quantizacao -> IDCT) via mm.jpegCompress,
    // na imagem classica do Cameraman (asset do capitulo) reduzida a 256x256.
    cv::Mat img_src_full = cv::imread("imagens/cameraman.png", cv::IMREAD_GRAYSCALE);
    cv::Mat img_src_resized;
    cv::resize(img_src_full, img_src_resized, cv::Size(256, 256));
    mm::Image img_src = mm::gray(img_src_resized);

    std::vector<mm::Image> imgs;
    std::vector<std::string> titles;
    imgs.push_back(img_src);
    titles.push_back("Original (Cameraman)");

    int qs[] = {10, 25, 50, 75, 90};
    for (int q : qs) {
        mm::Image rec = mm::jpegCompress(img_src, q);
        imgs.push_back(rec);
        char title[100];
        double psnr_val = mm::psnr(img_src, rec);
        std::snprintf(title, sizeof(title), "Q=%d (PSNR=%.1f dB)", q, psnr_val);
        titles.push_back(title);
    }

    mm::show(imgs, MM_OUT, titles, 3);

    return 0;
}

Overwriting tmp/fig_05_jpeg_pipeline.cpp


In [59]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_jpeg_pipeline.cpp -o tmp/fig_05_jpeg_pipeline -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_jpeg_pipeline \
  && test -f "tmp/fig_05_jpeg_pipeline.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_jpeg_pipeline.png"

[1] Original (Cameraman)
[2] Q=10 (PSNR=28.0 dB)
[3] Q=25 (PSNR=30.7 dB)
[4] Q=50 (PSNR=32.8 dB)
[5] Q=75 (PSNR=35.2 dB)
[6] Q=90 (PSNR=40.0 dB)


In [60]:
try:
    mm.show(mm.read("tmp/fig_05_jpeg_pipeline.png"), figsize=(14, 10))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_jpeg_pipeline.png (ver a versao Python)")

<Figure size 2100x1500 with 1 Axes>

**Figura 5.25:** *Pipeline* JPEG simplificado aplicado à imagem clássica do *Cameraman*: DCT em blocos 8×8, quantização com diferentes fatores de qualidade e reconstrução via IDCT. Os artefatos de bloco (*blocking artifacts*) tornam-se visualmente evidentes em fatores de qualidade reduzidos ($Q=10$ e $Q=25$).


### 5.9.1 Simulatore Interattivo: Quantizzazione DCT

Il simulatore della [Figura 5.26](#fig-05-sim-05-dct) consente di esplorare l'impatto del processo di quantizzazione su un blocco $8 \times 8$ estratto da un'immagine reale, sintetizzando in tempo reale le seguenti componenti:

* **Blocco originale e ricostruito:** Rappresentazione diretta dei pixel nel dominio spaziale in scala di grigi [0, 255].
* **Coefficienti DCT:** Distribuzione dell'energia mappata in modo logaritmico su un gradiente cromatico, evidenziando la concentrazione di intensità nel vertice superiore sinistro (basse frequenze).
* **Coefficienti quantizzati:** Visualizzazione dei valori interi risultanti dalla divisione per la matrice $Q(u,v)$, rendendo visivamente esplicita l'emergenza in massa di coefficienti nulli (in toni scuri) al diminuire del fattore di qualità.
* **Metriche di compressione:** Pannello di monitoraggio che quantifica l'Errore Quadratico Medio (MSE), il numero di coefficienti preservati e il volume di zeri generati per la codifica entropica.

In [61]:
# @title { display-mode: "form" }
from IPython.display import HTML

HTML("""
<div id="sim-05-dct" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-dct * { box-sizing: border-box; }
  #sim-05-dct canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-dct button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-dct button:hover { background: #e8dfcf; }
  #sim-05-dct .sim04_dct_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_dct_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim04_dct_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 90px; }
  .sim04_dct_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim04_dct_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim04_dct_slider_container { display: flex; align-items: center; gap: 8px; margin-bottom: 10px; }
  .sim04_dct_slider_container label { font-size: 11px; font-weight: 700; color: #5e5a4a; min-width: 110px; }
  .sim04_dct_slider_container input[type=range] { flex: 1; cursor: pointer; height: 4px; accent-color: #2980b9; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⊞ Simulatore: Quantizzazione DCT-JPEG (blocco 8×8)</span>
  <span class="sim04_dct_pill">blocchi 8×8</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Qualità</div><div id="sim04_dct_qual" class="sim04_dct_stat_value" style="color:#2980b9;">50</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Coef. ≠ 0</div><div id="sim04_dct_nonzero" class="sim04_dct_stat_value" style="color:#27ae60;">–</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Zeri</div><div id="sim04_dct_zeros" class="sim04_dct_stat_value" style="color:#c0392b;">–</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Errore MSE</div><div id="sim04_dct_mse" class="sim04_dct_stat_value" style="color:#b9770e;">–</div></div>
  </div>

  <!-- Grid de Visualização dos Blocos -->
  <div style="display:flex; gap:12px; flex-wrap:wrap; align-items:flex-start; justify-content:center; margin-bottom:14px;">
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Blocco Originale (8×8)</div>
      <canvas id="sim04_dct_cvOrig" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Coef. DCT (abs, log)</div>
      <canvas id="sim04_dct_cvDCT" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Coef. Quantizzati</div>
      <canvas id="sim04_dct_cvQuant" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Blocco Ricostruito</div>
      <canvas id="sim04_dct_cvRec" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
  </div>

  <!-- Controles -->
  <div class="sim04_dct_panel">
    <div class="sim04_dct_slider_container">
      <label>Qualità JPEG:</label>
      <input type="range" id="sim04_dct_slider" min="1" max="100" value="50">
      <span id="sim04_dct_slVal" style="font-size:12px; font-family:monospace; font-weight:700; min-width:30px; color:#26241d;">50</span>
    </div>
    <div style="display:flex; gap:6px; flex-wrap:wrap;">
      <button data-q="10" style="flex:1;">Q=10</button>
      <button data-q="25" style="flex:1;">Q=25</button>
      <button data-q="50" style="flex:1;">Q=50</button>
      <button data-q="75" style="flex:1;">Q=75</button>
      <button data-q="95" style="flex:1;">Q=95</button>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04DCT(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const dct_block = [
      [52,55,61,66,70,61,64,73],
      [63,59,55,90,109,85,69,72],
      [62,59,68,113,144,104,66,73],
      [63,58,71,122,154,106,70,69],
      [67,61,68,104,126,88,68,70],
      [79,65,60,70,77,68,58,75],
      [85,71,64,59,55,61,65,83],
      [87,79,69,68,65,76,78,94]
    ];

    const Q_luma = [
      [16,11,10,16,24,40,51,61],[12,12,14,19,26,58,60,55],
      [14,13,16,24,40,57,69,56],[14,17,22,29,51,87,80,62],
      [18,22,37,56,68,109,103,77],[24,35,55,64,81,104,113,92],
      [49,64,78,87,103,121,120,101],[72,92,95,98,112,100,103,99]
    ];

    function dct1d(x) {
      const N = x.length, c = new Array(N).fill(0);
      for (let k = 0; k < N; k++) {
        let sum = 0;
        for (let n = 0; n < N; n++) sum += x[n] * Math.cos(Math.PI*(2*n+1)*k/(2*N));
        const alpha = k===0 ? Math.sqrt(1/N) : Math.sqrt(2/N);
        c[k] = alpha * sum;
      }
      return c;
    }

    function idct1d(c) {
      const N = c.length, x = new Array(N).fill(0);
      for (let n = 0; n < N; n++) {
        let sum = 0;
        for (let k = 0; k < N; k++) {
          const alpha = k===0 ? Math.sqrt(1/N) : Math.sqrt(2/N);
          sum += alpha * c[k] * Math.cos(Math.PI*(2*n+1)*k/(2*N));
        }
        x[n] = sum;
      }
      return x;
    }

    function dct2d(blk) {
      const N=8, rows=blk.map(r=>dct1d(r));
      const cols=[];
      for(let j=0;j<N;j++){const col=rows.map(r=>r[j]);cols.push(dct1d(col));}
      const out=Array.from({length:N},()=>new Array(N));
      for(let i=0;i<N;i++) for(let j=0;j<N;j++) out[i][j]=cols[j][i];
      return out;
    }

    function idct2d(C) {
      const N=8, cols=[];
      for(let j=0;j<N;j++){const col=C.map(r=>r[j]);cols.push(idct1d(col));}
      const rows=Array.from({length:N},()=>new Array(N));
      for(let i=0;i<N;i++) for(let j=0;j<N;j++) rows[i][j]=cols[j][i];
      return rows.map(r=>idct1d(r));
    }

    function getQ(quality) {
      const s = quality<50 ? 5000/quality : 200-2*quality;
      return Q_luma.map(row=>row.map(v=>Math.max(1,Math.min(255,Math.round(v*s/100)))));
    }

    function drawPixels(canvas, data, minV, maxV) {
      const ctx = canvas.getContext('2d');
      const sz = canvas.width/8;
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        const v = (data[i][j]-minV)/(maxV-minV);
        const g = Math.round(v*255);
        ctx.fillStyle='rgb('+g+','+g+','+g+')';
        ctx.fillRect(j*sz,i*sz,sz,sz);
        ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=0.5;
        ctx.strokeRect(j*sz,i*sz,sz,sz);
        ctx.fillStyle=g>128?'#26241d':'#fafaf7';
        ctx.font='bold ' + Math.round(sz*0.28) + 'px monospace';
        ctx.textAlign='center'; ctx.textBaseline='middle';
        const val = data[i][j];
        ctx.fillText(Math.abs(val)>999?'…':String(Math.round(val)), j*sz+sz/2, i*sz+sz/2);
      }
    }

    function drawHeatmap(canvas, data) {
      const ctx = canvas.getContext('2d');
      const sz = canvas.width/8;
      const flat=data.flat(); const mn=Math.min(...flat), mx=Math.max(...flat);
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        const v=(data[i][j]-mn)/(mx-mn||1);
        const r=Math.round(v*220+35), gb=Math.round((1-v)*180+30);
        ctx.fillStyle='rgb('+r+','+gb+','+gb+')';
        ctx.fillRect(j*sz,i*sz,sz,sz);
        ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=0.5;
        ctx.strokeRect(j*sz,i*sz,sz,sz);
        ctx.fillStyle='#ffffff'; ctx.font='bold ' + Math.round(sz*0.22) + 'px monospace';
        ctx.textAlign='center'; ctx.textBaseline='middle';
        const val = data[i][j];
        ctx.fillText(Math.abs(val)>999?'…':String(Math.round(val)), j*sz+sz/2, i*sz+sz/2);
      }
    }

    function update(quality) {
      const Q = getQ(quality);
      const centered = dct_block.map(r=>r.map(v=>v-128));
      const C = dct2d(centered);
      const Cq = C.map((r,i)=>r.map((v,j)=>Math.round(v/Q[i][j])));
      const Cdq = Cq.map((r,i)=>r.map((v,j)=>v*Q[i][j]));
      const rec = idct2d(Cdq).map(r=>r.map(v=>Math.max(0,Math.min(255,Math.round(v+128)))));

      const Clog = C.map(r=>r.map(v=>Math.log1p(Math.abs(v))*(v>=0?1:-1)));
      const Cqlog = Cq.map(r=>r.map(v=>v));

      let nz=0, mse=0;
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        if(Cq[i][j]!==0) nz++;
        mse+=(dct_block[i][j]-rec[i][j])**2;
      }
      mse/=64;

      drawPixels(root.querySelector('#sim04_dct_cvOrig'), dct_block, 0, 255);
      drawHeatmap(root.querySelector('#sim04_dct_cvDCT'), Clog);
      drawHeatmap(root.querySelector('#sim04_dct_cvQuant'), Cqlog);
      drawPixels(root.querySelector('#sim04_dct_cvRec'), rec, 0, 255);

      root.querySelector('#sim04_dct_qual').textContent = quality;
      root.querySelector('#sim04_dct_nonzero').textContent = nz;
      root.querySelector('#sim04_dct_zeros').textContent = (64-nz);
      root.querySelector('#sim04_dct_mse').textContent = mse.toFixed(1);
    }

    window.dct_setQ = function(q){
      root.querySelector('#sim04_dct_slider').value = q;
      root.querySelector('#sim04_dct_slVal').textContent = q;
      update(q);
    };

    root.querySelector('#sim04_dct_slider').addEventListener('input', function(){
      root.querySelector('#sim04_dct_slVal').textContent = this.value;
      update(+this.value);
    });

    root.querySelectorAll('[data-q]').forEach(btn => {
      btn.addEventListener('click', function() {
        dct_setQ(parseInt(this.getAttribute('data-q'), 10));
      });
    });

    update(50);
  }

  function tryInitSim04DCT(){
    var root = document.getElementById('sim-05-dct');
    if (root) initSim04DCT(root); else setTimeout(tryInitSim04DCT, 200);
  }
  tryInitSim04DCT();
})();
</script>
""")

**Figura 5.26:** Simulatore interattivo di compressione DCT-JPEG: regola il fattore di qualità e visualizza in tempo reale i coefficienti azzerati, il blocco ricostruito e l


<figure id="fig-05-sim-05-dct">
  <img src="imagens/fig-05-sim-05-dct.png" alt=" Simulatore interattivo di compressione DCT-JPEG: regola il fattore di qualità e visualizza in tempo reale i coefficienti azzerati, il blocco ricostruito e l'errore di quantizzazione. " style="max-width:80%" />
  <figcaption><strong>Figura 5.26:</strong>  Simulatore interattivo di compressione DCT-JPEG: regola il fattore di qualità e visualizza in tempo reale i coefficienti azzerati, il blocco ricostruito e l'errore di quantizzazione. </figcaption>
</figure>

## 5.10 Confronto dei Formati Immagine

La scelta di un formato di memorizzazione digitale incide direttamente sul compromesso tra qualità visiva, dimensione del file e costo computazionale della decodifica. I tre formati di maggiore rilevanza per architetture *web* e sistemi di calcolo visivo sono JPEG, PNG e WebP.

### 5.10.1 Caratteristiche dei Formati

La [Tabela 5.8](#tbl-05-formatos) sintetizza le proprietà strutturali dei principali formati di immagine rasterizzati.

<a id="tbl-05-formatos"></a>

**Tabela 5.8:** Confronto strutturale tra i principali formati di immagine rasterizzati.

| Caratteristica | JPEG | PNG | WebP |
|:---|:---:|:---:|:---:|
| **Compressione** | Con perdita | Senza perdita | Con e senza perdita. |
| **Trasparenza (canale alfa)** | No | Sì | Sì. |
| **Supporto animazioni** | No | Limitato (APNG) | Sì. |
| **Algoritmo di base** | DCT + Huffman | DEFLATE (LZ77 + Huffman) | VP8 / VP8L. |
| **Ideale per** | Fotografia | Grafica, testo e icone | Uso universale in ambiente Web. |
| **Inadeguato per** | Testo e bordi netti | Immagini fotografiche complesse | Compatibilità legacy. |


### 5.10.2 Metriche di Valutazione della Qualità

Due metriche oggettive sono ampiamente adottate per quantificare la distorsione introdotta dai processi di compressione:

**Picco del Rapporto Segnale-Rumore (PSNR, *Peak Signal-to-Noise Ratio*):**
<a id="eq-05-psnr"></a>
$$
\text{PSNR} = 10\,\log_{10}\!\left(\frac{L^2}{\text{MSE}}\right) \quad [\text{dB}] \tag{5.10}
$$


dove $L = 255$ per immagini quantizzate a 8 bit e $\text{MSE}$ rappresenta l'**Errore Quadratico Medio** (*Mean Squared Error*). Valori di PSNR superiori a 40 dB indicano fedeltà eccellente; tra 30 dB e 40 dB rappresentano buona qualità; valori inferiori a 30 dB corrispondono a degradazioni visive facilmente percepibili.

**Indice di Similarità Strutturale (SSIM, *Structural Similarity Index*):**
<a id="eq-05-ssim"></a>
$$
\text{SSIM}(f,g) = \frac{(2\mu_f\mu_g + c_1)(2\sigma_{fg} + c_2)}{(\mu_f^2+\mu_g^2+c_1)(\sigma_f^2+\sigma_g^2+c_2)} \tag{5.11}
$$


Lo SSIM valuta finestre locali dell'immagine basandosi su tre componenti complementari: **luminanza** ($\mu_f, \mu_g$), **contrasto** ($\sigma_f, \sigma_g$) e **struttura** ($\sigma_{fg}$), ponderate da costanti di stabilità $c_1$ e $c_2$. L'indice varia nell'intervallo $[-1, 1]$, dove l'unità rappresenta l'identità perfetta. A differenza del PSNR, lo SSIM considera l'organizzazione spaziale degli errori, allineandosi alla percezione del sistema visivo umano (SVH).

> ### 📝 PSNR vs SSIM: Applicazione di Metriche Percettive
>
> Il PSNR possiede una formulazione matematica semplice e un basso costo computazionale; tuttavia tende a sovrastimare la qualità in immagini con distorsioni localizzate o a sottostimarla in variazioni globali di luminosità tollerate dall'osservatore. Lo SSIM modella con maggiore fedeltà la percezione biologica, ma richiede un maggiore sforzo di elaborazione. Per analisi rigorose dei codec, si raccomanda di riportare entrambe le metriche statistiche in carattere complementare.

### 5.10.3 Ispezione Visiva: Natura degli Artefatti di Compressione

La natura matematica del codificatore determina il tipo di degrado introdotto a bitrate ridotti. Come illustrato nella [Figura 5.27](#fig-05-zoom-artefatos), la compressione aggressiva tramite DCT nello standard JPEG segmenta l'immagine in griglie rigide, generando gli **artefatti a blocchi** (*blocking artifacts*). Al contrario, gli algoritmi basati su codifica predittiva o rappresentazioni sottoposte a trasformate spaziali avanzate (come WebP e JPEG 2000) eliminano le discontinuità di blocco, ma introducono perdita di texture fine e sfocature caratteristiche attorno ai bordi ad alto contrasto.

In [62]:
%%writefile tmp/fig_05_zoom_artefatos.cpp
#define MM_OUT "tmp/fig_05_zoom_artefatos.png"
//| label: fig-05-zoom-artefatos
//| fig-cap: "Análise comparativa de artefatos de compressão sob fator de qualidade reduzido ($Q=10$). À esquerda, observa-se o artefato de bloco característico da discretização por DCT no JPEG. À direita, evidencia-se o efeito de atenuação e suavização de bordas intrínseco ao padrão WebP."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <filesystem>
#include "morph.hpp"

// Função de zoom com interpolação NEAREST
cv::Mat zoom(const cv::Mat& img) {
    cv::Mat roi = img(cv::Rect(150, 120, 80, 80));  // img[120:200, 150:230]
    cv::Mat resized;
    cv::resize(roi, resized, cv::Size(320, 320), 0, 0, cv::INTER_NEAREST);
    return resized;
}

int main() {
    // Cria diretório temporário se não existir
    std::filesystem::create_directories("tmp");

    // Lê e redimensiona a imagem original
    cv::Mat img_src_full = cv::imread("imagens/cameraman.png", cv::IMREAD_GRAYSCALE);
    cv::Mat img_src_resized;
    cv::resize(img_src_full, img_src_resized, cv::Size(256, 256));
    mm::Image img_src = img_src_resized;  // Converte para mm::Image

    // Salva versões comprimidas
    cv::Mat img_src_mat = img_src;  // Converte de volta para cv::Mat
    cv::imwrite("tmp/zoom_q10.jpg", img_src_mat, {cv::IMWRITE_JPEG_QUALITY, 10});
    cv::imwrite("tmp/zoom_q10.webp", img_src_mat, {cv::IMWRITE_WEBP_QUALITY, 10});

    // Aplica zoom em cada versão
    cv::Mat z_orig = zoom(img_src_mat);
    cv::Mat z_jpeg = zoom(cv::imread("tmp/zoom_q10.jpg", cv::IMREAD_GRAYSCALE));
    cv::Mat z_webp = zoom(cv::imread("tmp/zoom_q10.webp", cv::IMREAD_GRAYSCALE));

    // Mostra os resultados
    mm::show(
        {z_orig, z_jpeg, z_webp},
        MM_OUT,
        {"Zoom Original", "JPEG Q=10 (Artefato de Bloco)", "WebP Q=10 (Suavizacao)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(z_orig, "tmp/fig_05_zoom_artefatos_0.png");
mm::write(z_jpeg, "tmp/fig_05_zoom_artefatos_1.png");
mm::write(z_webp, "tmp/fig_05_zoom_artefatos_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_05_zoom_artefatos.cpp


In [63]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_zoom_artefatos.cpp -o tmp/fig_05_zoom_artefatos -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_zoom_artefatos \
  && test -f "tmp/fig_05_zoom_artefatos.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_zoom_artefatos.png"

[1] Zoom Original
[2] JPEG Q=10 (Artefato de Bloco)
[3] WebP Q=10 (Suavizacao)


In [64]:
try:
    mm.show(
        [
            mm.read("tmp/fig_05_zoom_artefatos_0.png"),
            mm.read("tmp/fig_05_zoom_artefatos_1.png"),
            mm.read("tmp/fig_05_zoom_artefatos_2.png"),
        ],
        titles=[
            'Zoom Original',
            'JPEG Q=10 (Artefato de Bloco)',
            'WebP Q=10 (Suavizacao)',
        ],
        cols=3,
        figsize=(14, 5),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_zoom_artefatos_0.png (ver a versao Python)")

<Figure size 2100x750 with 3 Axes>

**Figura 5.27:** Análise comparativa de artefatos de compressão sob fator de qualidade reduzido ($Q=10$). À esquerda, observa-se o artefato de bloco característico da discretização por DCT no JPEG. À direita, evidencia-se o efeito de atenuação e suavização de bordas intrínseco ao padrão WebP.


### 5.10.4 Valutazione Quantitativa e Spaziale della Compressione

La validazione degli algoritmi di compressione con perdita richiede un'analisi che correli il costo di archiviazione alla fedeltà del segnale ricostruito. Tale valutazione viene condotta in modo complementare attraverso curve di prestazione globale e mediante la mappatura locale delle distorsioni indotte dai codificatori.

#### 5.10.4.1 Curve di Rateo-Distorsione

La [Figura 5.28](#fig-05-formatos-comparacao) presenta la valutazione empirica del *pipeline* JPEG e WebP tramite **curve di rateo-distorsione**, che monitorano il guadagno di compressione (dimensione del file in KB) in funzione del PSNR. Il formato PNG funge da linea di base ideale ($\text{PSNR} = \infty$), poiché la sua natura *lossless* impedisce qualsiasi degradazione, sebbene richieda un volume di dati notevolmente maggiore.

L'analisi delle curve dimostra la superiorità e l'efficienza dello standard WebP rispetto al JPEG tradizionale: per raggiungere lo stesso livello di fedeltà matematica (come la fascia di qualità eccellente, dove $\text{PSNR} > 40\text{ dB}$), il codificatore WebP genera file significativamente più piccoli. Questo comportamento riflette l'impatto pratico dell'evoluzione degli algoritmi nell'ottimizzazione dei sistemi di trasmissione e archiviazione digitale.

In [65]:
import cv2
import os, cv2

src = mm.gray(cv2.resize(mm.read("imagens/cameraman.png"), (256, 256)))

jpeg_kb, jpeg_psnr = [], []
for q in [10, 20, 30, 40, 50, 60, 70, 80, 90, 95]:
    p = f"tmp/fmt_q{q}.jpg"
    cv2.imwrite(p, src, [cv2.IMWRITE_JPEG_QUALITY, q])
    rec = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    jpeg_kb.append(os.path.getsize(p) / 1024.0)
    jpeg_psnr.append(mm.psnr(src, rec))

webp_kb, webp_psnr = [], []
for q in [30, 50, 70, 85, 95]:
    p = f"tmp/fmt_w{q}.webp"
    cv2.imwrite(p, src, [cv2.IMWRITE_WEBP_QUALITY, q])
    rec = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    webp_kb.append(os.path.getsize(p) / 1024.0)
    webp_psnr.append(mm.psnr(src, rec))

p_png = "tmp/fmt.png"
cv2.imwrite(p_png, src, [cv2.IMWRITE_PNG_COMPRESSION, 9])
png_kb = os.path.getsize(p_png) / 1024.0

chart = mm.lineChart(
    [jpeg_kb, webp_kb], [jpeg_psnr, webp_psnr],
    labels=["JPEG", "WebP"],
    title=f"Curva Taxa-Distorcao (PNG sem perda: {png_kb:.1f} KB)",
    xlabel="Tamanho do arquivo (KB)", ylabel="PSNR (dB)"
)
mm.show([chart], titles=["JPEG vs WebP vs PNG"], cols=1)


<Figure size 750x750 with 1 Axes>

**Figura 5.28:** Curva distorsione-tasso: PSNR vs dimensione del file per JPEG, WebP e PNG applicata all


> ### 📝 5.11 Dimensione originale dell'immagine
>
> L'immagine *Cameraman* ($256 \times 256$ pixel in scala di grigi) occupa **64 KB** in formato grezzo (senza compressione). Come riferimento, il PNG *lossless* comprime questo volume a **36,2 KB** — evidenziando che la compressione senza perdita riduce già significativamente lo spazio di archiviazione per immagini con regioni omogenee. In contrapposizione, i formati con perdita (JPEG e WebP) raggiungono dimensioni ancora minori: il JPEG con qualità 95 occupa 22,3 KB (PSNR ≈ 45 dB), mentre il WebP con qualità 90 raggiunge 12,5 KB con PSNR equivalente, dimostrando la sua superiorità in efficienza di compressione.

#### 5.11.0.1 Mappatura Spaziale degli Errori e Correlazione Percettiva

Sebbene il PSNR offra un indicatore numerico rapido, le metriche globali non riescono a distinguere come la perdita di informazione si distribuisca geometricamente sull'immagine. La [Figura 5.29](#fig-05-ssim-artefatos) risolve questa limitazione associando le ricostruzioni a diverse qualità ai rispettivi mappe di errore assoluto e all'SSIM.

Le mappe residue — ottenute dalla differenza assoluta normalizzata tra l'immagine originale e quella compressa — rivelano la firma spaziale intrinseca di ciascuna architettura di codifica:

* **Ad alte qualità ($Q=95$ a $Q=75$):** Le distorsioni si concentrano prevalentemente attorno a transizioni brusche di intensità (bordi), a causa del ripiegamento spettrale derivante dallo scarto delle alte frequenze. L'indice SSIM rimane prossimo all'unità, attestando l'integrità delle strutture originali.
* **A qualità aggressive ($Q=50$ a $Q=25$):** L'errore assume una struttura a maglia ortogonale regolarizzata. Questo pattern geometrico evidenzia l'emergere degli **artefatti a blocchi** (*blocking artifacts*), indicando che la quantizzazione severa ha corrotto la correlazione spaziale tra blocchi adiacenti di $8 \times 8$ pixel.

L'SSIM cattura questa degradazione morfologica in modo molto più sensibile rispetto al PSNR, penalizzando il punteggio finale man mano che l'organizzazione strutturale e le trame fini — alle quali il sistema visivo umano è altamente reattivo — vengono eliminate dal codificatore.

In [66]:
%%writefile tmp/fig_05_ssim_artefatos.cpp
#define MM_OUT "tmp/fig_05_ssim_artefatos.png"
//| label: fig-05-ssim-artefatos
//| fig-cap: "Análise espacial de degradação: imagens reconstruídas e respectivos mapas de erro absoluto normalizados para diferentes fatores de qualidade JPEG."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include <cstdio>
#include "morph.hpp"

// Implementazione SSIM (Wang et al.) con finestra gaussiana
static double compute_ssim(const cv::Mat& img1, const cv::Mat& img2) {
    const double C1 = 6.5025, C2 = 58.5225;
    cv::Mat I1, I2;
    img1.convertTo(I1, CV_64F);
    img2.convertTo(I2, CV_64F);

    cv::Mat kernel = cv::getGaussianKernel(11, 1.5);
    cv::Mat window = kernel * kernel.t();

    cv::Mat mu1, mu2, mu1_sq, mu2_sq, mu1_mu2;
    cv::filter2D(I1, mu1, -1, window);
    cv::filter2D(I2, mu2, -1, window);
    cv::pow(mu1, 2, mu1_sq);
    cv::pow(mu2, 2, mu2_sq);
    mu1_mu2 = mu1.mul(mu2);

    cv::Mat sigma1_sq, sigma2_sq, sigma12;
    cv::filter2D(I1.mul(I1), sigma1_sq, -1, window);
    sigma1_sq -= mu1_sq;
    cv::filter2D(I2.mul(I2), sigma2_sq, -1, window);
    sigma2_sq -= mu2_sq;
    cv::filter2D(I1.mul(I2), sigma12, -1, window);
    sigma12 -= mu1_mu2;

    cv::Mat ssim_map;
    cv::Mat cs_map;
    cv::Mat t1 = 2 * mu1_mu2 + C1;
    cv::Mat t2 = 2 * sigma12 + C2;
    cv::Mat t3 = t1.mul(t2);
    cv::divide(t3, (mu1_sq + mu2_sq + C1).mul(sigma1_sq + sigma2_sq + C2), ssim_map);

    cv::Scalar mssim = cv::mean(ssim_map);
    return mssim[0];
}

int main() {
    // Cameraman (asset del capitolo), 256x256 — stessa immagine del percorso py.
    cv::Mat img_tmp = cv::imread("imagens/cameraman.png", cv::IMREAD_GRAYSCALE);
    cv::Mat img_resized;
    cv::resize(img_tmp, img_resized, cv::Size(256, 256));
    mm::Image img_gray = mm::gray(mm::Image(img_resized));

    std::vector<mm::Image> imgs_ssim;
    std::vector<std::string> titles_ssim;
    imgs_ssim.push_back(img_gray);
    titles_ssim.push_back("Originale");

    int qs[] = {25, 50, 75, 95};
    for (int i = 0; i < 4; i++) {
        int q = qs[i];
        mm::Image rec = mm::jpegCompress(img_gray, q);  // DCT 8x8 -> quant -> IDCT

        cv::Mat img_gray_cv = img_gray;
        cv::Mat rec_cv = rec;

        double psnr_v = mm::psnr(img_gray, rec);

        // Calcola SSIM
        double ssim_v = compute_ssim(img_gray_cv, rec_cv);

        // Mappa di errore normalizzata
        cv::Mat img_float, rec_float, diff;
        img_gray_cv.convertTo(img_float, CV_64F);
        rec_cv.convertTo(rec_float, CV_64F);
        diff = cv::abs(img_float - rec_float);

        cv::Mat diff_vis_m;
        cv::normalize(diff, diff_vis_m, 0, 255, cv::NORM_MINMAX, CV_8U);
        mm::Image diff_vis(diff_vis_m);

        imgs_ssim.push_back(rec);
        imgs_ssim.push_back(diff_vis);

        char title_buf[256];
        std::snprintf(title_buf, sizeof(title_buf), "Q=%d (PSNR=%.1fdB | SSIM=%.3f)", q, psnr_v, ssim_v);
        titles_ssim.push_back(std::string(title_buf));

        char err_buf[256];
        std::snprintf(err_buf, sizeof(err_buf), "Mappa di errore (Q=%d) - bordi e blocco", q);
        titles_ssim.push_back(std::string(err_buf));
    }

    mm::show(imgs_ssim, MM_OUT, titles_ssim, 3);

    return 0;
}

Overwriting tmp/fig_05_ssim_artefatos.cpp


In [67]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_ssim_artefatos.cpp -o tmp/fig_05_ssim_artefatos -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_ssim_artefatos \
  && test -f "tmp/fig_05_ssim_artefatos.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_ssim_artefatos.png"

[1] Originale
[2] Q=25 (PSNR=30.7dB | SSIM=0.860)
[3] Mappa di errore (Q=25) - bordi e blocco
[4] Q=50 (PSNR=32.8dB | SSIM=0.905)
[5] Mappa di errore (Q=50) - bordi e blocco
[6] Q=75 (PSNR=35.2dB | SSIM=0.938)
[7] Mappa di errore (Q=75) - bordi e blocco
[8] Q=95 (PSNR=44.8dB | SSIM=0.990)
[9] Mappa di errore (Q=95) - bordi e blocco


In [68]:
try:
    mm.show(mm.read("tmp/fig_05_ssim_artefatos.png"), figsize=(14, 14))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_05_ssim_artefatos.png (ver a versao Python)")

<Figure size 2100x2100 with 1 Axes>

**Figura 5.29:** Análise espacial de degradação: imagens reconstruídas e respectivos mapas de erro absoluto normalizados para diferentes fatores de qualidade JPEG.


> ### 📝 5.12 Interpretando le mappe di errore
>
> Le mappe di errore presentate sono state **normalizzate singolarmente** (`cv2.NORM_MINMAX`) per massimizzare il contrasto visivo e rivelare la struttura spaziale delle distorsioni. Ciò significa che:
>
> - In **Q=95**, l'errore assoluto è dell'ordine di **0.5–1.5 livelli di grigio** (impercettibile visivamente), ma la normalizzazione lo amplifica in bianco e nero per evidenziarne la localizzazione su bordi e transizioni.
> - In **Q=25**, l'errore assoluto è **10–20 volte maggiore** (5–15 livelli di grigio), ma la normalizzazione lo porta anch'esso allo stesso intervallo [0, 255].
>
> Pertanto, **l'intensità del bianco nelle mappe NON è confrontabile tra diverse qualità** — le mappe servono solo a rivelare la **firma spaziale** dell'errore (bordi vs blocchi), non la sua ampiezza. L'ampiezza corretta è fornita dai valori di PSNR e SSIM, che mostrano chiaramente che Q=95 ha un errore molto minore rispetto a Q=25.

### Sintesi — Compressione JPEG

Il processo di compressione nello standard JPEG si basa sull'applicazione combinata di trasformazioni spaziali, percettive e statistiche per ridurre le ridondanze di un'immagine. La [Tabela 5.9](#tbl-05-sintese-jpeg) riassume il ruolo di ciascuna fase nel *pipeline* e il rispettivo impatto sulla riduzione dei dati.

<a id="tbl-05-sintese-jpeg"></a>

**Tabela 5.9:** Sintesi delle fasi del *pipeline* di compressione JPEG e dei rispettivi impatti.

| Fase | Operazione Analitica | Meccanismo di Guadagno / Compressione |
|:---|:---|:---|
| **Conversione $YC_bC_r$** | Isolamento dei canali di luminanza e crominanza. | Modella la percezione del SVH, consentendo di trattare colore e luminosità in modo indipendente. |
| **Sottocampionamento 4:2:0** | Riduzione della risoluzione spaziale dei canali di colore ($C_b$ e $C_r$). | Elimina circa il 50% dei dati grezzi con un impatto visivo minimo. |
| **DCT $8 \times 8$** | Mappatura dal dominio spaziale al dominio delle frequenze spaziali. | Compattazione dell'energia, concentrando l'informazione vitale nei primi coefficienti. |
| **Quantizzazione Lineare** | Divisione intera dei coefficienti per una matrice di ponderazione $Q(u,v)$. | Principale fonte di compressione con perdita; elimina le alte frequenze impercettibili. |
| **Codifica Entropica** | Applicazione di algoritmi RLE e codifica di Huffman. | Compressione statistica senza perdita, ottimizzata dalle lunghe sequenze di coefficienti nulli. |


#### Artefatti di Degradazione Caratteristici

L'applicazione di tassi di compressione eccessivamente aggressivi (fattori di qualità ridotti) introduce distorsioni prevedibili nell'immagine ricostruita, derivanti dalle limitazioni matematiche del modello:

* **Artefatti a blocchi (*blocking artifacts*):** Discontinuità geometriche visibili ai confini dei blocchi di $8 \times 8$ pixel, causate dalla perdita di correlazione spaziale dopo la quantizzazione severa delle componenti AC.
* **Effetto di alone (*ringing*):** Oscillazioni fantasma o distorsioni "a fumo" attorno ai bordi netti e ad alto contrasto, provocate dall'eliminazione brusca delle armoniche ad alta frequenza necessarie per ricostruire funzioni a gradino.
* **Perdita di texture fine:** Attenuazione dei dettagli ad alta frequenza e a basso contrasto (come prati, tessuti o porosità), rendendo le regioni originariamente strutturate eccessivamente lisce o omogenee.

## 5.13 Applicazione Pratica: Rimozione del Rumore mediante Filtraggio Ibrido

Integrando le tecniche consolidate nel corso di questo capitolo, si presenta un *pipeline* completo di **restauro delle immagini** che combina l'analisi spettrale nel dominio della frequenza con il filtraggio adattivo nel dominio spaziale. L'obiettivo è attenuare un rumore misto (composto da degradazione gaussiana e interferenza periodica) preservando al massimo i dettagli strutturali dell'immagine originale.

$$
\text{Immagine Rumorosa} \xrightarrow{\text{FFT2}} \xrightarrow{\text{Filtro Notch Gaussiano}} \xrightarrow{\text{IFFT2}} \xrightarrow{\text{Filtro Bilaterale}} \text{Immagine Restaurata}
$$

> ### 📝 Valutazione Complementare: PSNR vs. SSIM
>
> La coppia di metriche statistiche PSNR e SSIM fornisce una valutazione qualitativa e morfologica complementare del processo di restauro:
>
> * **PSNR:** Penalizza uniformemente lo scarto quadratico medio pixel per pixel.
> * **SSIM:** Valuta la preservazione di strutture locali percettivamente rilevanti (luminanza, contrasto e contorni).
>
> Nella pratica, esiste un compromesso analitico (*trade-off*) tra **riduzione del rumore** e **preservazione dei dettagli**: filtri spaziali eccessivamente aggressivi attenuano bene il rumore ad alta frequenza, ma degradano trame fini e smussano bordi netti — il che **riduce simultaneamente** sia il PSNR che lo SSIM rispetto all'immagine originale. La sfida nella progettazione dei filtri è trovare il punto di equilibrio che massimizzi entrambe le metriche, garantendo un restauro fedele e visivamente gradevole.

### 5.13.1 Analisi delle Prestazioni e Conclusione del Capitolo

I risultati numerici e visivi generati dalla [Figura 5.30](#fig-05-pipeline-denoising) dimostrano la rilevanza pratica di associare diversi domini di elaborazione. L'inserimento simultaneo di rumore periodico e stocastico corrompe le proprietà morfologiche del segnale, riducendo severamente gli indici di similarità e il rapporto segnale-rumore dell'immagine di riferimento.

L'isolamento e la soppressione dei picchi armonici nel dominio della frequenza tramite la maschera *notch* rimuovono le frange di interferenza sinusoidali sparse nello spazio bidimensionale. Come evidenziato nei dati stampati della [Figura 5.30](#fig-05-pipeline-denoising), questa filtrazione chirurgica promuove un salto immediato e sostanziale nella metrica PSNR. Tuttavia, il rumore gaussiano ad alta frequenza rimane attivo in modo omogeneo nello spettro, richiedendo un approccio complementare.

Il restauro finale è consolidato nel dominio spaziale con l'introduzione del filtro bilaterale. Diversamente dagli operatori passa-basso convenzionali (come quello gaussiano o di media), che smusserebbero indiscriminatamente rumore e contorni strutturali, la filtrazione bilaterale calcola pesi ponderati in base alla prossimità geometrica e alla differenza di intensità radiometrica. Questo comportamento adattivo attenua le fluttuazioni stocastiche residue nelle regioni di transizione graduale e preserva la nitidezza dei bordi spaziali.

La convergenza di entrambi gli approcci risulta in un **miglioramento sostanziale e simultaneo** di PSNR e SSIM rispetto all'immagine rumorosa — sebbene i valori finali rimangano inferiori a quelli dell'immagine originale (PSNR = $\infty$, SSIM = 1,0), a causa della perdita inevitabile di informazioni spettrali e testurali durante i processi di filtrazione. L'attenuazione graduale (gaussiana) dei picchi nello spettro evita artefatti di *ringing*, mentre il filtro bilaterale elimina il rumore stocastico residuo senza compromettere la nitidezza dei bordi. I risultati confermano l'efficacia e la complementarità pratica degli strumenti di analisi di frequenza presentati in questo capitolo, dimostrando che la filtrazione ibrida (frequenza + spaziale) è superiore a qualsiasi approccio isolato per il restauro di immagini degradate da rumore misto.

In [69]:
import numpy as np, cv2, os  # [pdi] passthrough: garante imports desta trilha
import numpy as np, cv2

# Cameraman (asset del capitolo), 256x256 — stessa immagine della traccia py.
img_gray = mm.gray(cv2.resize(mm.read("imagens/cameraman.png"), (256, 256)))
h_img, w_img = img_gray.shape

# ── 1. Rumore misto: gaussiano + periodico ───────────────────────────────────
np.random.seed(42)
X2, Y2 = np.meshgrid(np.arange(w_img), np.arange(h_img))
u0, v0       = 15, 10
ruido_gauss  = np.random.normal(0, 15, img_gray.shape)
ruido_period = 30 * np.sin(2 * np.pi * (u0 * X2 / w_img + v0 * Y2 / h_img))
img_noisy    = np.clip(img_gray.astype(float) + ruido_gauss + ruido_period, 0, 255).astype(np.uint8)

# ── 2. Spettro (log-magnitudine) dell'immagine rumorosa ─────────────────────────
mag_n = mm.spectrumMag(img_noisy)

# ── 3. Maschera notch gaussiana sui 4 picchi periodici ──────────────────────
def suprimir_pico_gaussiano(mask, cy, cx, sigma=3.0):
    yy, xx = np.meshgrid(np.arange(mask.shape[0]), np.arange(mask.shape[1]), indexing="ij")
    notch = np.exp(-((yy - cy)**2 + (xx - cx)**2) / (2 * sigma**2))
    return mask * (1.0 - notch)

cy0, cx0 = h_img // 2, w_img // 2
mascara_notch = np.ones((h_img, w_img), dtype=np.float64)
for dy, dx in [(v0, u0), (-v0, -u0), (v0, -u0), (-v0, u0)]:
    mascara_notch = suprimir_pico_gaussiano(mascara_notch, cy0 + dy, cx0 + dx, 3.0)

# ── 4. Filtraggio: notch nel dominio della frequenza + bilaterale ───────────────
img_notch = mm.freqFilter(img_noisy, mascara_notch)
img_den   = cv2.bilateralFilter(img_notch, 7, 25, 7)
mascara_vis = (mascara_notch * 255).astype(np.uint8)

psnr_n  = mm.psnr(img_gray, img_noisy)
psnr_no = mm.psnr(img_gray, img_notch)
psnr_d  = mm.psnr(img_gray, img_den)
print(f"Rumorosa: PSNR={psnr_n:.2f} dB | Dopo notch: {psnr_no:.2f} dB | Notch+bilaterale: {psnr_d:.2f} dB")

mm.show(
    [img_gray, img_noisy, mag_n, mascara_vis, img_notch, img_den],
    titles=[
        "Originale",
        f"Rumorosa (PSNR={psnr_n:.1f} dB)",
        "Spettro (picchi visibili)",
        "Maschera notch (gaussiana)",
        f"Dopo notch (PSNR={psnr_no:.1f} dB)",
        f"Notch + bilaterale (PSNR={psnr_d:.1f} dB)",
    ],
    cols=6, figsize=(20, 4)
)


Rumorosa: PSNR=20.19 dB | Dopo notch: 22.40 dB | Notch+bilaterale: 23.61 dB


<Figure size 3000x600 with 6 Axes>

**Figura 5.30:** *Pipeline* completo di rimozione del rumore misto: (1) aggiunta di rumore gaussiano e periodico; (2) identificazione dei picchi di interferenza nello spettro delle frequenze; (3) applicazione di maschera *notch* con attenuazione gaussiana morbida; (4) post-elaborazione tramite filtro bilaterale per l


## 5.14 Riepilogo del Capitolo

La transizione dal **dominio spaziale** al **dominio delle frequenze** rivela la distribuzione spettrale dell'energia dell'immagine, stabilendo la base analitica per il filtraggio avanzato, il restauro e la compressione dei dati. L'articolazione strutturale di questi concetti è sintetizzata nella mappa concettuale della [Figura 5.31](#fig-05-mapa-conceitual).

<figure id="fig-05-mapa-conceitual" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-05-mapa-conceitual.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figura 5.31:</strong> Mappa concettuale delle trasformazioni e delle proprietà nel dominio delle frequenze.</figcaption>
</figure>

### Fondamenti Essenziali

* **DFT e Percezione Visiva:** Lo spettro scompone l'immagine in componenti armoniche. La **fase** mantiene l'intelligibilità geometrica della scena e la localizzazione dei contorni, mentre la **magnitudine** determina la distribuzione del contrasto e delle ampiezze globali.
* **Efficienza Algoritmica:** Il Teorema della Convoluzione rende possibile l'elaborazione di maschere su larga scala nel dominio della frequenza tramite FFT, riducendo la complessità computazionale asintotica da $O(N^2 K^2)$ nello spazio a $O(N^2 \log N)$.
* **Fenomeno di *Ringing*:** Tagli netti nello spettro (filtri ideali) generano oscillazioni spaziali indesiderate (fenomeno di Gibbs). L'attenuazione graduale mediante filtri di **Butterworth** o **Gaussiani** elimina queste discontinuità.
* **Analisi Multirisoluzione tramite *Wavelets*:** Superando il carattere puramente globale di Fourier, la DWT cattura simultaneamente frequenza e localizzazione spaziale, costituendo la base dello standard JPEG 2000 e supportando rappresentazioni gerarchiche analoghe alle estrazioni di caratteristiche nelle Reti Neurali Convoluzionali (CNN).
* **Compressione Percettiva (DCT):** La pipeline JPEG sfrutta i limiti di contrasto del sistema visivo umano alle alte frequenze spaziali. La DCT isola l'energia di blocchi $8 \times 8$, consentendo alla quantizzazione di scartare i coefficienti AC di dettagli fini senza un danno percettivo severo.


**Prossimi Passi:** Il **Capitolo 6** inaugura la Parte II dell'opera, applicando gli strumenti di elaborazione delle immagini alla risoluzione di problemi reali di ispezione industriale. Verranno esplorate tecniche di **segmentazione e analisi delle forme** per il rilevamento automatico di difetti nelle linee di produzione — dall'identificazione di difetti superficiali nei pezzi alla lettura di *QRCode* nelle prove, consolidando il ponte tra la teoria presentata nella Parte I e le esigenze pratiche della visione computazionale.

## 5.15 🤖 Uso del Gemini Notebook come Tutore Complementare

In questa edizione, si incoraggia l'uso della piattaforma **Gemini Notebook** come strumento complementare di apprendimento — **non come sostituto** della lettura attenta, della risoluzione degli esercizi o della sperimentazione pratica. Basato su architetture di intelligenza artificiale, il sistema utilizza esclusivamente il materiale didattico e i documenti forniti dall'autore come base di conoscenza, garantendo che le risposte generate siano concettualmente allineate al contenuto programmatico e all'approccio pedagogico adottato nel corso di quest'opera.

> ### ❗ Accesso al Tutore Intelligente
>
> [🚀 ACCEDI A Gemini Notebook: CAPITOLO 05](https://notebooklm.google.com/notebook/b8b6cd26-ef65-4a10-b7e7-e072d4870ddb)
>
> #### 🌐 Lingua e Linguaggio di Programmazione
>
> Il progetto di questo capitolo nel Gemini Notebook è stato realizzato esclusivamente con il testo in **portoghese** e gli esempi di codice in **Python**. Se stai studiando dall'edizione in inglese o francese, oppure seguendo il percorso in C++, le risposte del tutore potrebbero non corrispondere esattamente alla versione che stai leggendo.
>
> #### Linee Guida sul Contenuto Generato dall'Intelligenza Artificiale
>
> Sebbene gli strumenti di intelligenza artificiale costituiscano alleati efficienti nel processo di apprendimento e revisione, il contenuto generato è soggetto a incongruenze o imprecisioni tecniche. Pertanto, è indispensabile la consultazione sistematica di libri di testo, articoli scientifici e fonti accademiche indicizzate per una validazione rigorosa delle informazioni. Si raccomanda vivamente l'esecuzione e la modifica degli esempi pratici in Python forniti in questo capitolo come metodo primario di verifica sperimentale dei risultati.

## 5.16 Elenco di Esercizi

1. **(10%) Implementazione Diretta della DFT 2D:** Implementare analiticamente la Trasformata Discreta di Fourier 2D (DFT) senza l'ausilio di funzioni native di librerie (come `np.fft.fft2`), utilizzando strettamente la formulazione matematica definita in [Equação 5.1](#eq-05-dft) per una matrice di dimensioni $16 \times 16$. Eseguire la validazione numerica confrontando i coefficienti generati con i risultati della funzione `np.fft.fft2`, assicurandosi che la deviazione assoluta massima sia inferiore a $10^{-8}$. Misurare i tempi di esecuzione di entrambi i metodi e presentare una giustificazione teorica per la disparità osservata in termini di complessità asintotica.

2. **(15%) Soppressione del Rumore Periodico:** Aggiungere interferenze sinusoidali con frequenze spaziali $(u_0, v_0) \in \{(5,10), (20,5), (30,30)\}$ all'immagine di test del *Cameraman*. Per ogni scenario di degradazione, progettare una maschera di filtraggio *notch* specifica nel dominio della frequenza per isolare e attenuare i picchi armonici indesiderati. Valutare quantitativamente l'efficacia del processo di restauro mediante il calcolo delle metriche PSNR e SSIM. Discutere analiticamente il compromesso (*trade-off*) tra l'attenuazione del rumore sinusoidale e l'indesiderata attenuazione delle caratteristiche strutturali legittime dell'immagine.

3. **(15%) Analisi Comparativa degli Operatori Passa-Basso:** Condurre uno studio comparativo tra i filtri passa-basso Ideale, Gaussiano e Butterworth (con ordini armonici $n = 1, 2, 4$), parametrizzati con frequenze di taglio $D_0 = 20, 40, 60$ pixel. Per ogni combinazione strutturale, calcolare gli indici PSNR e SSIM dell'immagine risultante rispetto al segnale originale di riferimento. Organizzare i dati quantitativi in una tabella strutturata e tracciare i grafici unidimensionali delle funzioni di trasferimento corrispondenti lungo il profilo orizzontale $H(u, 0)$.

4. **(15%) Banco di Filtri Multirisoluzione di Haar:** Sviluppare uno script per eseguire manualmente la decomposizione *wavelet* discreta 2D di primo livello utilizzando la famiglia Haar. L'algoritmo deve calcolare i coefficienti dei filtri corrispondenti passa-basso ($h$) e passa-alto ($g$), applicandoli in modo separabile sulle righe e colonne della matrice, seguiti dall'operazione di decimazione (sottocampionamento spaziale per un fattore di 2). Validare numericamente l'accuratezza della propria implementazione confrontando le sottobande ottenute con l'output della funzione `pywt.dwt2(img, 'haar')`.

5. **(15%) Compressione Sparsa mediante Sogliatura Wavelet:** Applicare la tecnica di filtraggio per sogliatura netta (*hard thresholding*) sui coefficienti di dettaglio della decomposizione *wavelet*, adottando le soglie numeriche $T \in \{5, 10, 20, 40, 80\}$ per le famiglie Haar, Daubechies (`db4`) e Symlets (`sym4`). Dopo aver eseguito il processo di sintesi mediante la trasformata inversa (`pywt.waverec2`), calcolare i valori di PSNR e SSIM di ciascuna immagine ricostruita. Identificare e giustificare quale combinazione di famiglia *wavelet* e soglia $T$ massimizza la similarità strutturale.

6. **(15%) Costruzione di un Codificatore JPEG Semplificato:** Implementare la pipeline completa di compressione dei dati simulando lo standard JPEG. Il flusso deve comprendere: conversione spaziale $RGB \rightarrow YC_bC_r$, sottocampionamento cromatico nella proporzione 4:2:0, segmentazione della luminanza in blocchi disgiunti di $8 \times 8$ pixel, applicazione della DCT-II 2D ortogonale e quantizzazione lineare basata sulla matrice normalizzata di luminanza scalata per i fattori di qualità desiderati. Eseguire la decodifica inversa e confrontare quantitativamente le ricostruzioni con i file generati dalla funzione `cv2.imencode` per i fattori di qualità di 20, 50 e 80.

7. **(15%) Analisi Percettiva su Contenuti Eterogenei:** Sviluppare un'immagine sintetica composta da tre regioni distinte e di caratteristiche spettrali contrastanti: una texture fotografica complessa (che rappresenta alte frequenze stocastiche), un'area di testo vettorizzato con bordi netti (che rappresenta transizioni a gradino pure) e un gradiente lineare continuo (che rappresenta basse frequenze omogenee). Sottoporre questa immagine mista ai processi di compressione nei formati JPEG, PNG e WebP. Valutare e interpretare i risultati correlando la dimensione finale del file su disco alle metriche PSNR e SSIM ottenute, giustificando quale formato mostra le prestazioni migliori per segnali di natura eterogenea e perché tale vantaggio si verifica in termini di compattazione dell'energia e preservazione percettiva.

## Riferimenti del Capitolo

La base teorica e lo sviluppo analitico dei concetti trattati in questo capitolo si fondano sulle seguenti opere di riferimento:

* **Gonzalez (2018)** — Formulazioni classiche delle Trasformate Discrete di Fourier 2D (DFT), progettazione di filtri analitici nel dominio della frequenza, Trasformata Discreta del Coseno (DCT) e principi fondamentali dei sistemi di compressione delle immagini.
* **Oppenheim (2010)** — Teoria formale di segnali e sistemi applicati nel dominio discreto, che copre le proprietà matematiche della DFT e la modellazione analitica del Teorema della Convoluzione.
* **Mallat (1999)** — Fondamento matematico della teoria delle *wavelet*, formalizzazione dell'analisi multirisoluzione (MRA) e architettura dei banchi di filtri diadici.
* **Wallace (1991)** — Specifica originale e aspetti ingegneristici dello standard di compressione ISO/IEC JPEG, con particolare enfasi sui criteri psicovisuali per la progettazione delle matrici di quantizzazione DCT.
* **Szeliski (2022)** — Modellazione computazionale e caratterizzazione delle metriche moderne di fedeltà e qualità percettiva (PSNR e SSIM), nonché l'analisi comparativa dei formati di immagine rasterizzati ad alte prestazioni.

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.it/cap05/cap05.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 5.17 **5.1 Introduzione**

Questo capitolo tratta della **elaborazione di immagini con Python**, concentrandosi sulla applicazione di filtri e tecniche di manipolazione dei pixel. Verranno presentati esempi pratici e esercizi proposti (EP) per consolidare l'apprendimento.

## 5.18 **5.2 Filtri spaziali**

I filtri spaziali operano direttamente sui pixel di un'immagine. Le operazioni più comuni includono:

- **Filtro medio**: sostituisce ogni pixel con la media dei pixel vicini, riducendo il rumore.
- **Filtro di Sobel**: evidenzia i contorni calcolando il gradiente dell'intensità.
- **Filtro gaussiano**: applica una convoluzione con una funzione gaussiana per sfocare l'immagine.

## 5.19 **5.3 Esempi di codici**

Di seguito alcuni frammenti di codice per illustrare le tecniche.

```python
import numpy as np
from scipy import ndimage

# 5 Applicazione di un filtro medio
def filtro_medio(immagine, dimensione=3):
    kernel = np.ones((dimensione, dimensione)) / (dimensione ** 2)
    return ndimage.convolve(immagine, kernel)
```

```python
# 5 Filtro di Sobel per il rilevamento dei contorni
def filtro_sobel(immagine):
    sobel_x = np.array([[-1, 0, 1],
                        [-2, 0, 2],
                        [-1, 0, 1]])
    sobel_y = np.array([[-1, -2, -1],
                        [0, 0, 0],
                        [1, 2, 1]])
    grad_x = ndimage.convolve(immagine, sobel_x)
    grad_y = ndimage.convolve(immagine, sobel_y)
    return np.sqrt(grad_x**2 + grad_y**2)
```

## 5.1 **5.4 Esercizi proposti**

### 5.1.1 **EP1: Soglia adattiva**
Implementare una funzione che applica una soglia adattiva a un'immagine in scala di grigi, utilizzando la media locale dei pixel.

### 5.1.2 **EP2: Morfologia matematica**
Applicare operazioni di erosione e dilatazione a un'immagine binaria utilizzando elementi strutturanti di diversa forma.

### 5.1.3 **EP3: Trasformata di Hough**
Utilizzare la trasformata di Hough per rilevare linee in un'immagine e visualizzarle sovrapposte all'originale.

## 5.2 **5.5 Considerazioni finali**
Le tecniche presentate costituiscono la base per lo sviluppo di sistemi di visione artificiale più complessi. Si consiglia di sperimentare variando i parametri e analizzando gli effetti sulle immagini.

ZQREF123Z ZQREF456Z

## 5.3 💻 **Parte Pratica con Esercizi di Programmazione**

La presente lista di esercizi di programmazione (EP) consolida le formulazioni teoriche presentate nel corso del Capitolo 5 — Trasformate e Compressione — attraverso un percorso pratico applicato. Gli esercizi sono strutturati a partire da matrici di dimensioni ridotte, consentendo la validazione analitica e l'ispezione manuale di ciascun coefficiente, mantenendo la coerenza metodologica adottata nei capitoli precedenti.

L'incatenamento degli esercizi riproduce rigorosamente il flusso concettuale del capitolo: si inizia con l'implementazione esplicita della Trasformata Discreta di Fourier (DFT) a partire dalla sua definizione matematica fondamentale; si prosegue con la progettazione di filtri passa-basso e maschere *notch* nel dominio della frequenza; si applica la quantizzazione dei coefficienti (nucleo della compressione con perdita); e si conclude con l'integrazione di queste fasi nella costruzione di un *pipeline* di compressione JPEG semplificato e nell'analisi percettiva dei formati immagine.

> ### ❗ Linee Guida per la Risoluzione degli Esercizi di Programmazione
>
> In tutti gli esercizi di questo capitolo, le coordinate del **centro dello spettro** (origine delle frequenze spaziali dopo l'applicazione dello spostamento `fftshift`) devono essere determinate tramite divisione intera. Per una matrice con $L$ righe e $C$ colonne, la componente di frequenza nulla si trova nella posizione:
>
> $$
> (c_y, c_x) = \left( \left\lfloor \frac{L}{2} \right\rfloor, \left\lfloor \frac{C}{2} \right\rfloor \right)
> $$
>
> Questa convenzione è rigorosamente identica a quella adottata dalla funzione `np.fft.fftshift`. Inoltre, in tutte le fasi che richiedono discretizzazione o arrotondamento numerico (sia nella quantizzazione dei coefficienti AC sia nella ricostruzione finale dei pixel), si deve impiegare l'arrotondamento standard al numero intero più vicino (*round half away from zero*), mitigando ambiguità in valori con frazione esattamente pari a $0.5$.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EPs)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

#### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella qui sotto:

In [70]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True, cpp=True)
from morph import mm
from testsuite import TestSuite

✅ Ambiente pronto. Morph: 1.1.9 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Esecuzione dei Test
Per valutare i test, esegui `TestSuite("EP05_01.estensione").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola automaticamente il voto.

Per testare il codice Python direttamente, senza salvare un file, usa `run_code(codice)` passando il codice come *stringa* in una variabile `codice`:

```python
codice = """
from morph import mm
# 5 ... il tuo codice qui ...
"""
TestSuite("EP05_01").run_code(codice)
```

### 5.0.1 EP05_01 🟢 Filtro Passa-Basso Ideale per Distanza nello Spettro

In uno ***scanner* di documenti antico**, il sensore cattura carta stropicciata e la trama delle fibre insieme al testo — rumore ad alta frequenza che "inquina" lo spettro ai bordi. Il tecnico della manutenzione non ha accesso all'immagine originale, ma solo allo **spettro di magnitudo già calcolato** dal software dello *scanner*. Il suo compito è semplice e chirurgico: mantenere solo il **cerchio centrale** delle basse frequenze (la struttura globale del documento) ed eliminare tutto ciò che si trova al di fuori del raggio $D_0$, rimuovendo la trama fine senza nemmeno dover toccare l'immagine spaziale.

Questo è il **Filtro Passa-Basso Ideale (LPFI)**: l'operazione spettrale più diretta del capitolo, ma anche quella che meglio rivela l'anatomia di uno spettro centrato.

#### 5.0.1.1 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $L$ (righe) e $C$ (colonne) dello spettro di magnitudo — già fornito **centrato** (equivalente all'uscita di `np.fft.fftshift`).
2. **Frequenza di taglio:** Leggere l'intero $D_0$.
3. **Dati:** Leggere i valori interi della matrice di magnitudo, riga per riga.
4. **Centro dello spettro:** Calcolare $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
5. **Distanza:** Per ogni posizione $(u,v)$, calcolare
$$
D(u,v) = \sqrt{(u-c_y)^2 + (v-c_x)^2}
$$
6. **Maschera ideale:** Applicare
$$
H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}
$$
7. **Filtraggio:** Il valore di uscita è $\text{mag}'(u,v) = \text{mag}(u,v) \cdot H(u,v)$.
8. **Uscita:** Visualizzare la matrice filtrata con dimensioni $L \times C$.

#### 5.0.1.2 📌 Vincoli Computazionali

* **Confronto non stretto:** il criterio usa $D(u,v) \le D_0$ (il confine appartiene al filtro, cioè viene mantenuto).
* **Tipo:** tutti i valori di ingresso e uscita sono interi; la distanza è calcolata in virgola mobile solo internamente.
* **Nessun arrotondamento della magnitudo:** poiché l'ingresso è già intero e la maschera è binaria (0 o 1), l'uscita non richiede mai arrotondamento.

#### 5.0.1.3 🧠 Fondamenti Teorici

| Regione | Distanza dal centro | Effetto del filtro |
|---|---|---|
| **Centro** ($D \le D_0$) | Basse frequenze | Preservate — struttura globale mantenuta |
| **Bordi** ($D > D_0$) | Alte frequenze | Azzerate — trama e rumore rimossi |
| **$D_0$ piccolo** | — | L'immagine ricostruita sarebbe molto sfocata |
| **$D_0$ grande** | — | Poca filtrazione; quasi tutta l'energia preservata |

#### 5.0.1.4 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $D_0$.
* Righe successive: Elementi interi della matrice di magnitudo (centrata).

**Uscita:**

* Matrice filtrata con $L$ righe e $C$ colonne, separati da spazi.

#### 5.0.1.5 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 3<br>3<br>1<br>10 20 30<br>40 50 60<br>70 80 90 | 0 20 0<br>40 50 60<br>0 80 0 | Centro $(1,1)$. Gli angoli hanno $D=\sqrt{2}\approx1.41 > 1$, quindi vengono azzerati; i vicini ortogonali hanno $D=1 \le 1$ e sono mantenuti. |
| 1<br>3<br>0<br>5 9 7 | 0 9 0 | $L=1, C=3$: centro in $(0,1)$. Solo la posizione centrale stessa ($D=0$) sopravvive a $D_0=0$. |

In [71]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0501" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0501 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0501 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0501 button:hover { background: #e8dfcf; }
  #sim-ep0501 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0501_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0501_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0501_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-ep0501_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-ep0501_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-ep0501_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-ep0501_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_01: Filtro Passa-Basso Ideale</span>
  <span class="sim-ep0501_pill">H = (D &le; D₀) ? 1 : 0</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0501_panel" style="margin-bottom:14px;">
    
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Raggio di taglio (D₀): <span id="sim-ep0501_vl_d0" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    
    <input id="sim-ep0501_sl_d0" type="range" min="0" max="4" step="1" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola D₀ e osserva quali posizioni dello spettro 5&times;5 sopravvivono al filtro.
    </div>

  </div>

  <!-- Exibição das Grades de Espectro -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Spettro Originale (Magnitudine)
      </div>
      <div id="sim-ep0501_grid_orig" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Risultato Filtrato
      </div>
      <div id="sim-ep0501_grid_new" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0501_debug" class="sim-ep0501_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    –
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep01(root){
    if (!root || root.dataset.sim05Ep01Init) return;
    root.dataset.sim05Ep01Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(10 * (i + 1) + j + 1);
      }
      mag.push(row);
    }

    var d0el = root.querySelector('#sim-ep0501_sl_d0');
    var d0v  = root.querySelector('#sim-ep0501_vl_d0');
    var go   = root.querySelector('#sim-ep0501_grid_orig');
    var gn   = root.querySelector('#sim-ep0501_grid_new');
    var dbg  = root.querySelector('#sim-ep0501_debug');

    function cellStyle(active){
      if (active) {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      } else {
        return 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      }
    }

    function render(){
      var D0 = parseInt(d0el.value, 10);
      d0v.textContent = D0;
      go.innerHTML = '';
      gn.innerHTML = '';
      var kept = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d = Math.sqrt((i - cy) * (i - cy) + (j - cx) * (j - cx));
          var keep = d <= D0;
          if (keep) kept++;

          var co = document.createElement('div');
          co.className = 'sim-ep0501_cell';
          co.style.cssText = cellStyle(true);
          co.textContent = mag[i][j];
          go.appendChild(co);

          var cn = document.createElement('div');
          cn.className = 'sim-ep0501_cell';
          cn.style.cssText = cellStyle(keep);
          cn.textContent = keep ? mag[i][j] : 0;
          gn.appendChild(cn);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  D₀ = ' + D0 + '  |  Coeficientes mantidos: ' + kept + ' / ' + (N * N);
    }

    d0el.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep01(){
    var root = document.getElementById('sim-ep0501');
    if (root) initSim05Ep01(root); else setTimeout(tryInitSim05Ep01, 200);
  }
  tryInitSim05Ep01();
})();
</script>
""")

**Figura 5.32:** Simulatore EP05_01: Filtro Passa-Basso Ideale nello Spettro


<figure id="fig-05-sim-ep0501">
  <img src="imagens/fig-05-sim-ep0501.png" alt=" Simulatore EP05_01: Filtro Passa-Basso Ideale nello Spettro " style="max-width:80%" />
  <figcaption><strong>Figura 5.32:</strong>  Simulatore EP05_01: Filtro Passa-Basso Ideale nello Spettro </figcaption>
</figure>

In [72]:
%%writefile EP05_01.cpp
// your solution

Overwriting EP05_01.cpp


In [73]:
TestSuite("EP05_01.cpp").run()

### 5.0.2 EP05_02 🟡 Filtro *Notch*: Rimozione dei Picchi Periodici

Una telecamera di **ispezione industriale** acquisisce immagini di circuiti stampati, ma l'alimentazione della linea di produzione introduce un'**interferenza elettrica periodica** — un pattern di strisce quasi impercettibile a occhio nudo, che però appare nello spettro di Fourier come **coppie di picchi luminosi** posizionati simmetricamente attorno al centro. Il team di visione artificiale non può rielaborare l'acquisizione: deve **localizzare e cancellare chirurgicamente** queste coppie di picchi nello spettro, preservando tutta l'altra informazione utile dell'immagine.

Questo è il ruolo del **filtro rigetta-banda *notch***: a differenza del passa-basso (che interessa una regione continua), esso agisce su **punti specifici e sui loro simmetrici**, lasciando intatto il resto dello spettro.

#### 5.0.2.1 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $L$ (righe) e $C$ (colonne) dello spettro di magnitudine centrato.
2. **Dati:** Leggere i valori interi della matrice di magnitudine, riga per riga.
3. **Picchi:** Leggere l'intero $K$ (numero di coppie di picchi da rimuovere).
4. **Per ciascuno dei $K$ picchi:** leggere tre interi $\Delta v$, $\Delta u$, $r$ — spostamento verticale, spostamento orizzontale e raggio del *notch*.
5. **Centro dello spettro:** $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
6. **Soppressione simmetrica:** per ogni picco, azzerare **tutte** le posizioni $(u,v)$ tali che la distanza dal punto $(c_y+\Delta v,\, c_x+\Delta u)$ sia $\le r$, **e anche** tutte le posizioni con distanza $\le r$ dal punto simmetrico $(c_y-\Delta v,\, c_x-\Delta u)$.
7. **Uscita:** Visualizzare la matrice risultante con dimensioni $L \times C$.

#### 5.0.2.2 📌 Vincoli Computazionali

* **Simmetria obbligatoria:** ogni picco indicato genera **due** dischi azzerati (il punto e il suo simmetrico rispetto al centro) — dimenticare il simmetrico è l'errore più comune.
* **Sovrapposizione:** se due dischi si sovrappongono, la posizione rimane azzerata (non c'è "somma" o ripristino).
* **Confronto non stretto:** una posizione viene azzerata se $\text{distanza} \le r$.
* **Ordine di lettura:** i $K$ picchi devono essere elaborati nell'ordine in cui compaiono nell'input, ma il risultato finale non dipende dall'ordine (le operazioni di azzeramento sono commutative).

#### 5.0.2.3 🧠 Fondamenti Teorici

| Concetto | Ruolo nel filtro *notch* |
|---|---|
| **Picco in $(\Delta v, \Delta u)$** | Frequenza dell'interferenza periodica rilevata visivamente nello spettro |
| **Punto simmetrico $(-\Delta v,-\Delta u)$** | Ogni DFT di segnale reale è hermitiana: i picchi compaiono sempre in coppie simmetriche rispetto al centro |
| **Raggio $r$** | Controlla la "larghezza" della reiezione — un $r$ grande rimuove più energia attorno al picco, ma anche informazione utile |

#### 5.0.2.4 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Righe successive: Elementi interi della matrice di magnitudine (centrata), $L$ righe.
* Riga successiva: Intero $K$.
* $K$ righe successive: tre interi $\Delta v$, $\Delta u$, $r$ (separati da spazi).

**Output:**

* Matrice risultante in $L$ righe e $C$ colonne, separati da spazi.

#### 5.0.2.5 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 5<br>5<br>1 2 3 4 5<br>6 7 8 9 10<br>11 12 13 14 15<br>16 17 18 19 20<br>21 22 23 24 25<br>1<br>1 1 0 | 1 2 3 4 5<br>6 0 8 9 10<br>11 12 13 14 15<br>16 17 18 0 20<br>21 22 23 24 25 | Centro $(c_y, c_x) = (2, 2)$. Il picco indicato $(\Delta v, \Delta u) = (1, 1)$ genera il punto $(3, 3)$ (valore 19) e il suo simmetrico $(1, 1)$ (valore 7), entrambi azzerati con $r=0$ (solo i punti esatti). |

In [74]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0502" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0502 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0502 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0502 button:hover { background: #e8dfcf; }
  #sim-ep0502 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0502_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0502_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0502_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 12px; }
  .sim-ep0502_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_02: Filtro Notch</span>
  <span class="sim-ep0502_pill">Coppia simmetrica</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0502_panel" style="margin-bottom:14px;">
    <div class="sim-ep0502_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;v</label>
          <span id="sim-ep0502_vl_dv" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_dv" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;u</label>
          <span id="sim-ep0502_vl_du" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_du" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Raggio (r)</label>
          <span id="sim-ep0502_vl_r" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0502_sl_r" type="range" min="0" max="2" step="1" value="0">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Muovi &Delta;v e &Delta;u per scegliere il picco &mdash; osserva che anche la coppia simmetrica viene filtrata.
    </div>
  </div>

  <!-- Espectro 5x5 -->
  <div class="sim-ep0502_panel" style="text-align:center; margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
      Spettro 5&times;5 (Rosso = Rimosso dal Filtro)
    </div>
    <div id="sim-ep0502_grid" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0502_debug" class="sim-ep0502_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep02(root){
    if (!root || root.dataset.sim05Ep02Init) return;
    root.dataset.sim05Ep02Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(i * 5 + j + 1);
      }
      mag.push(row);
    }

    var dv  = root.querySelector('#sim-ep0502_sl_dv');
    var du  = root.querySelector('#sim-ep0502_sl_du');
    var r   = root.querySelector('#sim-ep0502_sl_r');
    var dvv = root.querySelector('#sim-ep0502_vl_dv');
    var duv = root.querySelector('#sim-ep0502_vl_du');
    var rv  = root.querySelector('#sim-ep0502_vl_r');

    var grid = root.querySelector('#sim-ep0502_grid');
    var dbg  = root.querySelector('#sim-ep0502_debug');

    function cellStyle(kill){
      if (kill) {
        return 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
      } else {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
    }

    function render(){
      var DV = parseInt(dv.value, 10);
      var DU = parseInt(du.value, 10);
      var R  = parseInt(r.value, 10);

      dvv.textContent = DV;
      duv.textContent = DU;
      rv.textContent  = R;

      var p1 = [cy + DV, cx + DU];
      var p2 = [cy - DV, cx - DU];

      grid.innerHTML = '';
      var removed = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d1 = Math.sqrt((i - p1[0]) * (i - p1[0]) + (j - p1[1]) * (j - p1[1]));
          var d2 = Math.sqrt((i - p2[0]) * (i - p2[0]) + (j - p2[1]) * (j - p2[1]));
          var kill = (d1 <= R) || (d2 <= R);

          if (kill) removed++;

          var c = document.createElement('div');
          c.className = 'sim-ep0502_cell';
          c.style.cssText = cellStyle(kill);
          c.textContent = kill ? 0 : mag[i][j];
          grid.appendChild(c);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  Pico = (' + p1[0] + ', ' + p1[1] + ')  |  Simétrico = (' + p2[0] + ', ' + p2[1] + ')  |  Removidos: ' + removed;
    }

    [dv, du, r].forEach(function(el){
      el.addEventListener('input', render);
    });

    render();
  }

  function tryInitSim05Ep02(){
    var root = document.getElementById('sim-ep0502');
    if (root) initSim05Ep02(root); else setTimeout(tryInitSim05Ep02, 200);
  }
  tryInitSim05Ep02();
})();
</script>
""")

**Figura 5.33:** Simulatore EP05_02: Filtro Notch


<figure id="fig-05-sim-ep0502">
  <img src="imagens/fig-05-sim-ep0502.png" alt=" Simulatore EP05_02: Filtro Notch " style="max-width:80%" />
  <figcaption><strong>Figura 5.33:</strong>  Simulatore EP05_02: Filtro Notch </figcaption>
</figure>

In [75]:
%%writefile EP05_02.cpp
// your solution

Overwriting EP05_02.cpp


In [76]:
TestSuite("EP05_02.cpp").run()

### 5.0.3 EP05_03 🟠 Quantizzazione DCT: la Vera Fonte di Compressione

Un'applicazione di **galleria fotografica** deve ridurre le dimensioni di migliaia di immagini prima di caricarle sul cloud, senza ricodificare tutto da zero. L'ingegnere responsabile ha già i **coefficienti DCT** di ogni blocco $4\times4$ calcolati (la fase computazionalmente onerosa è già stata eseguita) — manca solo applicare la **tabella di quantizzazione**, la fase che scarta realmente informazioni e genera compressione. I coefficienti ad alta frequenza, meno percettibili all'occhio umano, ricevono divisori grandi e tendono a diventare **zero**; i coefficienti a bassa frequenza, più percettibili, ricevono divisori piccoli e sopravvivono quasi intatti.

Dovrai implementare esattamente questa fase: **quantizzare e dequantizzare** (dividere, arrotondare, moltiplicare di nuovo) — il cuore della compressione *lossy* del JPEG.

#### 5.0.3.1 📋 Linee Guida di Implementazione

1. **Dimensione del blocco:** Leggere l'intero $N$ (blocco $N \times N$).
2. **Coefficienti:** Leggere la matrice $C$ dei coefficienti DCT, $N$ righe con $N$ interi ciascuna (possono essere negativi).
3. **Tabella di quantizzazione:** Leggere la matrice $Q$, $N$ righe con $N$ interi positivi ciascuna.
4. **Quantizzazione:** Per ogni posizione $(u,v)$, calcolare l'indice quantizzato
$$
\tilde{C}(u,v) = \text{round}\!\left(\frac{C(u,v)}{Q(u,v)}\right)
$$
usando l'arrotondamento standard all'intero più vicino (i valori intermedi `.5` non si verificano mai nei casi di test).
5. **Dequantizzazione (ricostruzione):** Calcolare
$$
C'(u,v) = \tilde{C}(u,v) \times Q(u,v)
$$
6. **Output:** Visualizzare la matrice ricostruita $C'$, $N \times N$, di interi.

#### 5.0.3.2 📌 Vincoli Computazionali

* ***Round-trip* completo:** l'output è il coefficiente **ricostruito** ($\tilde{C} \times Q$), non l'indice quantizzato isolato.
* **Divisione in virgola mobile:** la divisione $C(u,v)/Q(u,v)$ deve essere eseguita in virgola mobile prima dell'arrotondamento — la divisione intera troncata produrrà un risultato errato.
* **Segno preservato:** i coefficienti negativi mantengono il segno dopo la quantizzazione e la ricostruzione.
* **$Q(u,v) > 0$ sempre:** non è necessario gestire la divisione per zero.

#### 5.0.3.3 🧠 Fondamenti Teorici

| Coefficiente | Frequenza | Valore tipico di $Q$ | Effetto della quantizzazione |
|---|---|---|---|
| $C(0,0)$ | DC (media del blocco) | Piccolo | Quasi sempre sopravvive — domina l'energia |
| $C(u,v)$ con $u+v$ basso | Bassa frequenza | Piccolo/medio | Parzialmente preservato |
| $C(u,v)$ con $u+v$ alto | Alta frequenza | Grande | Spesso diventa zero — fonte della compressione |

#### 5.0.3.4 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $N$.
* $N$ righe successive: matrice $C$ (coefficienti DCT, interi, possono essere negativi).
* $N$ righe successive: matrice $Q$ (tabella di quantizzazione, interi positivi).

**Output:**

* Matrice ricostruita $C'$, $N \times N$, interi separati da spazio.

#### 5.0.3.5 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4<br>50 10 -5 0<br>8 -3 2 1<br>0 1 0 0<br>2 0 0 -1<br>2 5 7 8<br>4 7 8 11<br>6 8 11 12<br>9 11 12 14 | 50 10 -7 0<br>8 0 0 0<br>0 0 0 0<br>0 0 0 0 | $C(0,0)=50/2=25 \to 25\times2=50$ (preservato). $C(0,2)=-5/7\approx-0.71\to-1\to-1\times7=-7$. Invece $C(1,1)=-3/7\approx-0.43\to0$: azzerato dalla quantizzazione — la maggior parte del blocco diventa zero, illustrando la compattazione dell'energia nell'angolo superiore sinistro. |

In [77]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0503" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0503 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0503 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0503 button:hover { background: #e8dfcf; }
  #sim-ep0503 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0503_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0503_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0503_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_03: Quantizzazione DCT</span>
  <span class="sim-ep0503_pill">round(C / Q) &times; Q</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0503_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Scala di Q (Aggressività): <span id="sim-ep0503_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0503_sl_s" type="range" min="0.25" max="4" step="0.25" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola la scala di Q e osserva quanti coefficienti sopravvivono (diversi da zero) dopo il round-trip.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Coefficienti DCT (C)
      </div>
      <div id="sim-ep0503_grid_c" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Ricostruito (round(C / Q) &middot; Q)
      </div>
      <div id="sim-ep0503_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0503_debug" class="sim-ep0503_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep03(root){
    if (!root || root.dataset.sim05Ep03Init) return;
    root.dataset.sim05Ep03Init = "1";

    var C = [[50, 10, -5, 0], [8, -3, 2, 1], [0, 1, 0, 0], [2, 0, 0, -1]];
    var Qbase = [[2, 5, 7, 8], [4, 7, 8, 11], [6, 8, 11, 12], [9, 11, 12, 14]];

    var s   = root.querySelector('#sim-ep0503_sl_s');
    var sv  = root.querySelector('#sim-ep0503_vl_s');
    var gc  = root.querySelector('#sim-ep0503_grid_c');
    var gr  = root.querySelector('#sim-ep0503_grid_r');
    var dbg = root.querySelector('#sim-ep0503_debug');

    function cell(v, faded){
      var c = document.createElement('div');
      c.className = 'sim-ep0503_cell';
      if (faded) {
        c.style.cssText = 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      } else {
        c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      gc.innerHTML = '';
      gr.innerHTML = '';
      var zeros = 0, total = 16;

      for (var i = 0; i < 4; i++){
        for (var j = 0; j < 4; j++){
          gc.appendChild(cell(C[i][j], false));
          var Q = Qbase[i][j] * scale;
          var q = Math.round(C[i][j] / Q);
          var rec = Math.round(q * Q);
          if (rec === 0) zeros++;
          gr.appendChild(cell(rec, rec === 0));
        }
      }

      dbg.textContent = 'Zeri: ' + zeros + ' / ' + total + '  |  Quanto maior a escala de Q, mais zeros — maior compressão, menor qualidade.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep03(){
    var root = document.getElementById('sim-ep0503');
    if (root) initSim05Ep03(root); else setTimeout(tryInitSim05Ep03, 200);
  }
  tryInitSim05Ep03();
})();
</script>
""")

**Figura 5.34:** Simulatore EP05_03: Quantizzazione DCT (*round-trip*)


<figure id="fig-05-sim-ep0503">
  <img src="imagens/fig-05-sim-ep0503.png" alt=" Simulatore EP05_03: Quantizzazione DCT (*round-trip*) " style="max-width:80%" />
  <figcaption><strong>Figura 5.34:</strong>  Simulatore EP05_03: Quantizzazione DCT (*round-trip*) </figcaption>
</figure>

In [78]:
%%writefile EP05_03.cpp
// your solution

Overwriting EP05_03.cpp


In [79]:
TestSuite("EP05_03.cpp").run()

### 5.0.4 EP05_04 🔴 Implementazione della DFT 2D a partire dalla definizione

Un laboratorio di ricerca in **astronomia computazionale** ha ricevuto, da una missione remota, un piccolo sensore sperimentale i cui dati grezzi non possono essere elaborati tramite librerie moderne di FFT — l'ambiente di validazione è isolato e consente solo operazioni aritmetiche di base. Il team deve **reimplementare la Trasformata di Fourier Discreta 2D a partire dalla definizione matematica stessa**, cella per cella, per poi confrontare bit per bit con `np.fft.fft2` in un altro ambiente.

Questo è l'esercizio più concettuale della lista: non ci sono scorciatoie. Dovrai implementare direttamente la doppia sommatoria della [Equação 5.1](#eq-05-dft), evidenziando *perché* la FFT esiste — e il costo computazionale che essa evita.

#### 5.0.4.1 📋 Linee guida per l'implementazione

1. **Dimensioni:** Leggere i numeri interi $M$ (righe) e $N$ (colonne) dell'immagine $f(x,y)$.
2. **Dati:** Leggere i valori interi di $f(x,y)$, riga per riga.
3. **DFT 2D:** Per ogni coppia di frequenze $(u,v)$ con $u=0,\ldots,M-1$ e $v=0,\ldots,N-1$, calcolare
$$
F(u,v) = \sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$
usando l'identità di Eulero $e^{-j\theta} = \cos(\theta) - j\sin(\theta)$ per separare la parte reale e quella immaginaria — **non utilizzare alcuna funzione FFT predefinita**.
4. **Magnitudine:** Calcolare $|F(u,v)| = \sqrt{\text{Re}(F)^2 + \text{Im}(F)^2}$ e arrotondare all'intero più vicino.
5. **Output:** Visualizzare la matrice delle magnitudini arrotondate, $M \times N$, nello stesso ordine (senza `fftshift` — il componente DC rimane in $(0,0)$).

#### 5.0.4.2 📌 Vincoli computazionali

* **Vietato l'uso di librerie FFT:** l'implementazione deve calcolare esplicitamente le doppie sommatorie (cicli annidati), anche se più lenta.
* **Senza `fftshift`:** l'output mantiene la convenzione grezza della DFT, con il componente DC in $F(0,0)$ (angolo superiore sinistro).
* **Arrotondamento:** la magnitudine finale deve essere arrotondata all'intero più vicino; nei casi di test non vi è ambiguità `.5`.
* **Precisione:** piccoli errori di virgola mobile (ordine di $10^{-6}$) prima dell'arrotondamento sono previsti e non influenzano il risultato intero finale.

#### 5.0.4.3 🧠 Fondamenti teorici

| Elemento | Significato |
|---|---|
| $F(0,0)$ | Componente DC — somma di tutti i pixel, $F(0,0) = \sum f(x,y)$ |
| Parte reale $\text{Re}(F)$ | Proiezione del segnale sui coseni |
| Parte immaginaria $\text{Im}(F)$ | Proiezione del segnale sui seni |
| Complessità di questa implementazione | $\mathcal{O}((MN)^2)$ — ecco perché la FFT, con $\mathcal{O}(MN\log(MN))$, è indispensabile nelle immagini reali |

#### 5.0.4.4 📦 Specifica di input e output (VPL)

**Input:**

* Riga 1: intero $M$.
* Riga 2: intero $N$.
* Righe successive: elementi interi di $f(x,y)$, $M$ righe.

**Output:**

* Matrice delle magnitudini $|F(u,v)|$ arrotondate, $M \times N$, separate da spazio.

#### 5.0.4.5 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 2<br>2<br>1 2<br>3 4 | 10 2<br>4 0 | $F(0,0)=1+2+3+4=10$ (DC = somma totale). $F(0,1)=(1-2)+(3-4)=-2 \to |F|=2$. $F(1,0)=(1+2)-(3+4)=-4\to|F|=4$. $F(1,1)=(1-2)-(3-4)=0$. |

In [80]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0504" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0504 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0504 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0504 button:hover { background: #e8dfcf; }
  .sim-ep0504_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0504_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0504_cell { width: 52px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 13px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_04: DFT 2D &mdash; Definizione Diretta</span>
  <span class="sim-ep0504_pill">&Sigma;&Sigma; f(x,y) e<sup>-j2&pi;(&hellip;)</sup></span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Instrução -->
  <div class="sim-ep0504_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
      Clicca sulle celle di f(x,y) per modificarne i valori (incrementa +1; Shift + clic decrementa -1) e osserva |F(u,v)| ricalcolato in tempo reale.
    </div>
  </div>

  <!-- Exibição das Grades 2x2 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        f(x,y) &mdash; Dominio Spaziale
      </div>
      <div id="sim-ep0504_grid_f" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        |F(u,v)| &mdash; Magnitudine (Senza Shift)
      </div>
      <div id="sim-ep0504_grid_F" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0504_debug" class="sim-ep0504_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep04(root){
    if (!root || root.dataset.sim05Ep04Init) return;
    root.dataset.sim05Ep04Init = "1";

    var f = [[1, 2], [3, 4]];
    var gf  = root.querySelector('#sim-ep0504_grid_f');
    var gF  = root.querySelector('#sim-ep0504_grid_F');
    var dbg = root.querySelector('#sim-ep0504_debug');

    function render(){
      gf.innerHTML = '';
      gF.innerHTML = '';

      for (var x = 0; x < 2; x++){
        for (var y = 0; y < 2; y++){
          (function(xx, yy){
            var c = document.createElement('div');
            c.className = 'sim-ep0504_cell';
            c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7; cursor:pointer;';
            c.textContent = f[xx][yy];
            c.addEventListener('click', function(e){
              if (e.shiftKey){ f[xx][yy]--; } else { f[xx][yy]++; }
              render();
            });
            gf.appendChild(c);
          })(x, y);
        }
      }

      var M = 2, N = 2;
      for (var u = 0; u < M; u++){
        for (var v = 0; v < N; v++){
          var re = 0, im = 0;
          for (var x = 0; x < M; x++){
            for (var y = 0; y < N; y++){
              var theta = 2 * Math.PI * (u * x / M + v * y / N);
              re += f[x][y] * Math.cos(theta);
              im -= f[x][y] * Math.sin(theta);
            }
          }
          var mag = Math.round(Math.sqrt(re * re + im * im));
          var c = document.createElement('div');
          c.className = 'sim-ep0504_cell';
          c.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          c.textContent = mag;
          gF.appendChild(c);
        }
      }

      dbg.textContent = 'F(0,0) = somma di tutti i pixel = ' + (f[0][0] + f[0][1] + f[1][0] + f[1][1]) + ' (componente DC)';
    }

    render();
  }

  function tryInitSim05Ep04(){
    var root = document.getElementById('sim-ep0504');
    if (root) initSim05Ep04(root); else setTimeout(tryInitSim05Ep04, 200);
  }
  tryInitSim05Ep04();
})();
</script>
""")

**Figura 5.35:** Simulatore EP05_04: DFT 2D manuale


<figure id="fig-05-sim-ep0504">
  <img src="imagens/fig-05-sim-ep0504.png" alt=" Simulatore EP05_04: DFT 2D manuale " style="max-width:80%" />
  <figcaption><strong>Figura 5.35:</strong>  Simulatore EP05_04: DFT 2D manuale </figcaption>
</figure>

In [81]:
%%writefile EP05_04.cpp
// your solution

Overwriting EP05_04.cpp


In [82]:
TestSuite("EP05_04.cpp").run()

### 5.0.5 EP05_05 🏆 *Pipeline* JPEG Completo: DCT, Quantizzazione e Ricostruzione

Sei stato incaricato di creare, da zero, un **codec JPEG didattico** in un ambiente embedded, senza alcuna libreria di immagini disponibile — solo operazioni matematiche di base. Il cliente vuole capire esattamente dove la qualità viene persa e dove viene recuperata, blocco per blocco. Questa è la sfida finale del capitolo: integrare **tutto** ciò che è stato studiato — la DCT-II ortonormale, la quantizzazione percettiva e la ricostruzione tramite IDCT — in un unico *pipeline* end-to-end, elaborando un blocco $N \times N$ dall'inizio alla fine, esattamente come fa internamente lo standard JPEG, $8\times8$ pixel alla volta.

#### 5.0.5.1 📋 Linee Guida di Implementazione

1. **Dimensione del blocco:** Leggere il numero intero $N$.
2. **Blocco originale:** Leggere la matrice di pixel $f(x,y)$, $N$ righe con $N$ interi in $[0,255]$.
3. **Tabella di quantizzazione:** Leggere la matrice $Q$, $N \times N$ interi positivi.
4. **Centratura:** Sottrarre 128 da ogni pixel: $g(x,y) = f(x,y) - 128$.
5. **DCT-II 2D ortonormale:** Calcolare
$$
C(u,v) = \alpha(u)\,\alpha(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} g(x,y)\,\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]\cos\!\left[\frac{\pi(2y+1)v}{2N}\right]
$$
con $\alpha(0)=\sqrt{1/N}$ e $\alpha(k)=\sqrt{2/N}$ per $k>0$.
6. **Quantizzazione:** $\tilde{C}(u,v) = \text{round}(C(u,v)/Q(u,v))$.
7. **Dequantizzazione:** $C'(u,v) = \tilde{C}(u,v)\times Q(u,v)$.
8. **IDCT-II 2D (inversa ortonormale):** Calcolare $g'(x,y)$ da $C'(u,v)$ usando la trasformata inversa corrispondente (stessa base, sommatoria su $u,v$).
9. **Inversione della centratura e arrotondamento:** $f'(x,y) = \text{round}(g'(x,y) + 128)$, limitato all'intervallo $[0,255]$ (*clipping*).
10. **Output:** Visualizzare il blocco ricostruito $f'$, $N \times N$, interi.

#### 5.0.5.2 📌 Vincoli Computazionali

* ***Pipeline* completo obbligatorio:** tutte e sei le fasi (centrare, DCT, quantizzare, dequantizzare, IDCT, invertire) devono essere implementate — saltare la quantizzazione non supera i test, poiché il risultato sarebbe identico all'originale.
* ***Clipping*:** i valori ricostruiti al di fuori di $[0,255]$ devono essere troncati (0 se negativi, 255 se maggiori di 255).
* **Arrotondamento:** sia nella quantizzazione che nella ricostruzione finale dei pixel, usare l'arrotondamento standard; i casi di test evitano ambiguità `.5`.
* **Base ortonormale:** la normalizzazione $\alpha(u)$ e $\alpha(v)$ deve essere applicata esattamente come specificato — senza di essa, la IDCT non ricostruisce correttamente.

#### 5.0.5.3 🧠 Fondamenti Teorici

| Fase | Analoga nel vero standard JPEG | Dove la qualità viene persa |
|---|---|---|
| Centratura | Stessa — la DCT presuppone un segnale centrato su zero | Nessuna perdita |
| DCT-II | Fasi 3–4 del *pipeline* ([Tabela 5.7](#tbl-05-pipeline-jpeg)) | Nessuna perdita (trasformazione esatta e reversibile) |
| Quantizzazione | Fase 5 — divisione per $Q(u,v)$ | **Principale fonte di perdita** — i coefficienti ad alta frequenza diventano zero |
| IDCT | Ricostruzione finale | Ricostruisce esattamente i coefficienti *quantizzati*, non quelli originali |

#### 5.0.5.4 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $N$.
* $N$ righe successive: blocco originale $f(x,y)$, interi in $[0,255]$.
* $N$ righe successive: tabella di quantizzazione $Q$, interi positivi.

**Output:**

* Blocco ricostruito $f'(x,y)$, $N \times N$, interi in $[0,255]$, separati da spazi.

#### 5.0.5.5 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4<br>120 130 125 128<br>115 140 135 122<br>118 150 160 130<br>110 120 145 138<br>4 6 8 10<br>6 8 10 12<br>8 10 12 16<br>10 12 16 20 | 118 126 119 131<br>114 143 140 119<br>117 149 159 130<br>107 121 146 139 | Dopo la DCT, una quantizzazione aggressiva sulle alte frequenze (valori grandi di $Q$ nell'angolo in basso a destra) e la ricostruzione tramite IDCT, il blocco risulta **vicino** all'originale, ma non identico — la differenza è il costo della compressione *lossy*. |

#### 5.0.5.6 💡 Suggerimento per il Debug

Se il risultato non corrisponde, verifica in questo ordine: (1) i coefficienti DCT grezzi (prima della quantizzazione) — devono ricostruire l'originale **esattamente** tramite IDCT se salti le fasi 6–7; (2) la tabella $\alpha(u)$ — un errore comune è applicare $\sqrt{2/N}$ anche per $u=0$; (3) l'arrotondamento della quantizzazione, che deve avvenire **prima** di moltiplicare di nuovo per $Q$.

In [83]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0505" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0505 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0505 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0505 button:hover { background: #e8dfcf; }
  #sim-ep0505 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0505_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0505_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0505_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP05_05: Pipeline JPEG (Blocco 4&times;4)</span>
  <span class="sim-ep0505_pill">DCT &rarr; Q &rarr; IDCT</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0505_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Scala di Q (1 = Tabella Base, Maggiore = Più Perdita): <span id="sim-ep0505_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0505_sl_s" type="range" min="0.5" max="5" step="0.5" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola il fattore di scala di quantizzazione e osserva il blocco ricostruito allontanarsi (o avvicinarsi) dall'originale.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Blocco Originale
      </div>
      <div id="sim-ep0505_grid_o" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Ricostruito (DCT &rarr; Q &rarr; IDCT)
      </div>
      <div id="sim-ep0505_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0505_debug" class="sim-ep0505_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep05(root){
    if (!root || root.dataset.sim05Ep05Init) return;
    root.dataset.sim05Ep05Init = "1";

    var N = 4;
    var f = [[120, 130, 125, 128], [115, 140, 135, 122], [118, 150, 160, 130], [110, 120, 145, 138]];
    var Qbase = [[4, 6, 8, 10], [6, 8, 10, 12], [8, 10, 12, 16], [10, 12, 16, 20]];

    var s   = root.querySelector('#sim-ep0505_sl_s');
    var sv  = root.querySelector('#sim-ep0505_vl_s');
    var go  = root.querySelector('#sim-ep0505_grid_o');
    var gr  = root.querySelector('#sim-ep0505_grid_r');
    var dbg = root.querySelector('#sim-ep0505_debug');

    function alpha(k){ return k === 0 ? Math.sqrt(1 / N) : Math.sqrt(2 / N); }

    function dct2(g){
      var C = [];
      for (var u = 0; u < N; u++){ C.push(new Array(N).fill(0)); }
      for (var u = 0; u < N; u++){
        for (var v = 0; v < N; v++){
          var sum = 0;
          for (var x = 0; x < N; x++){
            for (var y = 0; y < N; y++){
              sum += g[x][y] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          C[u][v] = alpha(u) * alpha(v) * sum;
        }
      }
      return C;
    }

    function idct2(C){
      var g = [];
      for (var x = 0; x < N; x++){ g.push(new Array(N).fill(0)); }
      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          var sum = 0;
          for (var u = 0; u < N; u++){
            for (var v = 0; v < N; v++){
              sum += alpha(u) * alpha(v) * C[u][v] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          g[x][y] = sum;
        }
      }
      return g;
    }

    function cell(v){
      var c = document.createElement('div');
      c.className = 'sim-ep0505_cell';
      c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      go.innerHTML = '';
      gr.innerHTML = '';

      var g = [];
      for (var x = 0; x < N; x++){
        var row = [];
        for (var y = 0; y < N; y++){
          row.push(f[x][y] - 128);
        }
        g.push(row);
      }

      var C = dct2(g);
      var Cq = [];
      for (var u = 0; u < N; u++){
        var row = [];
        for (var v = 0; v < N; v++){
          var Qv = Qbase[u][v] * scale;
          var q = Math.round(C[u][v] / Qv);
          row.push(q * Qv);
        }
        Cq.push(row);
      }

      var gr2 = idct2(Cq);
      var diffSum = 0, n = 0;

      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          go.appendChild(cell(f[x][y]));
          var rec = Math.round(gr2[x][y] + 128);
          rec = Math.max(0, Math.min(255, rec));
          gr.appendChild(cell(rec));
          diffSum += Math.abs(rec - f[x][y]);
          n++;
        }
      }

      dbg.textContent = 'Errore medio assoluto per pixel: ' + (diffSum / n).toFixed(2) + '  |  Quanto maior a escala de Q, maior o erro de reconstrução.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep05(){
    var root = document.getElementById('sim-ep0505');
    if (root) initSim05Ep05(root); else setTimeout(tryInitSim05Ep05, 200);
  }
  tryInitSim05Ep05();
})();
</script>
""")

**Figura 5.36:** Simulatore EP05_05: *Pipeline* JPEG completo a blocchi


<figure id="fig-05-sim-ep0505">
  <img src="imagens/fig-05-sim-ep0505.png" alt=" Simulatore EP05_05: *Pipeline* JPEG completo a blocchi " style="max-width:80%" />
  <figcaption><strong>Figura 5.36:</strong>  Simulatore EP05_05: *Pipeline* JPEG completo a blocchi </figcaption>
</figure>

In [84]:
%%writefile EP05_05.cpp
// your solution

Overwriting EP05_05.cpp


In [85]:
TestSuite("EP05_05.cpp").run()

## Referências do Capítulo


GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

MALLAT, St{\'e}phane. **A wavelet tour of signal processing**. Elsevier, 1999.

OPPENHEIM, Alan V.; SCHAFER, Ronald W. **Discrete-Time Signal Processing**. Pearson, 2010.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

WALLACE, Gregory K. **The {JPEG} Still Picture Compression Standard**. 1991.